1. Import Libraries & Configuration
2. Load Raw and Phase-1 Processed Data
3. Standardize User Identifiers
4. Prepare Binary Labels
5. Remove Leakage / Invalid Features
6. Define Feature Groups
7. Clean Numerical Features
8. Clean Categorical / Boolean Features
9. Prepare Profile Text
10. Prepare User-level Tweet Data
11. Prepare Temporal / Behavioral Features
12. Prepare Graph Availability Features
13. Build Modality Masks
14. Build Final Labeled Dataset
15. Clean Unlabeled Pseudo-label Pool
16. Stratified Train / Validation / Test Split
17. Fit Preprocessing Only on Train
18. Transform Validation / Test / Unlabeled
19. Export Model-ready Files
20. Final Validation Report

In [3]:
# ============================================================
# 02_data_preprocessing_and_feature_engineering.ipynb
# Cell 1 — Imports & Global Configuration
# ============================================================

from pathlib import Path
import os
import random
import json
import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")


# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)


# ------------------------------------------------------------
# Project Paths
# Notebook is expected to be inside:
# Bot Detection Implementation/analysis/
# ------------------------------------------------------------

PROJECT_ROOT = Path("..").resolve()

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = PROJECT_ROOT / "processed_data"
RESULTS_DIR = PROJECT_ROOT / "results"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# Raw Dataset Paths
# ------------------------------------------------------------

LABELED_USERS_PATH = DATA_DIR / "1000user_sheet - 1k_users.csv"

# Master user pool — contains most of the labeled 1K users too
ALL_USERS_19K_PATH = DATA_DIR / "all_users_new - 19k.csv"

# Follower / Following information collected from the 19K universe
FOLLOWERS_GRAPH_PATH = DATA_DIR / "all_users_new - follower_following.csv"

# Tweet data mainly associated with the labeled 1K users
TWEETS_PATH = DATA_DIR / "1000user_sheet - tweets_meta_data.csv"


# ------------------------------------------------------------
# Phase-1 Processed Files
# ------------------------------------------------------------

LABELED_MODALITY_PATH = (
    PROCESSED_DIR / "labeled_modality_audit.csv"
)

UNLABELED_MODALITY_PATH = (
    PROCESSED_DIR / "unlabeled_modality_audit.csv"
)

TWEET_STATS_PATH = (
    PROCESSED_DIR / "tweet_user_statistics.csv"
)

TWEET_COUNT_PATH = (
    PROCESSED_DIR / "tweet_count_per_user.csv"
)

GRAPH_STATS_PATH = (
    PROCESSED_DIR / "graph_user_statistics.csv"
)

GRAPH_EDGES_PATH = (
    PROCESSED_DIR / "graph_edges.csv"
)

PSEUDO_POOL_PATH = (
    PROCESSED_DIR / "unlabeled_pseudo_label_pool.csv"
)


print("Project root:")
print(PROJECT_ROOT)

print("\nData directory:")
print(DATA_DIR)

print("\nProcessed directory:")
print(PROCESSED_DIR)

print("\nRandom seed:")
print(SEED)

Project root:
C:\Users\p.vazifeh\Desktop\University\Project\Bot Detection Implementation

Data directory:
C:\Users\p.vazifeh\Desktop\University\Project\Bot Detection Implementation\data

Processed directory:
C:\Users\p.vazifeh\Desktop\University\Project\Bot Detection Implementation\processed_data

Random seed:
42


In [4]:
# ============================================================
# Cell 2 — Load Raw Datasets & Verify Data Sources
# ============================================================

# ------------------------------------------------------------
# Load raw datasets
# ------------------------------------------------------------

labeled_users = pd.read_csv(
    LABELED_USERS_PATH,
    low_memory=False
)

all_users_19k = pd.read_csv(
    ALL_USERS_19K_PATH,
    low_memory=False
)

followers_graph = pd.read_csv(
    FOLLOWERS_GRAPH_PATH,
    low_memory=False
)

tweets = pd.read_csv(
    TWEETS_PATH,
    low_memory=False
)


# ------------------------------------------------------------
# Basic shapes
# ------------------------------------------------------------

print("=" * 70)
print("RAW DATASET SHAPES")
print("=" * 70)

print(f"Labeled users (1K):       {labeled_users.shape}")
print(f"All users (19K):          {all_users_19k.shape}")
print(f"Follower/Following data:  {followers_graph.shape}")
print(f"Tweets:                   {tweets.shape}")


# ------------------------------------------------------------
# Required identifier checks
# ------------------------------------------------------------

datasets = {
    "labeled_users": labeled_users,
    "all_users_19k": all_users_19k,
    "followers_graph": followers_graph,
    "tweets": tweets
}

print("\n" + "=" * 70)
print("IDENTIFIER COLUMNS")
print("=" * 70)

for name, df in datasets.items():
    
    has_id = "id" in df.columns
    has_screen_name = "screen_name" in df.columns
    
    print(
        f"{name:20s} | "
        f"id: {has_id!s:5s} | "
        f"screen_name: {has_screen_name}"
    )


# ------------------------------------------------------------
# Label column detection
# ------------------------------------------------------------

possible_label_columns = [
    "برچسب نهایی",
    "label",
    "class"
]

label_column = None

for col in possible_label_columns:
    if col in labeled_users.columns:
        label_column = col
        break

print("\n" + "=" * 70)
print("LABEL INFORMATION")
print("=" * 70)

print("Detected label column:", label_column)

if label_column is not None:
    print("\nLabel distribution:")
    print(
        labeled_users[label_column]
        .value_counts(dropna=False)
    )


# ------------------------------------------------------------
# Important reminder about dataset roles
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("DATASET ROLE CHECK")
print("=" * 70)

print("""
1K Users:
    Ground-truth labeled users.

19K Users:
    Master user pool.
    NOT a purely unlabeled dataset.

Tweets:
    Tweet modality collected mainly for the labeled users.

Follower/Following:
    Network information collected from the wider 19K user universe.

Next step:
    Normalize user identifiers and measure exact overlaps between
    labeled users, tweet users, graph users, and the 19K pool.
""")

RAW DATASET SHAPES
Labeled users (1K):       (1103, 101)
All users (19K):          (19510, 75)
Follower/Following data:  (3453, 4)
Tweets:                   (75913, 38)

IDENTIFIER COLUMNS
labeled_users        | id: True  | screen_name: True
all_users_19k        | id: True  | screen_name: True
followers_graph      | id: True  | screen_name: True
tweets               | id: True  | screen_name: True

LABEL INFORMATION
Detected label column: برچسب نهایی

Label distribution:
برچسب نهایی
human(2)         772
bot(1)           187
can’t find        58
unverified(4)     57
News Agent(3)     29
Name: count, dtype: int64

DATASET ROLE CHECK

1K Users:
    Ground-truth labeled users.

19K Users:
    Master user pool.
    NOT a purely unlabeled dataset.

Tweets:
    Tweet modality collected mainly for the labeled users.

Follower/Following:
    Network information collected from the wider 19K user universe.

Next step:
    Normalize user identifiers and measure exact overlaps between
    labeled u

In [5]:
# ============================================================
# Cell 3 — Standardize User Identifiers & Exact Modality Overlap
# ============================================================

# ------------------------------------------------------------
# 1. Create a canonical user key from screen_name
# ------------------------------------------------------------

def normalize_screen_name(series):
    """
    Normalize Twitter/X screen names for cross-dataset matching.
    - convert to pandas string dtype
    - strip spaces
    - remove leading @
    - lowercase
    - convert empty strings to missing values
    """
    s = series.astype("string")
    s = s.str.strip()
    s = s.str.replace(r"^@", "", regex=True)
    s = s.str.lower()
    s = s.replace("", pd.NA)
    return s


# Do not overwrite original screen_name columns.
labeled_users["user_key"] = normalize_screen_name(
    labeled_users["screen_name"]
)

all_users_19k["user_key"] = normalize_screen_name(
    all_users_19k["screen_name"]
)

followers_graph["user_key"] = normalize_screen_name(
    followers_graph["screen_name"]
)

tweets["user_key"] = normalize_screen_name(
    tweets["screen_name"]
)


# ------------------------------------------------------------
# 2. Basic identifier quality check
# ------------------------------------------------------------

print("=" * 75)
print("IDENTIFIER QUALITY CHECK")
print("=" * 75)

for name, df in datasets.items():
    # datasets was created in Cell 2, but those DataFrames are the same objects
    missing_keys = df["user_key"].isna().sum()
    unique_keys = df["user_key"].nunique(dropna=True)

    print(
        f"{name:20s} | "
        f"rows = {len(df):6d} | "
        f"unique users = {unique_keys:6d} | "
        f"missing user_key = {missing_keys:4d}"
    )


# ------------------------------------------------------------
# 3. Define users represented in each source
# ------------------------------------------------------------

labeled_set = set(
    labeled_users["user_key"].dropna().unique()
)

users_19k_set = set(
    all_users_19k["user_key"].dropna().unique()
)

tweet_user_set = set(
    tweets["user_key"].dropna().unique()
)

graph_record_user_set = set(
    followers_graph["user_key"].dropna().unique()
)


# ------------------------------------------------------------
# 4. Detect graph rows with usable follower/following information
# ------------------------------------------------------------

def has_graph_information(value):
    """
    True if follower/following cell contains usable information.

    Empty / missing representations are treated as unavailable.
    """
    if pd.isna(value):
        return False

    value = str(value).strip()

    invalid_values = {
        "",
        "[]",
        "{}",
        "nan",
        "none",
        "null",
        "<na>"
    }

    return value.lower() not in invalid_values


followers_graph["has_followers_info"] = (
    followers_graph["followers"]
    .apply(has_graph_information)
)

followers_graph["has_following_info"] = (
    followers_graph["following"]
    .apply(has_graph_information)
)

followers_graph["has_usable_graph"] = (
    followers_graph["has_followers_info"]
    |
    followers_graph["has_following_info"]
)


graph_usable_user_set = set(
    followers_graph.loc[
        followers_graph["has_usable_graph"],
        "user_key"
    ]
    .dropna()
    .unique()
)


# ------------------------------------------------------------
# 5. Exact overlap calculations
# ------------------------------------------------------------

labeled_in_19k = labeled_set & users_19k_set

labeled_with_tweets = (
    labeled_set
    & tweet_user_set
)

labeled_with_graph_record = (
    labeled_set
    & graph_record_user_set
)

labeled_with_usable_graph = (
    labeled_set
    & graph_usable_user_set
)

labeled_with_tweets_and_graph = (
    labeled_set
    & tweet_user_set
    & graph_usable_user_set
)

labeled_with_tweets_graph_and_19k = (
    labeled_set
    & tweet_user_set
    & graph_usable_user_set
    & users_19k_set
)


# ------------------------------------------------------------
# 6. Print overlap report
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("EXACT USER OVERLAP")
print("=" * 75)

print(f"Labeled users:                         {len(labeled_set)}")
print(f"19K unique users:                      {len(users_19k_set)}")
print(f"Users with tweets:                     {len(tweet_user_set)}")
print(f"Users with graph record:               {len(graph_record_user_set)}")
print(f"Users with usable graph information:   {len(graph_usable_user_set)}")

print("\n--- Labeled user coverage ---")

print(
    f"Labeled also present in 19K:           "
    f"{len(labeled_in_19k)}"
)

print(
    f"Labeled + Tweets:                      "
    f"{len(labeled_with_tweets)}"
)

print(
    f"Labeled + Graph record:                "
    f"{len(labeled_with_graph_record)}"
)

print(
    f"Labeled + Usable Graph:                "
    f"{len(labeled_with_usable_graph)}"
)

print(
    f"Labeled + Tweets + Usable Graph:       "
    f"{len(labeled_with_tweets_and_graph)}"
)

print(
    f"Labeled + Tweets + Graph + in 19K:     "
    f"{len(labeled_with_tweets_graph_and_19k)}"
)


# ------------------------------------------------------------
# 7. Build temporary complete-multimodal subset
# ------------------------------------------------------------

complete_multimodal_users = labeled_users[
    labeled_users["user_key"].isin(
        labeled_with_tweets_and_graph
    )
].copy()


print("\n" + "=" * 75)
print("LABEL DISTRIBUTION — LABELED + TWEETS + USABLE GRAPH")
print("=" * 75)

print(
    complete_multimodal_users[label_column]
    .value_counts(dropna=False)
)


# ------------------------------------------------------------
# 8. Binary Bot/Human subset inside complete multimodal users
# ------------------------------------------------------------

binary_labels = [
    "human(2)",
    "bot(1)"
]

complete_multimodal_binary = (
    complete_multimodal_users[
        complete_multimodal_users[label_column]
        .isin(binary_labels)
    ]
    .copy()
)


print("\n" + "=" * 75)
print("BOT/HUMAN COMPLETE-MULTIMODAL SUBSET")
print("=" * 75)

print(
    complete_multimodal_binary[label_column]
    .value_counts()
)

print(
    "\nTotal Bot/Human users with "
    "Label + Tweets + Usable Graph:",
    len(complete_multimodal_binary)
)


# ------------------------------------------------------------
# 9. Sanity check
# ------------------------------------------------------------

assert set(
    complete_multimodal_users["user_key"]
).issubset(labeled_set)

assert set(
    complete_multimodal_users["user_key"]
).issubset(tweet_user_set)

assert set(
    complete_multimodal_users["user_key"]
).issubset(graph_usable_user_set)

print("\nSanity checks passed.")

IDENTIFIER QUALITY CHECK
labeled_users        | rows =   1103 | unique users =   1103 | missing user_key =    0
all_users_19k        | rows =  19510 | unique users =  19435 | missing user_key =   75
followers_graph      | rows =   3453 | unique users =   3359 | missing user_key =   10
tweets               | rows =  75913 | unique users =   1099 | missing user_key =    0

EXACT USER OVERLAP
Labeled users:                         1103
19K unique users:                      19435
Users with tweets:                     1099
Users with graph record:               3359
Users with usable graph information:   3359

--- Labeled user coverage ---
Labeled also present in 19K:           1103
Labeled + Tweets:                      1099
Labeled + Graph record:                325
Labeled + Usable Graph:                325
Labeled + Tweets + Usable Graph:       324
Labeled + Tweets + Graph + in 19K:     324

LABEL DISTRIBUTION — LABELED + TWEETS + USABLE GRAPH
برچسب نهایی
human(2)         232
bot(1)  

In [6]:
# ============================================================
# Cell 4 — Verify Labeled 1K ↔ 19K Consistency
# ============================================================

# ------------------------------------------------------------
# 1. Check duplicate behavior before joining
# ------------------------------------------------------------

print("=" * 75)
print("DUPLICATE USER_KEY CHECK")
print("=" * 75)

for name, df in {
    "labeled_users": labeled_users,
    "all_users_19k": all_users_19k,
    "followers_graph": followers_graph,
}.items():

    valid = df[df["user_key"].notna()].copy()

    duplicate_rows = valid.duplicated(
        subset="user_key",
        keep=False
    ).sum()

    duplicate_users = (
        valid.loc[
            valid.duplicated("user_key", keep=False),
            "user_key"
        ]
        .nunique()
    )

    print(
        f"{name:20s} | "
        f"duplicate rows = {duplicate_rows:4d} | "
        f"duplicate users = {duplicate_users:4d}"
    )


# ------------------------------------------------------------
# 2. Build exact labeled / true-unlabeled partition
# ------------------------------------------------------------

valid_19k = all_users_19k[
    all_users_19k["user_key"].notna()
].copy()

true_unlabeled_19k = valid_19k[
    ~valid_19k["user_key"].isin(labeled_set)
].copy()


print("\n" + "=" * 75)
print("19K PARTITION")
print("=" * 75)

print(
    f"Valid unique users in 19K:    "
    f"{valid_19k['user_key'].nunique()}"
)

print(
    f"Labeled users inside 19K:     "
    f"{len(labeled_in_19k)}"
)

print(
    f"True unlabeled unique users:  "
    f"{true_unlabeled_19k['user_key'].nunique()}"
)


# ------------------------------------------------------------
# 3. Ensure zero overlap
# ------------------------------------------------------------

true_unlabeled_set = set(
    true_unlabeled_19k["user_key"].unique()
)

intersection = labeled_set & true_unlabeled_set

print(
    f"Labeled ∩ True Unlabeled:     "
    f"{len(intersection)}"
)

assert len(intersection) == 0


# ------------------------------------------------------------
# 4. Find columns shared between labeled 1K and 19K
# ------------------------------------------------------------

excluded_columns = {
    "user_key",
    label_column
}

shared_columns = sorted(
    (
        set(labeled_users.columns)
        & set(all_users_19k.columns)
    )
    - excluded_columns
)

print("\n" + "=" * 75)
print("SHARED COLUMNS BETWEEN 1K AND 19K")
print("=" * 75)

print(f"Number of shared columns: {len(shared_columns)}")

print("\nShared columns:")
print(shared_columns)


# ------------------------------------------------------------
# 5. Merge the 1103 overlapping users for consistency analysis
# ------------------------------------------------------------

comparison = labeled_users[
    ["user_key"] + shared_columns
].merge(
    valid_19k[
        ["user_key"] + shared_columns
    ],
    on="user_key",
    how="inner",
    suffixes=("_1k", "_19k"),
    validate="one_to_one"
)

print("\nMatched users:", len(comparison))


# ------------------------------------------------------------
# 6. Compare missingness and exact agreement per shared feature
# ------------------------------------------------------------

comparison_report = []

for col in shared_columns:

    col_1k = f"{col}_1k"
    col_19k = f"{col}_19k"

    s1 = comparison[col_1k]
    s2 = comparison[col_19k]

    both_present = s1.notna() & s2.notna()

    comparable_count = int(
        both_present.sum()
    )

    # Convert to normalized strings only for equality comparison.
    # We are not modifying the original values.
    a = (
        s1[both_present]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    b = (
        s2[both_present]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    agreement_count = int(
        (a == b).sum()
    )

    if comparable_count > 0:
        agreement_rate = (
            agreement_count
            / comparable_count
            * 100
        )
    else:
        agreement_rate = np.nan

    comparison_report.append({
        "feature": col,

        "missing_1k": int(
            s1.isna().sum()
        ),

        "missing_19k": int(
            s2.isna().sum()
        ),

        "both_present": comparable_count,

        "exact_agreement_count":
            agreement_count,

        "exact_agreement_pct":
            agreement_rate
    })


comparison_report = pd.DataFrame(
    comparison_report
).sort_values(
    by="exact_agreement_pct",
    ascending=True,
    na_position="last"
)


# ------------------------------------------------------------
# 7. Display features with lowest agreement first
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("1K ↔ 19K FEATURE CONSISTENCY")
print("=" * 75)

display(
    comparison_report.head(30)
)


print("\n" + "=" * 75)
print("CHECK PASSED")
print("=" * 75)

print(
    "The labeled and true-unlabeled populations "
    "are completely separated."
)

DUPLICATE USER_KEY CHECK
labeled_users        | duplicate rows =    0 | duplicate users =    0
all_users_19k        | duplicate rows =    0 | duplicate users =    0
followers_graph      | duplicate rows =  161 | duplicate users =   77

19K PARTITION
Valid unique users in 19K:    19435
Labeled users inside 19K:     1103
True unlabeled unique users:  18332
Labeled ∩ True Unlabeled:     0

SHARED COLUMNS BETWEEN 1K AND 19K
Number of shared columns: 68

Shared columns:
['Botometer Score', 'Grok 4', 'LLM(ChatGPT)', 'LLM(Gemini)', 'clean_description', 'cluster_id', 'created_at', 'default_profile', 'default_profile_image', 'description', 'fast_followers_count', 'favourites_count', 'follower_growth_rate', 'followers_count', 'following', 'friends_count', 'friends_growth_rate', 'has_custom_timelines', 'hashtag_in_description', 'id', 'is_translator', 'listed_count', 'location', 'max_occurence_of_same_gap', 'max_tweets_per_day', 'max_tweets_per_hour', 'mean_favourites_per_tweet', 'mean_no_hashtags

,feature,missing_1k,missing_19k,both_present,exact_agreement_count,exact_agreement_pct
7,default_profile,0,0,1103,0,0.000000
14,following,395,395,708,0,0.000000
11,favourites_count,0,0,1103,0,0.000000
15,friends_count,0,2,1101,0,0.000000
13,followers_count,0,2,1101,0,0.000000
10,fast_followers_count,0,0,1103,0,0.000000
32,media_count,0,0,1103,0,0.000000
21,listed_count,0,0,1103,0,0.000000
51,statuses_count,0,0,1103,0,0.000000
58,verified,0,0,1103,0,0.000000



CHECK PASSED
The labeled and true-unlabeled populations are completely separated.


In [7]:
# ============================================================
# Cell 5 — Type-Aware 1K ↔ 19K Feature Consistency
# ============================================================

# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def normalize_boolean_value(x):
    """
    Normalize common boolean representations to 0 / 1.
    """
    if pd.isna(x):
        return np.nan

    s = str(x).strip().lower()

    true_values = {
        "true", "1", "1.0", "yes", "y"
    }

    false_values = {
        "false", "0", "0.0", "no", "n"
    }

    if s in true_values:
        return 1

    if s in false_values:
        return 0

    return np.nan


def normalized_string_series(series):
    """
    Normalize text only for comparison.
    Does not modify original data.
    """
    return (
        series.astype("string")
        .str.strip()
        .str.lower()
    )


def numeric_conversion_rate(series):
    """
    Fraction of non-missing values that can be converted to numeric.
    """
    non_missing = series.dropna()

    if len(non_missing) == 0:
        return 0.0

    numeric = pd.to_numeric(
        non_missing,
        errors="coerce"
    )

    return numeric.notna().mean()


def boolean_conversion_rate(series):
    """
    Fraction of non-missing values that look boolean.
    """
    non_missing = series.dropna()

    if len(non_missing) == 0:
        return 0.0

    converted = non_missing.apply(
        normalize_boolean_value
    )

    return converted.notna().mean()


# ------------------------------------------------------------
# Type-aware comparison
# ------------------------------------------------------------

semantic_report = []

for col in shared_columns:

    s1 = comparison[f"{col}_1k"]
    s2 = comparison[f"{col}_19k"]

    both_present = (
        s1.notna()
        & s2.notna()
    )

    n = int(both_present.sum())

    if n == 0:
        semantic_report.append({
            "feature": col,
            "comparison_type": "no comparable data",
            "both_present": 0,
            "agreement_count": 0,
            "agreement_pct": np.nan,
            "missing_1k": int(s1.isna().sum()),
            "missing_19k": int(s2.isna().sum())
        })

        continue

    a = s1[both_present]
    b = s2[both_present]

    # --------------------------------------------------------
    # Detect whether feature behaves like boolean
    # --------------------------------------------------------

    bool_rate_a = boolean_conversion_rate(a)
    bool_rate_b = boolean_conversion_rate(b)

    num_rate_a = numeric_conversion_rate(a)
    num_rate_b = numeric_conversion_rate(b)

    if (
        bool_rate_a >= 0.95
        and bool_rate_b >= 0.95
    ):

        comparison_type = "boolean"

        aa = a.apply(
            normalize_boolean_value
        )

        bb = b.apply(
            normalize_boolean_value
        )

        matches = (
            aa.values == bb.values
        )

    # --------------------------------------------------------
    # Numeric comparison
    # --------------------------------------------------------

    elif (
        num_rate_a >= 0.95
        and num_rate_b >= 0.95
    ):

        comparison_type = "numeric"

        aa = pd.to_numeric(
            a,
            errors="coerce"
        )

        bb = pd.to_numeric(
            b,
            errors="coerce"
        )

        matches = np.isclose(
            aa.values,
            bb.values,
            rtol=1e-6,
            atol=1e-8,
            equal_nan=True
        )

    # --------------------------------------------------------
    # Text comparison
    # --------------------------------------------------------

    else:

        comparison_type = "text"

        aa = normalized_string_series(a)
        bb = normalized_string_series(b)

        matches = (
            aa.values == bb.values
        )

    agreement_count = int(
        np.sum(matches)
    )

    agreement_pct = (
        agreement_count
        / n
        * 100
    )

    semantic_report.append({
        "feature": col,
        "comparison_type": comparison_type,
        "both_present": n,
        "agreement_count": agreement_count,
        "agreement_pct": agreement_pct,
        "missing_1k": int(s1.isna().sum()),
        "missing_19k": int(s2.isna().sum())
    })


semantic_report = pd.DataFrame(
    semantic_report
).sort_values(
    by="agreement_pct",
    ascending=True,
    na_position="last"
)


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("=" * 75)
print("TYPE-AWARE 1K ↔ 19K CONSISTENCY")
print("=" * 75)

display(
    semantic_report.head(30)
)


# ------------------------------------------------------------
# High-agreement features
# ------------------------------------------------------------

high_agreement = semantic_report[
    semantic_report["agreement_pct"] >= 99
]

print("\n" + "=" * 75)
print("FEATURES WITH >= 99% AGREEMENT")
print("=" * 75)

print(
    f"{len(high_agreement)} / "
    f"{len(semantic_report)} shared features"
)


# ------------------------------------------------------------
# Low-agreement features
# ------------------------------------------------------------

low_agreement = semantic_report[
    semantic_report["agreement_pct"] < 95
]

print("\n" + "=" * 75)
print("FEATURES WITH < 95% AGREEMENT")
print("=" * 75)

display(
    low_agreement[
        [
            "feature",
            "comparison_type",
            "both_present",
            "agreement_pct",
            "missing_1k",
            "missing_19k"
        ]
    ]
)

TYPE-AWARE 1K ↔ 19K CONSISTENCY


,feature,comparison_type,both_present,agreement_count,agreement_pct,missing_1k,missing_19k
67,آیا پروفایل واقعی به‌نظر می‌رسد؟عکس واقعی، بیو...,boolean,68,67,98.529412,1035,1015
61,آیا تعامل با دیگران وجود دارد؟ریپلای، منشن، گف...,boolean,69,68,98.550725,1034,1014
66,آیا محتوای توییت‌ها خودکار به‌نظر می‌رسد؟تکرار...,boolean,70,69,98.571429,1033,1013
64,آیا رفتار زمانی توییت‌ها طبیعی است؟ارسال نامنظ...,boolean,70,69,98.571429,1033,1013
65,آیا فرکانس استفاده از یک هشتگ زیاد است,boolean,129,128,99.224806,974,935
22,location,text,639,635,99.374022,464,464
62,آیا توییت‌ها متنوع و طبیعی‌اند؟زبان انسانی، نظ...,boolean,172,171,99.418605,931,876
35,name,text,1103,1101,99.818676,0,0
15,friends_count,numeric,1101,1100,99.909173,0,2
13,followers_count,numeric,1101,1100,99.909173,0,2



FEATURES WITH >= 99% AGREEMENT
63 / 68 shared features

FEATURES WITH < 95% AGREEMENT


,feature,comparison_type,both_present,agreement_pct,missing_1k,missing_19k


In [8]:
# ============================================================
# Cell 6 — Column Provenance & Leakage Audit
# ============================================================

# ------------------------------------------------------------
# 1. Determine column provenance
# ------------------------------------------------------------

base_exclusions = {"user_key"}

columns_1k = set(labeled_users.columns) - base_exclusions
columns_19k = set(all_users_19k.columns) - base_exclusions

only_1k_columns = sorted(
    columns_1k - columns_19k
)

only_19k_columns = sorted(
    columns_19k - columns_1k
)

shared_columns_full = sorted(
    columns_1k & columns_19k
)


print("=" * 80)
print("COLUMN PROVENANCE")
print("=" * 80)

print(f"Columns only in labeled 1K: {len(only_1k_columns)}")
print(f"Columns only in 19K:        {len(only_19k_columns)}")
print(f"Shared columns:             {len(shared_columns_full)}")


# ------------------------------------------------------------
# 2. Display columns existing only in labeled 1K
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("COLUMNS ONLY IN LABELED 1K")
print("=" * 80)

only_1k_report = []

for col in only_1k_columns:

    only_1k_report.append({
        "column": col,
        "dtype": str(labeled_users[col].dtype),
        "missing_count": int(
            labeled_users[col].isna().sum()
        ),
        "missing_pct": round(
            labeled_users[col].isna().mean() * 100,
            2
        ),
        "n_unique": int(
            labeled_users[col].nunique(
                dropna=True
            )
        )
    })


only_1k_report = pd.DataFrame(
    only_1k_report
)

display(only_1k_report)


# ------------------------------------------------------------
# 3. Display columns existing only in 19K
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("COLUMNS ONLY IN 19K")
print("=" * 80)

only_19k_report = []

for col in only_19k_columns:

    only_19k_report.append({
        "column": col,
        "dtype": str(all_users_19k[col].dtype),
        "missing_count": int(
            all_users_19k[col].isna().sum()
        ),
        "missing_pct": round(
            all_users_19k[col].isna().mean() * 100,
            2
        ),
        "n_unique": int(
            all_users_19k[col].nunique(
                dropna=True
            )
        )
    })


only_19k_report = pd.DataFrame(
    only_19k_report
)

display(only_19k_report)


# ------------------------------------------------------------
# 4. Automatically flag suspicious / leakage-prone columns
# ------------------------------------------------------------

all_candidate_columns = sorted(
    set(labeled_users.columns)
    | set(all_users_19k.columns)
)

leakage_keywords = [
    "label",
    "class",
    "llm",
    "chatgpt",
    "gemini",
    "grok",
    "tagger",
    "prob",
    "reason",
    "botometer",
    "برچسب"
]

manual_review_prefixes = [
    "آیا "
]


def is_suspicious_column(col):

    col_lower = str(col).lower()

    keyword_match = any(
        keyword in col_lower
        for keyword in leakage_keywords
    )

    manual_match = any(
        str(col).startswith(prefix)
        for prefix in manual_review_prefixes
    )

    unnamed_match = str(col).lower().startswith(
        "unnamed"
    )

    return (
        keyword_match
        or manual_match
        or unnamed_match
    )


suspected_leakage_columns = [
    col
    for col in all_candidate_columns
    if is_suspicious_column(col)
]


print("\n" + "=" * 80)
print("SUSPECTED LEAKAGE / MANUAL-JUDGMENT COLUMNS")
print("=" * 80)

for i, col in enumerate(
    suspected_leakage_columns,
    start=1
):
    print(f"{i:02d}. {col}")


# ------------------------------------------------------------
# 5. Define columns that must NEVER be direct model inputs
# ------------------------------------------------------------

hard_exclude_columns = set(
    suspected_leakage_columns
)

hard_exclude_columns.update({
    label_column,
    "id",
    "screen_name",
    "user_key"
})


print("\n" + "=" * 80)
print("HARD-EXCLUDE COLUMN COUNT")
print("=" * 80)

print(
    len(hard_exclude_columns),
    "columns currently marked as non-model inputs."
)


# ------------------------------------------------------------
# 6. Important note
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("NOTE")
print("=" * 80)

print("""
This cell does NOT delete any column.

It only identifies:
1) columns unique to the labeled dataset,
2) columns unique to the 19K master pool,
3) potential target leakage / external detector outputs,
4) identifiers that must not be used as predictive features.

Final feature selection will be done explicitly in the next cells.
""")

COLUMN PROVENANCE
Columns only in labeled 1K: 33
Columns only in 19K:        7
Shared columns:             68

COLUMNS ONLY IN LABELED 1K


,column,dtype,missing_count,missing_pct,n_unique
0,Probe2,float64,0,0.00,11
1,Probe3,float64,0,0.00,11
2,Reason2,str,19,1.72,913
3,Reason3,str,28,2.54,733
4,Unnamed: 58,float64,1103,100.00,0
5,"class (1:bot, 2:human, 3:News Agent, 4:unverif...",float64,1,0.09,5
6,"class (1:bot, 2:human, 3:News Agent, 4:unverif...",int64,0,0.00,4
7,"class (1:bot, 2:human, 3:News Agent, 4:unverif...",int64,0,0.00,4
8,description_length,int64,0,0.00,164
9,followers_friend_ratio,float64,0,0.00,1103



COLUMNS ONLY IN 19K


,column,dtype,missing_count,missing_pct,n_unique
0,last_updated,str,76,0.39,19273
1,prob 2,float64,19009,97.43,20
2,prob1,str,17584,90.13,15
3,reason 1,str,17629,90.36,1505
4,reason 2,float64,19510,100.00,0
5,tagger's name 1,str,17466,89.52,9
6,tagger's name2,str,19009,97.43,1



SUSPECTED LEAKAGE / MANUAL-JUDGMENT COLUMNS
01. Botometer Score
02. Grok 4
03. LLM(ChatGPT)
04. LLM(Gemini)
05. Probe2
06. Probe3
07. Reason2
08. Reason3
09. Unnamed: 58
10. class (1:bot, 2:human, 3:News Agent, 4:unverified)
11. class (1:bot, 2:human, 3:News Agent, 4:unverified)_1
12. class (1:bot, 2:human, 3:News Agent, 4:unverified)_2
13. prob 2
14. prob1
15. reason 1
16. reason 2
17. tagger's name 1
18. tagger's name 2
19. tagger's name 3
20. tagger's name2
21. آیا تعامل با دیگران وجود دارد؟ریپلای، منشن، گفت‌وگو با کاربران (ویژگی‌های رفتاری در مکالمه (Conversational Features)
•        پاسخ‌دهی کم: حسابی که تقریباً هیچوقت جواب مستقیم نمی‌دهد.
•        پاسخ‌های ماشینی: جملات کلیشه‌ای و تکراری مثل "Great post!" یا "Nice info"
•        تأخیر یکنواخت در پاسخ: مثلاً همیشه ۳ ثانیه بعد جواب می‌دهد.
•        نبود تعامل شخصی: اشاره نکردن به تجربیات فردی یا مکالمات واقعی.)
22. آیا توییت‌ها متنوع و طبیعی‌اند؟زبان انسانی، نظر شخصی، طنز، تعامل
23. آیا دنبال‌شوندگان/دنبال‌کننده‌ها غیرعادی‌اند؟نسب

In [9]:
# ============================================================
# Cell 7 — Define Canonical Feature Groups
# ============================================================

# ============================================================
# IMPORTANT DESIGN DECISION
# ============================================================
#
# 19K will be the canonical source of user features because:
#   1) all 1103 labeled users exist inside it,
#   2) shared features are highly consistent,
#   3) it has fewer missing values for several features,
#   4) labeled and unlabeled users will therefore use the
#      same feature definitions and data snapshot.
#
# Ground-truth labels will still come ONLY from the 1K dataset.
# ============================================================


# ------------------------------------------------------------
# 1. Identifier / target columns
# ------------------------------------------------------------

IDENTIFIER_COLUMNS = [
    "id",
    "screen_name",
    "user_key"
]

TARGET_COLUMN = label_column


# ------------------------------------------------------------
# 2. External detector / leakage columns
# NEVER use these as model inputs
# ------------------------------------------------------------

EXTERNAL_DETECTOR_COLUMNS = [
    "Botometer Score",
    "LLM(ChatGPT)",
    "LLM(Gemini)",
    "Grok 4"
]


# ------------------------------------------------------------
# 3. Columns from annotation / taggers / manual judgments
# ------------------------------------------------------------

ANNOTATION_KEYWORDS = [
    "tagger",
    "prob",
    "reason",
    "probe",
    "class",
    "برچسب"
]


def is_annotation_column(col):

    c = str(col).strip()
    c_lower = c.lower()

    # LLM/manual questionnaire columns
    if c.startswith("آیا "):
        return True

    # tagger / probability / reason / class columns
    if any(
        keyword in c_lower
        for keyword in ANNOTATION_KEYWORDS
    ):
        return True

    return False


ANNOTATION_COLUMNS = [
    col
    for col in all_users_19k.columns
    if is_annotation_column(col)
]


# ------------------------------------------------------------
# 4. Raw fields retained for reference but NOT used directly
#    as tabular model features
# ------------------------------------------------------------

RAW_REFERENCE_COLUMNS = [
    "created_at",              # user_age is preferable
    "location",                # high-cardinality free text
    "name",                    # derive numeric characteristics instead
    "description",             # used only for text processing
    "clean_description",       # used only for text processing
    "url",
    "profile_banner_url",
    "profile_image_url_https",
    "pinned_tweet_ids_str",
    "cluster_id",              # dataset/community artifact
    "following",               # API account relation, NOT graph adjacency
    "last_updated"
]


# ------------------------------------------------------------
# 5. Very sparse / unsuitable raw profile fields
# ------------------------------------------------------------

SPARSE_OR_UNSTABLE_COLUMNS = [
    "profile_interstitial_type",
    "verified_type"
]


# ------------------------------------------------------------
# 6. Profile numerical features
# ------------------------------------------------------------

PROFILE_NUMERIC_FEATURES = [
    "followers_count",
    "friends_count",
    "favourites_count",
    "listed_count",
    "media_count",
    "statuses_count",
    "status_count",
    "fast_followers_count",
    "normal_followers_count",
    "user_age",
    "follower_growth_rate",
    "friends_growth_rate"
]


# ------------------------------------------------------------
# 7. Profile binary / categorical features
# ------------------------------------------------------------

PROFILE_BINARY_FEATURES = [
    "default_profile",
    "default_profile_image",
    "verified",
    "has_custom_timelines",
    "is_translator",
    "possibly_sensitive",
    "want_retweets",
    "hashtag_in_description",
    "numbers_in_description"
]


PROFILE_CATEGORICAL_FEATURES = [
    "translator_type"
]


# ------------------------------------------------------------
# 8. Behavioral / temporal features
# ------------------------------------------------------------

BEHAVIOR_TEMPORAL_FEATURES = [
    "no_type_tweet",
    "no_type_retweet_with_comment",
    "no_type_reply",

    "mean_no_media_per_tweet",
    "mean_no_words",
    "no_languages",
    "mean_no_hashtags",
    "mean_favourites_per_tweet",

    "time_between_tweets",
    "tweet_frequency",

    "min_tweets_per_hour",
    "min_tweets_per_day",
    "max_tweets_per_hour",
    "max_tweets_per_day",

    "max_occurence_of_same_gap",

    "unique_mention_rate_per_tweet",
    "mean_user_mentions_per_tweet",

    "retweet_as_tweet_rate",
    "no_retweet_tweets",
    "mean_retweets_per_tweet"
]


# ------------------------------------------------------------
# 9. Text modality fields
# ------------------------------------------------------------

TEXT_FEATURES = [
    "clean_description",
    "description"
]

# Tweet text itself remains in the separate tweets dataframe.
# It will NOT be flattened into the master user table.


# ------------------------------------------------------------
# 10. Reproducible engineered features
# These existed only in 1K, but we will recreate them
# ourselves for BOTH labeled and unlabeled users.
# ------------------------------------------------------------

DERIVED_FEATURES_TO_CREATE = [
    "description_length",
    "followers_friend_ratio",
    "listed_growth_rate",
    "num_digits_in_name",
    "num_digits_in_username",
    "url_in_description"
]


# ------------------------------------------------------------
# 11. 1K-only features NOT used directly
# ------------------------------------------------------------

SAFE_REDERIVABLE_1K_COLUMNS = set(
    DERIVED_FEATURES_TO_CREATE
)

ONE_K_ONLY_NOT_DIRECTLY_USED = sorted(
    set(only_1k_columns)
    - SAFE_REDERIVABLE_1K_COLUMNS
)


# ------------------------------------------------------------
# 12. Complete candidate feature collection
# ------------------------------------------------------------

BASE_TABULAR_FEATURES = (
    PROFILE_NUMERIC_FEATURES
    + PROFILE_BINARY_FEATURES
    + PROFILE_CATEGORICAL_FEATURES
    + BEHAVIOR_TEMPORAL_FEATURES
)


# ------------------------------------------------------------
# 13. Validate that expected features exist in 19K
# ------------------------------------------------------------

expected_columns = set(
    BASE_TABULAR_FEATURES
    + TEXT_FEATURES
)

missing_expected_columns = sorted(
    expected_columns
    - set(all_users_19k.columns)
)

print("=" * 80)
print("FEATURE GROUP VALIDATION")
print("=" * 80)

print(
    "Profile numeric features:      ",
    len(PROFILE_NUMERIC_FEATURES)
)

print(
    "Profile binary features:       ",
    len(PROFILE_BINARY_FEATURES)
)

print(
    "Profile categorical features:  ",
    len(PROFILE_CATEGORICAL_FEATURES)
)

print(
    "Behavior/temporal features:    ",
    len(BEHAVIOR_TEMPORAL_FEATURES)
)

print(
    "Base tabular features total:   ",
    len(BASE_TABULAR_FEATURES)
)

print(
    "Derived features to create:    ",
    len(DERIVED_FEATURES_TO_CREATE)
)

print(
    "Text fields:                   ",
    len(TEXT_FEATURES)
)


print("\nMissing expected columns:")

if missing_expected_columns:
    print(missing_expected_columns)
else:
    print("None")


assert len(missing_expected_columns) == 0


# ------------------------------------------------------------
# 14. Check accidental leakage
# ------------------------------------------------------------

selected_model_columns = set(
    BASE_TABULAR_FEATURES
)

forbidden_columns = (
    set(EXTERNAL_DETECTOR_COLUMNS)
    | set(ANNOTATION_COLUMNS)
    | set(IDENTIFIER_COLUMNS)
    | set(SPARSE_OR_UNSTABLE_COLUMNS)
)

leakage_overlap = (
    selected_model_columns
    & forbidden_columns
)

print("\n" + "=" * 80)
print("LEAKAGE CHECK")
print("=" * 80)

print(
    "Forbidden columns accidentally selected:",
    leakage_overlap
)

assert len(leakage_overlap) == 0


# ------------------------------------------------------------
# 15. Missingness summary for candidate tabular features
# ------------------------------------------------------------

feature_missingness = pd.DataFrame({
    "feature": BASE_TABULAR_FEATURES,
    "missing_count": [
        all_users_19k[col].isna().sum()
        for col in BASE_TABULAR_FEATURES
    ],
    "missing_pct": [
        all_users_19k[col].isna().mean() * 100
        for col in BASE_TABULAR_FEATURES
    ],
    "dtype": [
        str(all_users_19k[col].dtype)
        for col in BASE_TABULAR_FEATURES
    ]
}).sort_values(
    "missing_pct",
    ascending=False
)


print("\n" + "=" * 80)
print("CANDIDATE FEATURE MISSINGNESS — 19K MASTER POOL")
print("=" * 80)

display(feature_missingness)


print("\nFeature policy successfully defined.")

FEATURE GROUP VALIDATION
Profile numeric features:       12
Profile binary features:        9
Profile categorical features:   1
Behavior/temporal features:     20
Base tabular features total:    42
Derived features to create:     6
Text fields:                    2

Missing expected columns:
None

LEAKAGE CHECK
Forbidden columns accidentally selected: set()

CANDIDATE FEATURE MISSINGNESS — 19K MASTER POOL


,feature,missing_count,missing_pct,dtype
7,fast_followers_count,76,0.389544,float64
5,statuses_count,76,0.389544,float64
8,normal_followers_count,76,0.389544,float64
15,has_custom_timelines,76,0.389544,object
17,possibly_sensitive,76,0.389544,object
21,translator_type,76,0.389544,str
18,want_retweets,76,0.389544,object
16,is_translator,76,0.389544,object
13,default_profile_image,73,0.374167,object
0,followers_count,5,0.025628,float64



Feature policy successfully defined.


In [10]:
# ============================================================
# Cell 8 — Recreate & Validate Reproducible Derived Features
# ============================================================

# ------------------------------------------------------------
# 1. Start canonical working user table from valid 19K users
# ------------------------------------------------------------

users_master = valid_19k.copy()

print("=" * 80)
print("CANONICAL USER TABLE")
print("=" * 80)

print("Rows:", len(users_master))
print(
    "Unique users:",
    users_master["user_key"].nunique()
)

assert len(users_master) == 19435
assert users_master["user_key"].is_unique


# ------------------------------------------------------------
# 2. Helper — count Unicode digits
# ------------------------------------------------------------

def count_digits(value):

    if pd.isna(value):
        return 0

    return sum(
        char.isdigit()
        for char in str(value)
    )


# ------------------------------------------------------------
# 3. description_length
# Same definition as the existing 1K feature:
# number of characters in raw description
# ------------------------------------------------------------

users_master["description_length"] = (
    users_master["description"]
    .fillna("")
    .astype(str)
    .str.len()
)


# ------------------------------------------------------------
# 4. Number of digits in display name
# Unicode-aware: captures 4, ۴, ⁴, etc.
# ------------------------------------------------------------

users_master["num_digits_in_name"] = (
    users_master["name"]
    .apply(count_digits)
)


# ------------------------------------------------------------
# 5. Number of digits in username
# ------------------------------------------------------------

users_master["num_digits_in_username"] = (
    users_master["screen_name"]
    .apply(count_digits)
)


# ------------------------------------------------------------
# 6. URL in description
# ------------------------------------------------------------

users_master["url_in_description"] = (
    users_master["description"]
    .fillna("")
    .astype(str)
    .str.contains(
        r"https?://|www\.",
        case=False,
        regex=True
    )
    .astype(int)
)


# ------------------------------------------------------------
# 7. Followers / friends ratio
#
# Existing 1K behavior:
# followers_count / friends_count
#
# If friends_count == 0:
# use followers_count instead of infinity.
# ------------------------------------------------------------

followers = pd.to_numeric(
    users_master["followers_count"],
    errors="coerce"
)

friends = pd.to_numeric(
    users_master["friends_count"],
    errors="coerce"
)

users_master["followers_friend_ratio"] = np.where(
    friends > 0,
    followers / friends,
    followers
)


# ------------------------------------------------------------
# 8. Final reproducible derived feature list
# ------------------------------------------------------------

DERIVED_FEATURES = [
    "description_length",
    "followers_friend_ratio",
    "num_digits_in_name",
    "num_digits_in_username",
    "url_in_description"
]


print("\nDerived features created:")

for feature in DERIVED_FEATURES:
    print(" -", feature)


# ------------------------------------------------------------
# 9. Validate against existing values in labeled 1K
# ------------------------------------------------------------

validation_columns = [
    "user_key"
] + DERIVED_FEATURES

derived_validation = labeled_users[
    validation_columns
].merge(
    users_master[
        validation_columns
    ],
    on="user_key",
    how="inner",
    suffixes=("_1k", "_new"),
    validate="one_to_one"
)


validation_results = []

for feature in DERIVED_FEATURES:

    old = derived_validation[
        f"{feature}_1k"
    ]

    new = derived_validation[
        f"{feature}_new"
    ]

    valid = old.notna() & new.notna()

    if feature == "followers_friend_ratio":

        matches = np.isclose(
            pd.to_numeric(
                old[valid],
                errors="coerce"
            ),
            pd.to_numeric(
                new[valid],
                errors="coerce"
            ),
            rtol=1e-5,
            atol=1e-6
        )

    else:

        matches = (
            old[valid].values
            ==
            new[valid].values
        )

    agreement = (
        matches.mean() * 100
        if len(matches) > 0
        else np.nan
    )

    validation_results.append({
        "feature": feature,
        "compared_users": int(valid.sum()),
        "agreement_pct": agreement
    })


derived_validation_report = pd.DataFrame(
    validation_results
)


print("\n" + "=" * 80)
print("DERIVED FEATURE VALIDATION AGAINST 1K")
print("=" * 80)

display(
    derived_validation_report
)


# ------------------------------------------------------------
# 10. Final shared tabular feature set
# ------------------------------------------------------------

FINAL_TABULAR_FEATURES = (
    BASE_TABULAR_FEATURES
    + DERIVED_FEATURES
)


print("\n" + "=" * 80)
print("FINAL SHARED TABULAR FEATURE SPACE")
print("=" * 80)

print(
    "Base features:",
    len(BASE_TABULAR_FEATURES)
)

print(
    "Derived features:",
    len(DERIVED_FEATURES)
)

print(
    "Total tabular features:",
    len(FINAL_TABULAR_FEATURES)
)


# ------------------------------------------------------------
# 11. Safety checks
# ------------------------------------------------------------

assert all(
    feature in users_master.columns
    for feature in FINAL_TABULAR_FEATURES
)

assert (
    users_master["user_key"]
    .nunique()
    ==
    len(users_master)
)

print("\nDerived feature construction completed successfully.")

CANONICAL USER TABLE
Rows: 19435
Unique users: 19435

Derived features created:
 - description_length
 - followers_friend_ratio
 - num_digits_in_name
 - num_digits_in_username
 - url_in_description

DERIVED FEATURE VALIDATION AGAINST 1K


,feature,compared_users,agreement_pct
0,description_length,1103,100.000000
1,followers_friend_ratio,1101,99.909173
2,num_digits_in_name,1103,100.000000
3,num_digits_in_username,1103,100.000000
4,url_in_description,1103,100.000000



FINAL SHARED TABULAR FEATURE SPACE
Base features: 42
Derived features: 5
Total tabular features: 47

Derived feature construction completed successfully.


In [11]:
# ============================================================
# Cell 9 — Build Canonical Labeled & True-Unlabeled Master Tables
# ============================================================

# ------------------------------------------------------------
# 1. Columns to retain in canonical master tables
# ------------------------------------------------------------

MASTER_REFERENCE_COLUMNS = [
    "id",
    "screen_name",
    "user_key",

    # raw text / reference information
    "name",
    "description",
    "clean_description",
    "created_at"
]

MASTER_FEATURE_COLUMNS = (
    FINAL_TABULAR_FEATURES
)

MASTER_COLUMNS = list(dict.fromkeys(
    MASTER_REFERENCE_COLUMNS
    + MASTER_FEATURE_COLUMNS
))


# ------------------------------------------------------------
# 2. Build clean label table from labeled 1K
# ------------------------------------------------------------

label_lookup = labeled_users[
    [
        "user_key",
        label_column
    ]
].copy()

label_lookup = label_lookup.rename(
    columns={
        label_column: "label_raw"
    }
)


# ------------------------------------------------------------
# 3. Create binary label
#
# human = 0
# bot   = 1
#
# Other classes remain NaN and will NOT be used in the
# main binary supervised task.
# ------------------------------------------------------------

binary_label_map = {
    "human(2)": 0,
    "bot(1)": 1
}

label_lookup["label_binary"] = (
    label_lookup["label_raw"]
    .map(binary_label_map)
)


# ------------------------------------------------------------
# 4. Build labeled master from canonical 19K user table
# ------------------------------------------------------------

labeled_master = (
    users_master[
        users_master["user_key"].isin(
            labeled_set
        )
    ][MASTER_COLUMNS]
    .merge(
        label_lookup,
        on="user_key",
        how="left",
        validate="one_to_one"
    )
    .copy()
)


# ------------------------------------------------------------
# 5. Build true-unlabeled master
# ------------------------------------------------------------

true_unlabeled_master = (
    users_master[
        ~users_master["user_key"].isin(
            labeled_set
        )
    ][MASTER_COLUMNS]
    .copy()
)

true_unlabeled_master["label_raw"] = pd.NA
true_unlabeled_master["label_binary"] = pd.NA


# ------------------------------------------------------------
# 6. Add modality availability masks
# ------------------------------------------------------------

def add_modality_masks(df):

    df = df.copy()

    # -------------------------
    # Profile modality
    # -------------------------

    profile_check_cols = [
        "followers_count",
        "friends_count",
        "user_age"
    ]

    df["has_profile"] = (
        df[profile_check_cols]
        .notna()
        .any(axis=1)
        .astype(int)
    )

    # -------------------------
    # Raw tweet modality
    # -------------------------

    df["has_raw_tweets"] = (
        df["user_key"]
        .isin(tweet_user_set)
        .astype(int)
    )

    # -------------------------
    # Description text
    # -------------------------

    clean_desc_present = (
        df["clean_description"]
        .fillna("")
        .astype(str)
        .str.strip()
        .ne("")
    )

    raw_desc_present = (
        df["description"]
        .fillna("")
        .astype(str)
        .str.strip()
        .ne("")
    )

    df["has_description"] = (
        clean_desc_present
        | raw_desc_present
    ).astype(int)

    # -------------------------
    # Behavioral / temporal modality
    # -------------------------

    df["has_behavior_temporal"] = (
        df[BEHAVIOR_TEMPORAL_FEATURES]
        .notna()
        .all(axis=1)
        .astype(int)
    )

    # -------------------------
    # Graph modality
    # -------------------------

    df["has_graph"] = (
        df["user_key"]
        .isin(graph_usable_user_set)
        .astype(int)
    )

    return df


labeled_master = add_modality_masks(
    labeled_master
)

true_unlabeled_master = add_modality_masks(
    true_unlabeled_master
)


# ------------------------------------------------------------
# 7. Build binary supervised dataset
# ------------------------------------------------------------

labeled_binary_master = (
    labeled_master[
        labeled_master["label_binary"]
        .notna()
    ]
    .copy()
)

labeled_binary_master[
    "label_binary"
] = labeled_binary_master[
    "label_binary"
].astype(int)


# ------------------------------------------------------------
# 8. Build complete multimodal binary subset
#
# Required:
#   binary label
#   profile
#   raw tweets
#   temporal/behavioral
#   graph
# ------------------------------------------------------------

complete_multimodal_binary = (
    labeled_binary_master[
        (
            labeled_binary_master["has_profile"] == 1
        )
        &
        (
            labeled_binary_master["has_raw_tweets"] == 1
        )
        &
        (
            labeled_binary_master["has_behavior_temporal"] == 1
        )
        &
        (
            labeled_binary_master["has_graph"] == 1
        )
    ]
    .copy()
)


# ------------------------------------------------------------
# 9. Report dataset sizes
# ------------------------------------------------------------

print("=" * 80)
print("CANONICAL MASTER DATASETS")
print("=" * 80)

print(
    f"Labeled master users:        "
    f"{len(labeled_master)}"
)

print(
    f"Binary Bot/Human users:      "
    f"{len(labeled_binary_master)}"
)

print(
    f"True unlabeled users:        "
    f"{len(true_unlabeled_master)}"
)

print(
    f"Complete multimodal binary:  "
    f"{len(complete_multimodal_binary)}"
)


# ------------------------------------------------------------
# 10. Binary label distribution
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("BINARY LABEL DISTRIBUTION")
print("=" * 80)

print(
    labeled_binary_master[
        "label_raw"
    ].value_counts()
)


# ------------------------------------------------------------
# 11. Modality coverage — labeled binary
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("MODALITY COVERAGE — BINARY LABELED USERS")
print("=" * 80)

modality_columns = [
    "has_profile",
    "has_description",
    "has_raw_tweets",
    "has_behavior_temporal",
    "has_graph"
]

for col in modality_columns:

    count = int(
        labeled_binary_master[col].sum()
    )

    pct = (
        count
        / len(labeled_binary_master)
        * 100
    )

    print(
        f"{col:25s}: "
        f"{count:4d} "
        f"({pct:6.2f}%)"
    )


# ------------------------------------------------------------
# 12. Complete multimodal binary label distribution
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("COMPLETE MULTIMODAL BINARY LABEL DISTRIBUTION")
print("=" * 80)

print(
    complete_multimodal_binary[
        "label_raw"
    ].value_counts()
)


# ------------------------------------------------------------
# 13. Modality coverage — true unlabeled
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("MODALITY COVERAGE — TRUE UNLABELED USERS")
print("=" * 80)

for col in modality_columns:

    count = int(
        true_unlabeled_master[col].sum()
    )

    pct = (
        count
        / len(true_unlabeled_master)
        * 100
    )

    print(
        f"{col:25s}: "
        f"{count:5d} "
        f"({pct:6.2f}%)"
    )


# ------------------------------------------------------------
# 14. Critical leakage / overlap checks
# ------------------------------------------------------------

labeled_master_set = set(
    labeled_master["user_key"]
)

unlabeled_master_set = set(
    true_unlabeled_master["user_key"]
)

overlap_check = (
    labeled_master_set
    & unlabeled_master_set
)


assert len(labeled_master) == 1103

assert len(
    labeled_binary_master
) == 959

assert len(
    true_unlabeled_master
) == 18332

assert len(overlap_check) == 0

assert (
    labeled_master["user_key"]
    .is_unique
)

assert (
    true_unlabeled_master["user_key"]
    .is_unique
)


print("\n" + "=" * 80)
print("SANITY CHECKS")
print("=" * 80)

print(
    "Labeled ∩ True Unlabeled:",
    len(overlap_check)
)

print(
    "All dataset separation checks passed."
)

CANONICAL MASTER DATASETS
Labeled master users:        1103
Binary Bot/Human users:      959
True unlabeled users:        18332
Complete multimodal binary:  276

BINARY LABEL DISTRIBUTION
label_raw
human(2)    772
bot(1)      187
Name: count, dtype: int64

MODALITY COVERAGE — BINARY LABELED USERS
has_profile              :  959 (100.00%)
has_description          :  915 ( 95.41%)
has_raw_tweets           :  959 (100.00%)
has_behavior_temporal    :  959 (100.00%)
has_graph                :  276 ( 28.78%)

COMPLETE MULTIMODAL BINARY LABEL DISTRIBUTION
label_raw
human(2)    232
bot(1)       44
Name: count, dtype: int64

MODALITY COVERAGE — TRUE UNLABELED USERS
has_profile              : 18332 (100.00%)
has_description          : 11648 ( 63.54%)
has_raw_tweets           :     0 (  0.00%)
has_behavior_temporal    : 18332 (100.00%)
has_graph                :  3033 ( 16.54%)

SANITY CHECKS
Labeled ∩ True Unlabeled: 0
All dataset separation checks passed.


In [12]:
# ============================================================
# Cell 10 — Save Canonical Master Datasets
# ============================================================

# ------------------------------------------------------------
# Output paths
# ------------------------------------------------------------

LABELED_MASTER_PATH = (
    PROCESSED_DIR / "labeled_master.csv"
)

LABELED_BINARY_MASTER_PATH = (
    PROCESSED_DIR / "labeled_binary_master.csv"
)

TRUE_UNLABELED_MASTER_PATH = (
    PROCESSED_DIR / "true_unlabeled_master.csv"
)

COMPLETE_MULTIMODAL_BINARY_PATH = (
    PROCESSED_DIR / "complete_multimodal_binary.csv"
)

FEATURE_POLICY_PATH = (
    PROCESSED_DIR / "feature_policy.json"
)


# ------------------------------------------------------------
# Save master datasets
# ------------------------------------------------------------

labeled_master.to_csv(
    LABELED_MASTER_PATH,
    index=False,
    encoding="utf-8-sig"
)

labeled_binary_master.to_csv(
    LABELED_BINARY_MASTER_PATH,
    index=False,
    encoding="utf-8-sig"
)

true_unlabeled_master.to_csv(
    TRUE_UNLABELED_MASTER_PATH,
    index=False,
    encoding="utf-8-sig"
)

complete_multimodal_binary.to_csv(
    COMPLETE_MULTIMODAL_BINARY_PATH,
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# Save feature policy / schema
# ------------------------------------------------------------

feature_policy = {

    "target": {
        "column": "label_binary",
        "human": 0,
        "bot": 1
    },

    "profile_numeric_features":
        PROFILE_NUMERIC_FEATURES,

    "profile_binary_features":
        PROFILE_BINARY_FEATURES,

    "profile_categorical_features":
        PROFILE_CATEGORICAL_FEATURES,

    "behavior_temporal_features":
        BEHAVIOR_TEMPORAL_FEATURES,

    "derived_features":
        DERIVED_FEATURES,

    "final_tabular_features":
        FINAL_TABULAR_FEATURES,

    "text_features":
        TEXT_FEATURES,

    "external_detector_columns_excluded":
        EXTERNAL_DETECTOR_COLUMNS,

    "annotation_columns_excluded":
        ANNOTATION_COLUMNS
}


with open(
    FEATURE_POLICY_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        feature_policy,
        f,
        ensure_ascii=False,
        indent=4
    )


# ------------------------------------------------------------
# Verify saved files
# ------------------------------------------------------------

print("=" * 80)
print("SAVED PHASE-2 MASTER FILES")
print("=" * 80)

saved_files = [
    LABELED_MASTER_PATH,
    LABELED_BINARY_MASTER_PATH,
    TRUE_UNLABELED_MASTER_PATH,
    COMPLETE_MULTIMODAL_BINARY_PATH,
    FEATURE_POLICY_PATH
]

for path in saved_files:

    print(
        f"{path.name:40s}",
        "OK" if path.exists() else "MISSING"
    )


# ------------------------------------------------------------
# File sizes
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("DATASET SUMMARY")
print("=" * 80)

print(
    f"labeled_master:              "
    f"{labeled_master.shape}"
)

print(
    f"labeled_binary_master:       "
    f"{labeled_binary_master.shape}"
)

print(
    f"true_unlabeled_master:       "
    f"{true_unlabeled_master.shape}"
)

print(
    f"complete_multimodal_binary:  "
    f"{complete_multimodal_binary.shape}"
)


# ------------------------------------------------------------
# Final integrity checks
# ------------------------------------------------------------

assert (
    pd.read_csv(
        LABELED_BINARY_MASTER_PATH,
        low_memory=False
    ).shape[0]
    == 959
)

assert (
    pd.read_csv(
        TRUE_UNLABELED_MASTER_PATH,
        low_memory=False
    ).shape[0]
    == 18332
)

assert (
    pd.read_csv(
        COMPLETE_MULTIMODAL_BINARY_PATH,
        low_memory=False
    ).shape[0]
    == 276
)


print("\nAll canonical datasets saved successfully.")

SAVED PHASE-2 MASTER FILES
labeled_master.csv                       OK
labeled_binary_master.csv                OK
true_unlabeled_master.csv                OK
complete_multimodal_binary.csv           OK
feature_policy.json                      OK

DATASET SUMMARY
labeled_master:              (1103, 61)
labeled_binary_master:       (959, 61)
true_unlabeled_master:       (18332, 61)
complete_multimodal_binary:  (276, 61)

All canonical datasets saved successfully.


In [13]:
# ============================================================
# Cell 11 — Data Type Cleaning & Validation
# ============================================================

# ------------------------------------------------------------
# 1. Define feature groups by expected data type
# ------------------------------------------------------------

NUMERIC_FEATURES = (
    PROFILE_NUMERIC_FEATURES
    + BEHAVIOR_TEMPORAL_FEATURES
    + DERIVED_FEATURES
)

BINARY_FEATURES = PROFILE_BINARY_FEATURES

CATEGORICAL_FEATURES = PROFILE_CATEGORICAL_FEATURES


print("=" * 80)
print("EXPECTED FEATURE TYPES")
print("=" * 80)

print("Numeric features:      ", len(NUMERIC_FEATURES))
print("Binary features:       ", len(BINARY_FEATURES))
print("Categorical features:  ", len(CATEGORICAL_FEATURES))

print(
    "Total:",
    len(NUMERIC_FEATURES)
    + len(BINARY_FEATURES)
    + len(CATEGORICAL_FEATURES)
)


# ------------------------------------------------------------
# 2. Robust binary conversion
# ------------------------------------------------------------

TRUE_VALUES = {
    "true",
    "1",
    "1.0",
    "yes",
    "y"
}

FALSE_VALUES = {
    "false",
    "0",
    "0.0",
    "no",
    "n"
}


def clean_binary_value(value):

    if pd.isna(value):
        return np.nan

    value_str = str(value).strip().lower()

    if value_str in TRUE_VALUES:
        return 1.0

    if value_str in FALSE_VALUES:
        return 0.0

    return np.nan


# ------------------------------------------------------------
# 3. Cleaning function
# ------------------------------------------------------------

def clean_master_dtypes(df, dataset_name="dataset"):

    df = df.copy()

    report = {
        "dataset": dataset_name,
        "numeric_coercion_to_nan": {},
        "binary_unknown_values": {},
        "infinite_values_before": 0,
        "infinite_values_after": 0
    }

    # --------------------------------------------------------
    # Numeric features
    # --------------------------------------------------------

    for col in NUMERIC_FEATURES:

        before_missing = df[col].isna().sum()

        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

        after_missing = df[col].isna().sum()

        newly_missing = (
            after_missing
            - before_missing
        )

        report[
            "numeric_coercion_to_nan"
        ][col] = int(newly_missing)

    # --------------------------------------------------------
    # Binary features
    # --------------------------------------------------------

    for col in BINARY_FEATURES:

        original_non_missing = (
            df[col]
            .dropna()
            .astype(str)
            .str.strip()
            .str.lower()
        )

        recognized = (
            original_non_missing.isin(
                TRUE_VALUES | FALSE_VALUES
            )
        )

        unknown_values = sorted(
            original_non_missing[
                ~recognized
            ]
            .unique()
            .tolist()
        )

        report[
            "binary_unknown_values"
        ][col] = unknown_values

        df[col] = (
            df[col]
            .apply(clean_binary_value)
            .astype("float64")
        )

    # --------------------------------------------------------
    # Categorical features
    # --------------------------------------------------------

    for col in CATEGORICAL_FEATURES:

        df[col] = (
            df[col]
            .astype("string")
            .str.strip()
            .str.lower()
        )

        df[col] = df[col].replace(
            {
                "": pd.NA,
                "nan": pd.NA,
                "none": pd.NA,
                "<na>": pd.NA
            }
        )

    # --------------------------------------------------------
    # Detect infinity
    # --------------------------------------------------------

    numeric_array = (
        df[NUMERIC_FEATURES + BINARY_FEATURES]
        .to_numpy(dtype=float)
    )

    report[
        "infinite_values_before"
    ] = int(
        np.isinf(numeric_array).sum()
    )

    # Replace ±inf with NaN
    df[
        NUMERIC_FEATURES + BINARY_FEATURES
    ] = df[
        NUMERIC_FEATURES + BINARY_FEATURES
    ].replace(
        [np.inf, -np.inf],
        np.nan
    )

    numeric_array_after = (
        df[NUMERIC_FEATURES + BINARY_FEATURES]
        .to_numpy(dtype=float)
    )

    report[
        "infinite_values_after"
    ] = int(
        np.isinf(
            numeric_array_after
        ).sum()
    )

    # --------------------------------------------------------
    # Modality masks
    # --------------------------------------------------------

    modality_cols = [
        "has_profile",
        "has_description",
        "has_raw_tweets",
        "has_behavior_temporal",
        "has_graph"
    ]

    for col in modality_cols:
        df[col] = (
            pd.to_numeric(
                df[col],
                errors="coerce"
            )
            .astype("Int8")
        )

    return df, report


# ------------------------------------------------------------
# 4. Clean all master datasets consistently
# ------------------------------------------------------------

labeled_master_clean, report_labeled = (
    clean_master_dtypes(
        labeled_master,
        "labeled_master"
    )
)

labeled_binary_clean, report_binary = (
    clean_master_dtypes(
        labeled_binary_master,
        "labeled_binary_master"
    )
)

true_unlabeled_clean, report_unlabeled = (
    clean_master_dtypes(
        true_unlabeled_master,
        "true_unlabeled_master"
    )
)

complete_multimodal_binary_clean, report_complete = (
    clean_master_dtypes(
        complete_multimodal_binary,
        "complete_multimodal_binary"
    )
)


# ------------------------------------------------------------
# 5. Numeric conversion report
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("NUMERIC COERCION CHECK")
print("=" * 80)

for report in [
    report_labeled,
    report_unlabeled
]:

    problematic = {
        key: value
        for key, value
        in report[
            "numeric_coercion_to_nan"
        ].items()
        if value > 0
    }

    print(
        f"\n{report['dataset']}:"
    )

    if problematic:
        print(problematic)
    else:
        print(
            "No unexpected numeric values."
        )


# ------------------------------------------------------------
# 6. Binary value report
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("BINARY VALUE CHECK")
print("=" * 80)

for col in BINARY_FEATURES:

    unknown_labeled = (
        report_labeled[
            "binary_unknown_values"
        ][col]
    )

    unknown_unlabeled = (
        report_unlabeled[
            "binary_unknown_values"
        ][col]
    )

    print(
        f"{col:28s} | "
        f"labeled unknown: {unknown_labeled} | "
        f"unlabeled unknown: {unknown_unlabeled}"
    )


# ------------------------------------------------------------
# 7. Categorical value inspection
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("CATEGORICAL FEATURE VALUES")
print("=" * 80)

for col in CATEGORICAL_FEATURES:

    print(f"\nFeature: {col}")

    print(
        pd.concat([
            labeled_binary_clean[col],
            true_unlabeled_clean[col]
        ])
        .value_counts(
            dropna=False
        )
    )


# ------------------------------------------------------------
# 8. Infinity check
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("INFINITY CHECK")
print("=" * 80)

for report in [
    report_labeled,
    report_binary,
    report_unlabeled,
    report_complete
]:

    print(
        f"{report['dataset']:32s} | "
        f"before = "
        f"{report['infinite_values_before']:4d} | "
        f"after = "
        f"{report['infinite_values_after']:4d}"
    )


# ------------------------------------------------------------
# 9. Verify binary features now contain only 0, 1, NaN
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("STANDARDIZED BINARY FEATURES")
print("=" * 80)

binary_validation = []

for col in BINARY_FEATURES:

    labeled_values = set(
        labeled_binary_clean[col]
        .dropna()
        .unique()
    )

    unlabeled_values = set(
        true_unlabeled_clean[col]
        .dropna()
        .unique()
    )

    valid_labeled = labeled_values.issubset(
        {0.0, 1.0}
    )

    valid_unlabeled = unlabeled_values.issubset(
        {0.0, 1.0}
    )

    binary_validation.append({
        "feature": col,
        "labeled_values":
            sorted(labeled_values),
        "unlabeled_values":
            sorted(unlabeled_values),
        "valid":
            valid_labeled
            and valid_unlabeled
    })


binary_validation = pd.DataFrame(
    binary_validation
)

display(binary_validation)


# ------------------------------------------------------------
# 10. Final integrity checks
# ------------------------------------------------------------

assert len(
    labeled_binary_clean
) == 959

assert len(
    true_unlabeled_clean
) == 18332

assert all(
    binary_validation["valid"]
)

assert set(
    FINAL_TABULAR_FEATURES
).issubset(
    labeled_binary_clean.columns
)

assert set(
    FINAL_TABULAR_FEATURES
).issubset(
    true_unlabeled_clean.columns
)


print("\n" + "=" * 80)
print("DATA TYPE CLEANING COMPLETE")
print("=" * 80)

print(
    "All 47 tabular features are present."
)

print(
    "No train/test information has been used."
)

print(
    "No imputer, scaler, or encoder has been fitted yet."
)

EXPECTED FEATURE TYPES
Numeric features:       37
Binary features:        9
Categorical features:   1
Total: 47

NUMERIC COERCION CHECK

labeled_master:
No unexpected numeric values.

true_unlabeled_master:
No unexpected numeric values.

BINARY VALUE CHECK
default_profile              | labeled unknown: [] | unlabeled unknown: []
default_profile_image        | labeled unknown: [] | unlabeled unknown: []
verified                     | labeled unknown: [] | unlabeled unknown: []
has_custom_timelines         | labeled unknown: [] | unlabeled unknown: []
is_translator                | labeled unknown: [] | unlabeled unknown: []
possibly_sensitive           | labeled unknown: [] | unlabeled unknown: []
want_retweets                | labeled unknown: [] | unlabeled unknown: []
hashtag_in_description       | labeled unknown: [] | unlabeled unknown: ['1.19363e+18']
numbers_in_description       | labeled unknown: [] | unlabeled unknown: ['1.19363e+18']

CATEGORICAL FEATURE VALUES

Feature: tran

,feature,labeled_values,unlabeled_values,valid
0,default_profile,"[0.0, 1.0]","[0.0, 1.0]",True
1,default_profile_image,"[0.0, 1.0]","[0.0, 1.0]",True
2,verified,[0.0],[0.0],True
3,has_custom_timelines,"[0.0, 1.0]","[0.0, 1.0]",True
4,is_translator,[0.0],[0.0],True
5,possibly_sensitive,"[0.0, 1.0]","[0.0, 1.0]",True
6,want_retweets,[0.0],[0.0],True
7,hashtag_in_description,"[0.0, 1.0]","[0.0, 1.0]",True
8,numbers_in_description,"[0.0, 1.0]","[0.0, 1.0]",True



DATA TYPE CLEANING COMPLETE
All 47 tabular features are present.
No train/test information has been used.
No imputer, scaler, or encoder has been fitted yet.


In [14]:
# ============================================================
# Cell 12 — Inspect Data Anomalies & Near-Constant Features
# ============================================================

# ------------------------------------------------------------
# 1. Locate invalid raw values in binary features
# ------------------------------------------------------------

print("=" * 80)
print("INVALID RAW BINARY VALUES")
print("=" * 80)

binary_anomaly_rows = []

for col in BINARY_FEATURES:

    raw_values = (
        users_master[col]
        .dropna()
        .astype(str)
        .str.strip()
        .str.lower()
    )

    valid_mask = raw_values.isin(
        TRUE_VALUES | FALSE_VALUES
    )

    invalid_indices = raw_values[
        ~valid_mask
    ].index

    if len(invalid_indices) > 0:

        print(f"\nFeature: {col}")
        print(
            users_master.loc[
                invalid_indices,
                [
                    "id",
                    "screen_name",
                    "user_key",
                    col
                ]
            ]
        )

        for idx in invalid_indices:
            binary_anomaly_rows.append({
                "row_index": idx,
                "feature": col,
                "value": users_master.loc[idx, col],
                "user_key": users_master.loc[idx, "user_key"]
            })


binary_anomaly_report = pd.DataFrame(
    binary_anomaly_rows
)

print("\nTotal invalid binary cells:",
      len(binary_anomaly_report))


# ------------------------------------------------------------
# 2. If anomaly exists, inspect the full suspicious row
# ------------------------------------------------------------

if len(binary_anomaly_report) > 0:

    suspicious_indices = (
        binary_anomaly_report[
            "row_index"
        ]
        .unique()
    )

    print("\n" + "=" * 80)
    print("SUSPICIOUS RAW ROWS")
    print("=" * 80)

    for idx in suspicious_indices:

        print(f"\n--- Row index: {idx} ---")

        row = users_master.loc[idx]

        non_missing_row = row[
            row.notna()
        ]

        display(
            non_missing_row.to_frame(
                name="value"
            )
        )


# ------------------------------------------------------------
# 3. Check number of unique values for all 47 model features
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FEATURE CARDINALITY")
print("=" * 80)

cardinality_report = []

for col in FINAL_TABULAR_FEATURES:

    labeled_unique = (
        labeled_binary_clean[col]
        .nunique(dropna=True)
    )

    unlabeled_unique = (
        true_unlabeled_clean[col]
        .nunique(dropna=True)
    )

    combined_unique = (
        pd.concat([
            labeled_binary_clean[col],
            true_unlabeled_clean[col]
        ])
        .nunique(dropna=True)
    )

    cardinality_report.append({
        "feature": col,
        "labeled_unique": labeled_unique,
        "unlabeled_unique": unlabeled_unique,
        "combined_unique": combined_unique,
        "labeled_missing_pct":
            labeled_binary_clean[col]
            .isna()
            .mean() * 100,
        "unlabeled_missing_pct":
            true_unlabeled_clean[col]
            .isna()
            .mean() * 100
    })


cardinality_report = pd.DataFrame(
    cardinality_report
).sort_values(
    by=[
        "labeled_unique",
        "combined_unique"
    ]
)


display(
    cardinality_report
)


# ------------------------------------------------------------
# 4. Constant features in labeled binary dataset
# ------------------------------------------------------------

constant_labeled_features = (
    cardinality_report.loc[
        cardinality_report[
            "labeled_unique"
        ] <= 1,
        "feature"
    ]
    .tolist()
)


print("\n" + "=" * 80)
print("CONSTANT FEATURES — LABELED BINARY")
print("=" * 80)

print(constant_labeled_features)


# ------------------------------------------------------------
# 5. Extremely sparse categorical feature
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("TRANSLATOR_TYPE COVERAGE")
print("=" * 80)

for name, df in {
    "binary labeled": labeled_binary_clean,
    "true unlabeled": true_unlabeled_clean
}.items():

    present = df["translator_type"].notna().sum()
    total = len(df)

    print(
        f"{name:20s}: "
        f"{present:5d}/{total:5d} "
        f"({present / total * 100:.3f}%)"
    )

    print(
        df["translator_type"]
        .value_counts(dropna=False)
    )


# ------------------------------------------------------------
# 6. Check whether suspicious row belongs to labeled/unlabeled
# ------------------------------------------------------------

if len(binary_anomaly_report) > 0:

    print("\n" + "=" * 80)
    print("ANOMALY DATASET MEMBERSHIP")
    print("=" * 80)

    anomaly_users = set(
        binary_anomaly_report[
            "user_key"
        ]
        .dropna()
    )

    print(
        "In labeled set:",
        len(
            anomaly_users
            & labeled_set
        )
    )

    print(
        "In true unlabeled set:",
        len(
            anomaly_users
            & true_unlabeled_set
        )
    )

INVALID RAW BINARY VALUES

Feature: hashtag_in_description
               id          screen_name             user_key  \
198  1.193630e+18  1193634420000993280  1193634420000993280   

     hashtag_in_description  
198            1.193630e+18  

Feature: numbers_in_description
               id          screen_name             user_key  \
198  1.193630e+18  1193634420000993280  1193634420000993280   

     numbers_in_description  
198            1.193630e+18  

Total invalid binary cells: 2

SUSPICIOUS RAW ROWS

--- Row index: 198 ---


,value
id,1193630000000000000.0
cluster_id,1193630000000000000.0
screen_name,1193634420000993280
name,1193634420000993280
clean_description,1193634420000993280
followers_count,1193630000000000000.0
friends_count,1193630000000000000.0
default_profile,1.0
default_profile_image,False
verified,0.0



FEATURE CARDINALITY


,feature,labeled_unique,unlabeled_unique,combined_unique,labeled_missing_pct,unlabeled_missing_pct
7,fast_followers_count,1,1,1,0.000000,0.005455
14,verified,1,1,1,0.000000,0.000000
16,is_translator,1,1,1,0.000000,0.005455
18,want_retweets,1,1,1,0.000000,0.005455
21,translator_type,1,1,1,98.331595,99.694523
12,default_profile,2,2,2,0.000000,0.000000
13,default_profile_image,2,2,2,0.000000,0.000000
15,has_custom_timelines,2,2,2,0.000000,0.005455
17,possibly_sensitive,2,2,2,0.000000,0.005455
19,hashtag_in_description,2,2,2,0.000000,0.005455



CONSTANT FEATURES — LABELED BINARY
['fast_followers_count', 'verified', 'is_translator', 'want_retweets', 'translator_type']

TRANSLATOR_TYPE COVERAGE
binary labeled      :    16/  959 (1.668%)
translator_type
<NA>       943
regular     16
Name: count, dtype: Int64
true unlabeled      :    56/18332 (0.305%)
translator_type
<NA>       18276
regular       56
Name: count, dtype: Int64

ANOMALY DATASET MEMBERSHIP
In labeled set: 0
In true unlabeled set: 1


In [15]:
# ============================================================
# Cell 13 — Remove Corrupt User & Finalize Model Feature Space
# ============================================================

# ------------------------------------------------------------
# 1. Explicitly record malformed unlabeled users
# ------------------------------------------------------------

CORRUPT_USER_KEYS = {
    "1193634420000993280"
}


print("=" * 80)
print("CORRUPT USER REMOVAL")
print("=" * 80)

print(
    "Corrupt users identified:",
    CORRUPT_USER_KEYS
)


# ------------------------------------------------------------
# 2. Remove corrupt users ONLY from true-unlabeled pool
# ------------------------------------------------------------

before_unlabeled = len(
    true_unlabeled_clean
)

true_unlabeled_clean = (
    true_unlabeled_clean[
        ~true_unlabeled_clean[
            "user_key"
        ].isin(CORRUPT_USER_KEYS)
    ]
    .copy()
)

after_unlabeled = len(
    true_unlabeled_clean
)


print(
    f"\nTrue unlabeled before: "
    f"{before_unlabeled}"
)

print(
    f"True unlabeled after:  "
    f"{after_unlabeled}"
)

print(
    f"Removed:               "
    f"{before_unlabeled - after_unlabeled}"
)


# ------------------------------------------------------------
# 3. Features removed from predictive model
# ------------------------------------------------------------

CONSTANT_OR_UNUSABLE_FEATURES = [
    "fast_followers_count",
    "verified",
    "is_translator",
    "want_retweets",
    "translator_type"
]


# ------------------------------------------------------------
# 4. Final predictive tabular feature space
# ------------------------------------------------------------

FINAL_MODEL_TABULAR_FEATURES = [
    feature
    for feature in FINAL_TABULAR_FEATURES
    if feature
    not in CONSTANT_OR_UNUSABLE_FEATURES
]


# Updated type groups
FINAL_NUMERIC_FEATURES = [
    feature
    for feature in NUMERIC_FEATURES
    if feature
    in FINAL_MODEL_TABULAR_FEATURES
]

FINAL_BINARY_FEATURES = [
    feature
    for feature in BINARY_FEATURES
    if feature
    in FINAL_MODEL_TABULAR_FEATURES
]

FINAL_CATEGORICAL_FEATURES = [
    feature
    for feature in CATEGORICAL_FEATURES
    if feature
    in FINAL_MODEL_TABULAR_FEATURES
]


# ------------------------------------------------------------
# 5. Report final feature counts
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FINAL MODEL FEATURE SPACE")
print("=" * 80)

print(
    "Original tabular features:",
    len(FINAL_TABULAR_FEATURES)
)

print(
    "Removed constant/unusable:",
    len(CONSTANT_OR_UNUSABLE_FEATURES)
)

print(
    "Final tabular features:",
    len(FINAL_MODEL_TABULAR_FEATURES)
)

print(
    "\nFinal numeric:",
    len(FINAL_NUMERIC_FEATURES)
)

print(
    "Final binary:",
    len(FINAL_BINARY_FEATURES)
)

print(
    "Final categorical:",
    len(FINAL_CATEGORICAL_FEATURES)
)


print("\nRemoved features:")

for feature in CONSTANT_OR_UNUSABLE_FEATURES:
    print(" -", feature)


# ------------------------------------------------------------
# 6. Verify no remaining feature is constant in labeled data
# ------------------------------------------------------------

remaining_constant_features = []

for feature in FINAL_MODEL_TABULAR_FEATURES:

    n_unique = (
        labeled_binary_clean[
            feature
        ]
        .nunique(dropna=True)
    )

    if n_unique <= 1:
        remaining_constant_features.append(
            feature
        )


print("\n" + "=" * 80)
print("REMAINING CONSTANT FEATURE CHECK")
print("=" * 80)

print(
    remaining_constant_features
)

assert (
    len(
        remaining_constant_features
    )
    == 0
)


# ------------------------------------------------------------
# 7. Verify corrupt user is gone
# ------------------------------------------------------------

assert not any(
    true_unlabeled_clean[
        "user_key"
    ].isin(CORRUPT_USER_KEYS)
)


# ------------------------------------------------------------
# 8. Verify labeled dataset was NOT affected
# ------------------------------------------------------------

assert len(
    labeled_binary_clean
) == 959


# ------------------------------------------------------------
# 9. Final dataset sizes
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FINAL DATASET STATUS BEFORE SPLIT")
print("=" * 80)

print(
    "Binary labeled users:",
    len(labeled_binary_clean)
)

print(
    "Clean true-unlabeled users:",
    len(true_unlabeled_clean)
)

print(
    "Final model features:",
    len(FINAL_MODEL_TABULAR_FEATURES)
)


print(
    "\nFeature space finalized successfully."
)

CORRUPT USER REMOVAL
Corrupt users identified: {'1193634420000993280'}

True unlabeled before: 18332
True unlabeled after:  18331
Removed:               1

FINAL MODEL FEATURE SPACE
Original tabular features: 47
Removed constant/unusable: 5
Final tabular features: 42

Final numeric: 36
Final binary: 6
Final categorical: 0

Removed features:
 - fast_followers_count
 - verified
 - is_translator
 - want_retweets
 - translator_type

REMAINING CONSTANT FEATURE CHECK
[]

FINAL DATASET STATUS BEFORE SPLIT
Binary labeled users: 959
Clean true-unlabeled users: 18331
Final model features: 42

Feature space finalized successfully.


In [16]:
# ============================================================
# Cell 14 — Save Final QC-Cleaned Modeling Data
# ============================================================

# ------------------------------------------------------------
# 1. Columns that must remain in model-ready user tables
# ------------------------------------------------------------


MODEL_REFERENCE_COLUMNS = [
    "id",
    "screen_name",
    "user_key",
    "name",
    "description",
    "clean_description",
    "created_at"
]

MODEL_LABEL_COLUMNS = [
    "label_raw",
    "label_binary"
]

MODALITY_MASK_COLUMNS = [
    "has_profile",
    "has_description",
    "has_raw_tweets",
    "has_behavior_temporal",
    "has_graph"
]


MODEL_READY_COLUMNS = list(dict.fromkeys(
    MODEL_REFERENCE_COLUMNS
    + FINAL_MODEL_TABULAR_FEATURES
    + MODEL_LABEL_COLUMNS
    + MODALITY_MASK_COLUMNS
))


# ------------------------------------------------------------
# 2. Create trimmed model-ready datasets
# ------------------------------------------------------------

modeling_labeled_binary = (
    labeled_binary_clean[
        MODEL_READY_COLUMNS
    ]
    .copy()
)

modeling_true_unlabeled = (
    true_unlabeled_clean[
        MODEL_READY_COLUMNS
    ]
    .copy()
)

modeling_complete_multimodal = (
    complete_multimodal_binary_clean[
        MODEL_READY_COLUMNS
    ]
    .copy()
)


# ------------------------------------------------------------
# 3. Output directory & paths
# ------------------------------------------------------------

PHASE2_DIR = PROCESSED_DIR / "phase2"
PHASE2_DIR.mkdir(parents=True, exist_ok=True)


MODELING_LABELED_PATH = (
    PHASE2_DIR /
    "modeling_labeled_binary.csv"
)

MODELING_UNLABELED_PATH = (
    PHASE2_DIR /
    "modeling_true_unlabeled.csv"
)

MODELING_COMPLETE_PATH = (
    PHASE2_DIR /
    "modeling_complete_multimodal_binary.csv"
)

MODEL_FEATURE_POLICY_PATH = (
    PHASE2_DIR /
    "model_feature_policy.json"
)

DATA_QUALITY_EXCLUSIONS_PATH = (
    PHASE2_DIR /
    "data_quality_exclusions.json"
)

print("Phase 2 output directory:")
print(PHASE2_DIR)


# ------------------------------------------------------------
# 4. Save final QC-cleaned datasets
# ------------------------------------------------------------

modeling_labeled_binary.to_csv(
    MODELING_LABELED_PATH,
    index=False,
    encoding="utf-8-sig"
)

modeling_true_unlabeled.to_csv(
    MODELING_UNLABELED_PATH,
    index=False,
    encoding="utf-8-sig"
)

modeling_complete_multimodal.to_csv(
    MODELING_COMPLETE_PATH,
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# 5. Save FINAL model feature policy
# ------------------------------------------------------------

model_feature_policy = {

    "target": {
        "column": "label_binary",
        "human": 0,
        "bot": 1
    },

    "final_model_tabular_features":
        FINAL_MODEL_TABULAR_FEATURES,

    "numeric_features":
        FINAL_NUMERIC_FEATURES,

    "binary_features":
        FINAL_BINARY_FEATURES,

    "categorical_features":
        FINAL_CATEGORICAL_FEATURES,

    "text_fields": [
        "description",
        "clean_description"
    ],

    "modality_masks":
        MODALITY_MASK_COLUMNS,

    "removed_constant_or_unusable_features":
        CONSTANT_OR_UNUSABLE_FEATURES,

    "external_detector_features_excluded":
        EXTERNAL_DETECTOR_COLUMNS,

    "annotation_columns_excluded":
        ANNOTATION_COLUMNS,

    "n_final_tabular_features":
        len(FINAL_MODEL_TABULAR_FEATURES)
}


with open(
    MODEL_FEATURE_POLICY_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        model_feature_policy,
        f,
        ensure_ascii=False,
        indent=4
    )


# ------------------------------------------------------------
# 6. Save data-quality exclusions
# ------------------------------------------------------------

data_quality_exclusions = {
    "corrupt_user_keys": sorted(
        CORRUPT_USER_KEYS
    ),

    "reason": (
        "Malformed row containing repeated Twitter/X ID "
        "values across numerous unrelated profile and "
        "behavioral features."
    ),

    "raw_true_unlabeled_count": 18332,
    "clean_true_unlabeled_count": 18331
}


with open(
    DATA_QUALITY_EXCLUSIONS_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        data_quality_exclusions,
        f,
        ensure_ascii=False,
        indent=4
    )


# ------------------------------------------------------------
# 7. Verification report
# ------------------------------------------------------------

print("=" * 80)
print("FINAL MODEL-READY DATASETS")
print("=" * 80)

print(
    "Labeled binary:",
    modeling_labeled_binary.shape
)

print(
    "True unlabeled:",
    modeling_true_unlabeled.shape
)

print(
    "Complete multimodal binary:",
    modeling_complete_multimodal.shape
)

print(
    "\nFinal tabular features:",
    len(FINAL_MODEL_TABULAR_FEATURES)
)


print("\n" + "=" * 80)
print("FINAL FEATURE TYPES")
print("=" * 80)

print(
    "Numeric:",
    len(FINAL_NUMERIC_FEATURES)
)

print(
    "Binary:",
    len(FINAL_BINARY_FEATURES)
)

print(
    "Categorical:",
    len(FINAL_CATEGORICAL_FEATURES)
)


# ------------------------------------------------------------
# 8. Critical integrity checks
# ------------------------------------------------------------

assert len(modeling_labeled_binary) == 959

assert len(modeling_true_unlabeled) == 18331

assert len(modeling_complete_multimodal) == 276

assert len(FINAL_MODEL_TABULAR_FEATURES) == 42

assert not any(
    modeling_true_unlabeled[
        "user_key"
    ].isin(CORRUPT_USER_KEYS)
)

assert set(
    CONSTANT_OR_UNUSABLE_FEATURES
).isdisjoint(
    set(FINAL_MODEL_TABULAR_FEATURES)
)

assert set(
    EXTERNAL_DETECTOR_COLUMNS
).isdisjoint(
    set(modeling_labeled_binary.columns)
)


print("\n" + "=" * 80)
print("SAVED FILES")
print("=" * 80)

for path in [
    MODELING_LABELED_PATH,
    MODELING_UNLABELED_PATH,
    MODELING_COMPLETE_PATH,
    MODEL_FEATURE_POLICY_PATH,
    DATA_QUALITY_EXCLUSIONS_PATH
]:

    print(
        f"{path.name:45s}",
        "OK" if path.exists() else "MISSING"
    )


print(
    "\nFinal QC-cleaned modeling datasets saved successfully."
)

Phase 2 output directory:
C:\Users\p.vazifeh\Desktop\University\Project\Bot Detection Implementation\processed_data\phase2
FINAL MODEL-READY DATASETS
Labeled binary: (959, 56)
True unlabeled: (18331, 56)
Complete multimodal binary: (276, 56)

Final tabular features: 42

FINAL FEATURE TYPES
Numeric: 36
Binary: 6
Categorical: 0

SAVED FILES
modeling_labeled_binary.csv                   OK
modeling_true_unlabeled.csv                   OK
modeling_complete_multimodal_binary.csv       OK
model_feature_policy.json                     OK
data_quality_exclusions.json                  OK

Final QC-cleaned modeling datasets saved successfully.


In [18]:
# ============================================================
# Cell 15 — Inspect Final Operational Datasets
# ============================================================

from pathlib import Path
import pandas as pd
import json

# ------------------------------------------------------------
# 1. Final data directory
# ------------------------------------------------------------

FINAL_DATA_DIR = PROJECT_ROOT / "final_data"

USERS_DIR = FINAL_DATA_DIR / "users"
TWEETS_DIR = FINAL_DATA_DIR / "tweets"
GRAPH_DIR = FINAL_DATA_DIR / "graph"
CONFIG_DIR = FINAL_DATA_DIR / "config"


# ------------------------------------------------------------
# 2. Define final operational files
# ------------------------------------------------------------

FINAL_FILES = {

    "Labeled Binary Users":
        USERS_DIR / "modeling_labeled_binary.csv",

    "True Unlabeled Users":
        USERS_DIR / "modeling_true_unlabeled.csv",

    "Complete Multimodal Binary":
        USERS_DIR / "modeling_complete_multimodal_binary.csv",

    "Tweets":
        TWEETS_DIR / "tweets_meta_data.csv",

    "Graph Edges":
        GRAPH_DIR / "graph_edges.csv",

    "Graph User Statistics":
        GRAPH_DIR / "graph_user_statistics.csv"
}


# ------------------------------------------------------------
# 3. Inspect CSV datasets
# ------------------------------------------------------------

print("=" * 100)
print("FINAL OPERATIONAL DATASET INSPECTION")
print("=" * 100)

final_datasets = {}

for name, path in FINAL_FILES.items():

    print("\n" + "=" * 100)
    print(name)
    print("=" * 100)

    print("Path:")
    print(path)

    print("\nExists:", path.exists())

    if not path.exists():
        print("FILE NOT FOUND")
        continue

    df = pd.read_csv(
        path,
        low_memory=False
    )

    final_datasets[name] = df

    print("\nShape:")
    print(df.shape)

    print("\nColumns:")
    print(df.columns.tolist())

    print("\nFirst 5 rows:")

    display(
        df.head(5)
    )


# ------------------------------------------------------------
# 4. Inspect configuration JSON files
# ------------------------------------------------------------

CONFIG_FILES = {

    "Model Feature Policy":
        CONFIG_DIR / "model_feature_policy.json",

    "Data Quality Exclusions":
        CONFIG_DIR / "data_quality_exclusions.json"
}


print("\n" + "=" * 100)
print("FINAL CONFIGURATION FILES")
print("=" * 100)

for name, path in CONFIG_FILES.items():

    print("\n" + "-" * 100)
    print(name)
    print("-" * 100)

    print("Path:")
    print(path)

    print("\nExists:", path.exists())

    if not path.exists():
        print("FILE NOT FOUND")
        continue

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        content = json.load(f)

    print("\nContent:")

    print(
        json.dumps(
            content,
            ensure_ascii=False,
            indent=2
        )
    )

FINAL OPERATIONAL DATASET INSPECTION

Labeled Binary Users
Path:
C:\Users\p.vazifeh\Desktop\University\Project\Bot Detection Implementation\final_data\users\modeling_labeled_binary.csv

Exists: True

Shape:
(959, 56)

Columns:
['id', 'screen_name', 'user_key', 'name', 'description', 'clean_description', 'created_at', 'followers_count', 'friends_count', 'favourites_count', 'listed_count', 'media_count', 'statuses_count', 'status_count', 'normal_followers_count', 'user_age', 'follower_growth_rate', 'friends_growth_rate', 'default_profile', 'default_profile_image', 'has_custom_timelines', 'possibly_sensitive', 'hashtag_in_description', 'numbers_in_description', 'no_type_tweet', 'no_type_retweet_with_comment', 'no_type_reply', 'mean_no_media_per_tweet', 'mean_no_words', 'no_languages', 'mean_no_hashtags', 'mean_favourites_per_tweet', 'time_between_tweets', 'tweet_frequency', 'min_tweets_per_hour', 'min_tweets_per_day', 'max_tweets_per_hour', 'max_tweets_per_day', 'max_occurence_of_same_gap

,id,screen_name,user_key,name,description,clean_description,created_at,followers_count,friends_count,favourites_count,...,num_digits_in_name,num_digits_in_username,url_in_description,label_raw,label_binary,has_profile,has_description,has_raw_tweets,has_behavior_temporal,has_graph
0,1.707640e+18,KathyMobarez,kathymobarez,Kathy Mobarez,NaN,NaN,2023-09-29T06:22:15+00:00,3586.0,1672.0,310972.0,...,0,0,0,bot(1),1,1,0,1,1,0
1,1.530060e+18,rahe_AZADI_5,rahe_azadi_5,KRATOS Spartan Rage👑,NaN,NaN,2022-05-27T05:48:47+00:00,NaN,NaN,288142.0,...,0,1,0,bot(1),1,1,0,1,1,0
2,1.391600e+18,dadkhahim,dadkhahim,دادخواهی‌م,#کشتار۶۷\n#تیر۷۸\n#خرداد۸۸\n#دی۹۶\n#آبان۹۸\n#ه...,NaN,2021-05-10T03:52:12+00:00,NaN,NaN,8895.0,...,0,0,0,bot(1),1,1,1,1,1,0
3,1.568300e+18,HiwaTubeAI,hiwatubeai,Hiwa,شاه، میهن، پرچم,The official handle of the Republic Media Netw...,2022-09-09T18:23:28+00:00,2841497.0,8.0,288898.0,...,0,0,0,bot(1),1,1,1,1,1,0
4,1.464650e+18,kazeroonkings,kazeroonkings,@kazeroonkings,ملت ایران ۹۹ دشمن و یک دوست بنام ملکه شهبانوفر...,ملت ایران ۹۹ دشمن و یک دوست بنام ملکه شهبانوفر...,2021-11-27T17:24:09+00:00,6830.0,5175.0,217554.0,...,0,0,0,bot(1),1,1,1,1,1,0



True Unlabeled Users
Path:
C:\Users\p.vazifeh\Desktop\University\Project\Bot Detection Implementation\final_data\users\modeling_true_unlabeled.csv

Exists: True

Shape:
(18331, 56)

Columns:
['id', 'screen_name', 'user_key', 'name', 'description', 'clean_description', 'created_at', 'followers_count', 'friends_count', 'favourites_count', 'listed_count', 'media_count', 'statuses_count', 'status_count', 'normal_followers_count', 'user_age', 'follower_growth_rate', 'friends_growth_rate', 'default_profile', 'default_profile_image', 'has_custom_timelines', 'possibly_sensitive', 'hashtag_in_description', 'numbers_in_description', 'no_type_tweet', 'no_type_retweet_with_comment', 'no_type_reply', 'mean_no_media_per_tweet', 'mean_no_words', 'no_languages', 'mean_no_hashtags', 'mean_favourites_per_tweet', 'time_between_tweets', 'tweet_frequency', 'min_tweets_per_hour', 'min_tweets_per_day', 'max_tweets_per_hour', 'max_tweets_per_day', 'max_occurence_of_same_gap', 'unique_mention_rate_per_tweet',

,id,screen_name,user_key,name,description,clean_description,created_at,followers_count,friends_count,favourites_count,...,num_digits_in_name,num_digits_in_username,url_in_description,label_raw,label_binary,has_profile,has_description,has_raw_tweets,has_behavior_temporal,has_graph
0,6.178539e+08,PatriotPointman,patriotpointman,Brett Murphy,Writer & Poet | Videos & Live Streams | Americ...,Writer & Poet | Videos & Live Streams | Americ...,2012-06-25T04:38:20+00:00,159406.0,39400.0,42127.0,...,0,0,0,NaN,NaN,1,1,0,1,0
1,1.580000e+18,IranZiba1401,iranziba1401,Free Iran,"💚🦁🤍🌞❤️\n\n@PahlaviReza, https://t.co/dlBsD3cnj...","@PahlaviReza, \n @PahlaviComms, \n @ShahbanouF...",2022-10-12T00:53:19+00:00,2020.0,1938.0,49690.0,...,0,4,1,NaN,NaN,1,1,0,1,0
2,1.716080e+18,MIIran20194,miiran20194,دخترایران از نسل رستم,من دخترایران \nاز نسل رستم\nمیجنگم برای ایران ...,من دخترایران \n از نسل رستم\n میجنگم برای ایرا...,2023-10-22T13:33:33+00:00,917.0,558.0,68844.0,...,0,5,0,NaN,NaN,1,1,0,1,0
3,1.684610e+18,Artemis540721,artemis540721,👑Artemis👑,تا ابد و یک روز جاوید شاه،پاینده ایران,تا ابد و یک روز جاوید شاه،پاینده ایران,2023-07-27T17:19:20+00:00,1052.0,266.0,244259.0,...,0,6,0,NaN,NaN,1,1,0,1,0
4,1.800450e+18,tifraghe_n,tifraghe_n,Tifraghe,parody and political satire,parody and political satire,2024-06-11T08:55:49+00:00,2838.0,1826.0,120843.0,...,0,0,0,NaN,NaN,1,1,0,1,0



Complete Multimodal Binary
Path:
C:\Users\p.vazifeh\Desktop\University\Project\Bot Detection Implementation\final_data\users\modeling_complete_multimodal_binary.csv

Exists: True

Shape:
(276, 56)

Columns:
['id', 'screen_name', 'user_key', 'name', 'description', 'clean_description', 'created_at', 'followers_count', 'friends_count', 'favourites_count', 'listed_count', 'media_count', 'statuses_count', 'status_count', 'normal_followers_count', 'user_age', 'follower_growth_rate', 'friends_growth_rate', 'default_profile', 'default_profile_image', 'has_custom_timelines', 'possibly_sensitive', 'hashtag_in_description', 'numbers_in_description', 'no_type_tweet', 'no_type_retweet_with_comment', 'no_type_reply', 'mean_no_media_per_tweet', 'mean_no_words', 'no_languages', 'mean_no_hashtags', 'mean_favourites_per_tweet', 'time_between_tweets', 'tweet_frequency', 'min_tweets_per_hour', 'min_tweets_per_day', 'max_tweets_per_hour', 'max_tweets_per_day', 'max_occurence_of_same_gap', 'unique_mention_

...


Graph Edges
Path:
C:\Users\p.vazifeh\Desktop\University\Project\Bot Detection Implementation\final_data\graph\graph_edges.csv

Exists: True

Shape:
(796485, 6)

Columns:
['source', 'target', 'relation', 'source_in_dataset', 'target_in_dataset', 'internal_edge']

First 5 rows:


,source,target,relation,source_in_dataset,target_in_dataset,internal_edge
0,chaiiee,SajjadSade48567,follows,True,True,True
1,dr_lizzii,SajjadSade48567,follows,False,True,False
2,nathaniel2026,SajjadSade48567,follows,False,True,False
3,julian270zi,SajjadSade48567,follows,False,True,False
4,kongoshadha,SajjadSade48567,follows,True,True,True



Graph User Statistics
Path:
C:\Users\p.vazifeh\Desktop\University\Project\Bot Detection Implementation\final_data\graph\graph_user_statistics.csv

Exists: True

Shape:
(3443, 8)

Columns:
['id', 'screen_name', 'followers', 'following', 'followers_list', 'following_list', 'collected_followers_count', 'collected_following_count']

First 5 rows:


,id,screen_name,followers,following,followers_list,following_list,collected_followers_count,collected_following_count
0,1.780000e+18,SajjadSade48567,"chaiiee, dr_lizzii, nathaniel2026, julian270zi...","grok, USABehFarsi, Psiphon_Fa, PersianDJT, Pho...","['chaiiee', 'dr_lizzii', 'nathaniel2026', 'jul...","['grok', 'USABehFarsi', 'Psiphon_Fa', 'Persian...",7,42
1,2.511595e+08,mojtaba2a,"aryan78093797, BanihashemiNav1, Riseagain979, ...","behrouzina, twiterrism819, monikaa2500, aryame...","['aryan78093797', 'BanihashemiNav1', 'Riseagai...","['behrouzina', 'twiterrism819', 'monikaa2500',...",193,160
2,1.790000e+18,niya64_,"Mentttaallll, TalbB909091, santinokarimi, Java...","chizbiin, Alireza774365, zahraaa1988, nooshiin...","['Mentttaallll', 'TalbB909091', 'santinokarimi...","['chizbiin', 'Alireza774365', 'zahraaa1988', '...",206,189
3,1.870000e+18,arash_the3rd,"Avayiran, darkpixele, sjavidnia, AnahitaSarab,...","pirozzzzzzzzz, Earendilist, salargholamiii, si...","['Avayiran', 'darkpixele', 'sjavidnia', 'Anahi...","['pirozzzzzzzzz', 'Earendilist', 'salargholami...",179,162
4,1.520000e+18,kaveAhanga2022,"leovirgo_17_, yutaabm, ShahdadmmMajid, Aliasad...","FanpageYaspah, leovirgo_17_, seyedhadikasaei, ...","['leovirgo_17_', 'yutaabm', 'ShahdadmmMajid', ...","['FanpageYaspah', 'leovirgo_17_', 'seyedhadika...",166,111



FINAL CONFIGURATION FILES

----------------------------------------------------------------------------------------------------
Model Feature Policy
----------------------------------------------------------------------------------------------------
Path:
C:\Users\p.vazifeh\Desktop\University\Project\Bot Detection Implementation\final_data\config\model_feature_policy.json

Exists: True

Content:
{
  "target": {
    "column": "label_binary",
    "human": 0,
    "bot": 1
  },
  "final_model_tabular_features": [
    "followers_count",
    "friends_count",
    "favourites_count",
    "listed_count",
    "media_count",
    "statuses_count",
    "status_count",
    "normal_followers_count",
    "user_age",
    "follower_growth_rate",
    "friends_growth_rate",
    "default_profile",
    "default_profile_image",
    "has_custom_timelines",
    "possibly_sensitive",
    "hashtag_in_description",
    "numbers_in_description",
    "no_type_tweet",
    "no_type_retweet_with_comment",
    "no_typ

In [19]:
# ============================================================
# Cell 16 — Final Cross-Modal Integrity Audit
# ============================================================

# ------------------------------------------------------------
# 1. Load from FINAL_DATA only
# ------------------------------------------------------------

users_labeled = final_datasets[
    "Labeled Binary Users"
].copy()

users_unlabeled = final_datasets[
    "True Unlabeled Users"
].copy()

users_complete = final_datasets[
    "Complete Multimodal Binary"
].copy()

tweets_final = final_datasets[
    "Tweets"
].copy()

graph_edges_final = final_datasets[
    "Graph Edges"
].copy()

graph_stats_final = final_datasets[
    "Graph User Statistics"
].copy()


# ------------------------------------------------------------
# 2. Canonical handle normalization
# ------------------------------------------------------------

def make_user_key(series):

    s = series.astype("string")

    s = (
        s
        .str.strip()
        .str.replace(
            r"^@",
            "",
            regex=True
        )
        .str.lower()
    )

    s = s.replace({
        "": pd.NA,
        "nan": pd.NA,
        "none": pd.NA,
        "<na>": pd.NA
    })

    return s


# ------------------------------------------------------------
# 3. Add canonical keys to Tweets and Graph
# ------------------------------------------------------------

tweets_final["user_key"] = make_user_key(
    tweets_final["screen_name"]
)

graph_stats_final["user_key"] = make_user_key(
    graph_stats_final["screen_name"]
)

graph_edges_final["source_key"] = make_user_key(
    graph_edges_final["source"]
)

graph_edges_final["target_key"] = make_user_key(
    graph_edges_final["target"]
)


# ------------------------------------------------------------
# 4. Build user sets
# ------------------------------------------------------------

L = set(
    users_labeled["user_key"]
    .dropna()
)

U = set(
    users_unlabeled["user_key"]
    .dropna()
)

C = set(
    users_complete["user_key"]
    .dropna()
)

T = set(
    tweets_final["user_key"]
    .dropna()
)

G_OWNER = set(
    graph_stats_final["user_key"]
    .dropna()
)

G_NODES = (
    set(
        graph_edges_final[
            "source_key"
        ].dropna()
    )
    |
    set(
        graph_edges_final[
            "target_key"
        ].dropna()
    )
)


# ------------------------------------------------------------
# 5. Core integrity checks
# ------------------------------------------------------------

print("=" * 85)
print("FINAL CROSS-MODAL INTEGRITY")
print("=" * 85)

print(f"Labeled users:                 {len(L)}")
print(f"True unlabeled users:          {len(U)}")
print(f"Complete multimodal users:     {len(C)}")

print(f"\nTweet users:                   {len(T)}")
print(f"Graph owner users:             {len(G_OWNER)}")
print(f"All graph nodes:               {len(G_NODES)}")


# ------------------------------------------------------------
# 6. Labeled ↔ Tweet coverage
# ------------------------------------------------------------

print("\n" + "=" * 85)
print("LABELED ↔ TWEETS")
print("=" * 85)

print(
    "Labeled users with tweets:",
    len(L & T)
)

print(
    "Labeled users missing tweets:",
    len(L - T)
)

print(
    "Tweet users outside labeled binary:",
    len(T - L)
)


# ------------------------------------------------------------
# 7. Labeled ↔ Graph coverage
# ------------------------------------------------------------

print("\n" + "=" * 85)
print("LABELED ↔ GRAPH")
print("=" * 85)

print(
    "Labeled users with graph owner record:",
    len(L & G_OWNER)
)

print(
    "Labeled users appearing anywhere in graph:",
    len(L & G_NODES)
)


# ------------------------------------------------------------
# 8. Complete multimodal validation
# ------------------------------------------------------------

print("\n" + "=" * 85)
print("COMPLETE MULTIMODAL VALIDATION")
print("=" * 85)

print(
    "Complete users with tweets:",
    len(C & T),
    "/",
    len(C)
)

print(
    "Complete users with graph owner record:",
    len(C & G_OWNER),
    "/",
    len(C)
)

print(
    "Complete users appearing in graph:",
    len(C & G_NODES),
    "/",
    len(C)
)


# ------------------------------------------------------------
# 9. Unlabeled graph coverage
# ------------------------------------------------------------

print("\n" + "=" * 85)
print("UNLABELED ↔ GRAPH")
print("=" * 85)

print(
    "Unlabeled users with graph owner record:",
    len(U & G_OWNER)
)

print(
    "Unlabeled users appearing anywhere in graph:",
    len(U & G_NODES)
)


# ------------------------------------------------------------
# 10. Graph statistics duplicate audit
# ------------------------------------------------------------

print("\n" + "=" * 85)
print("GRAPH USER STATISTICS DUPLICATES")
print("=" * 85)

valid_graph_stats = graph_stats_final[
    graph_stats_final["user_key"].notna()
].copy()

duplicate_graph_rows = (
    valid_graph_stats
    .duplicated(
        subset="user_key",
        keep=False
    )
)

print(
    "Rows:",
    len(valid_graph_stats)
)

print(
    "Unique graph owners:",
    valid_graph_stats[
        "user_key"
    ].nunique()
)

print(
    "Rows involved in duplicates:",
    int(
        duplicate_graph_rows.sum()
    )
)

print(
    "Duplicated user_keys:",
    valid_graph_stats.loc[
        duplicate_graph_rows,
        "user_key"
    ].nunique()
)


# ------------------------------------------------------------
# 11. Strong consistency assertions
# ------------------------------------------------------------

assert len(L) == 959
assert len(U) == 18331
assert len(C) == 276

assert len(L & U) == 0

assert C.issubset(L)

assert len(C & T) == 276

assert len(C & G_OWNER) == 276


print("\n" + "=" * 85)
print("FINAL RESULT")
print("=" * 85)

print(
    "Core cross-modal integrity checks passed."
)

FINAL CROSS-MODAL INTEGRITY
Labeled users:                 959
True unlabeled users:          18331
Complete multimodal users:     276

Tweet users:                   1099
Graph owner users:             3359
All graph nodes:               230791

LABELED ↔ TWEETS
Labeled users with tweets: 959
Labeled users missing tweets: 0
Tweet users outside labeled binary: 140

LABELED ↔ GRAPH
Labeled users with graph owner record: 276
Labeled users appearing anywhere in graph: 935

COMPLETE MULTIMODAL VALIDATION
Complete users with tweets: 276 / 276
Complete users with graph owner record: 276 / 276
Complete users appearing in graph: 276 / 276

UNLABELED ↔ GRAPH
Unlabeled users with graph owner record: 3033
Unlabeled users appearing anywhere in graph: 12450

GRAPH USER STATISTICS DUPLICATES
Rows: 3443
Unique graph owners: 3359
Rows involved in duplicates: 161
Duplicated user_keys: 77

FINAL RESULT
Core cross-modal integrity checks passed.


In [20]:
# ============================================================
# Cell 17 — Final User-Level Train / Validation / Test Split
# ============================================================

from pathlib import Path
import pandas as pd

from sklearn.model_selection import train_test_split


# ------------------------------------------------------------
# 1. Reproducibility
# ------------------------------------------------------------

SEED = 42


# ------------------------------------------------------------
# 2. Final operational paths
# ------------------------------------------------------------

FINAL_DATA_DIR = PROJECT_ROOT / "final_data"

LABELED_PATH = (
    FINAL_DATA_DIR
    / "users"
    / "modeling_labeled_binary.csv"
)


# ------------------------------------------------------------
# 3. Load final Gold Binary dataset
# ------------------------------------------------------------

gold_df = pd.read_csv(
    LABELED_PATH,
    low_memory=False
)


print("=" * 90)
print("GOLD BINARY DATASET")
print("=" * 90)

print("Shape:", gold_df.shape)

print(
    "\nLabel distribution:"
)

print(
    gold_df["label_raw"]
    .value_counts()
)

print(
    "\nGraph availability:"
)

print(
    gold_df["has_graph"]
    .value_counts()
    .sort_index()
)


# ------------------------------------------------------------
# 4. Basic integrity checks
# ------------------------------------------------------------

assert len(gold_df) == 959

assert gold_df["user_key"].notna().all()

assert gold_df["user_key"].is_unique

assert gold_df["label_binary"].notna().all()

assert set(
    gold_df["label_binary"].unique()
) == {0, 1}


# Every binary Gold user must have tweets
assert (
    gold_df["has_raw_tweets"] == 1
).all()


# Every binary Gold user must have profile + behavior
assert (
    gold_df["has_profile"] == 1
).all()

assert (
    gold_df["has_behavior_temporal"] == 1
).all()


# ------------------------------------------------------------
# 5. Inspect the four stratification groups
# ------------------------------------------------------------

gold_df["split_stratum"] = (
    gold_df["label_binary"]
    .astype(int)
    .astype(str)
    + "_graph_"
    + gold_df["has_graph"]
    .astype(int)
    .astype(str)
)


print("\n" + "=" * 90)
print("STRATIFICATION GROUPS")
print("=" * 90)

print(
    gold_df["split_stratum"]
    .value_counts()
    .sort_index()
)


print("\nReadable form:")

print(
    pd.crosstab(
        gold_df["label_raw"],
        gold_df["has_graph"],
        margins=True
    )
)


# ------------------------------------------------------------
# 6. First split
#
# 70% Train
# 30% Temporary
# ------------------------------------------------------------

train_df, temp_df = train_test_split(

    gold_df,

    test_size=0.30,

    random_state=SEED,

    stratify=gold_df["split_stratum"]
)


# ------------------------------------------------------------
# 7. Second split
#
# Temporary 30%
#      ↓
# 15% Validation
# 15% Test
# ------------------------------------------------------------

val_df, test_df = train_test_split(

    temp_df,

    test_size=0.50,

    random_state=SEED,

    stratify=temp_df["split_stratum"]
)


# ------------------------------------------------------------
# 8. Remove temporary helper column
# ------------------------------------------------------------

train_df = train_df.drop(
    columns=["split_stratum"]
).copy()

val_df = val_df.drop(
    columns=["split_stratum"]
).copy()

test_df = test_df.drop(
    columns=["split_stratum"]
).copy()


# ------------------------------------------------------------
# 9. Split report
# ------------------------------------------------------------

def report_split(df, split_name):

    print("\n" + "=" * 90)
    print(split_name)
    print("=" * 90)

    print(
        f"Users: {len(df)}"
    )

    print(
        "\nLabel distribution:"
    )

    print(
        df["label_raw"]
        .value_counts()
    )

    print(
        "\nGraph availability:"
    )

    print(
        df["has_graph"]
        .value_counts()
        .sort_index()
    )

    print(
        "\nLabel × Graph:"
    )

    print(
        pd.crosstab(
            df["label_raw"],
            df["has_graph"]
        )
    )


report_split(
    train_df,
    "TRAIN SET"
)

report_split(
    val_df,
    "VALIDATION SET"
)

report_split(
    test_df,
    "TEST SET"
)


# ------------------------------------------------------------
# 10. Leakage / overlap checks
# ------------------------------------------------------------

train_users = set(
    train_df["user_key"]
)

val_users = set(
    val_df["user_key"]
)

test_users = set(
    test_df["user_key"]
)


assert train_users.isdisjoint(
    val_users
)

assert train_users.isdisjoint(
    test_users
)

assert val_users.isdisjoint(
    test_users
)


assert (
    len(train_df)
    + len(val_df)
    + len(test_df)
    == 959
)


assert (
    train_users
    | val_users
    | test_users
) == set(
    gold_df["user_key"]
)


# ------------------------------------------------------------
# 11. Final split sizes
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("FINAL USER-LEVEL SPLIT")
print("=" * 90)

print(
    f"Train      : {len(train_df):4d} "
    f"({len(train_df) / 959 * 100:.2f}%)"
)

print(
    f"Validation : {len(val_df):4d} "
    f"({len(val_df) / 959 * 100:.2f}%)"
)

print(
    f"Test       : {len(test_df):4d} "
    f"({len(test_df) / 959 * 100:.2f}%)"
)

print(
    "\n✓ No user overlap between Train / Validation / Test"
)

print(
    "✓ All 959 Gold Binary users are accounted for"
)

print(
    "✓ Tweet availability is 100% for all Gold Binary users"
)

print(
    "✓ Graph availability was included in stratification"
)

GOLD BINARY DATASET
Shape: (959, 56)

Label distribution:
label_raw
human(2)    772
bot(1)      187
Name: count, dtype: int64

Graph availability:
has_graph
0    683
1    276
Name: count, dtype: int64

STRATIFICATION GROUPS
split_stratum
0_graph_0    540
0_graph_1    232
1_graph_0    143
1_graph_1     44
Name: count, dtype: int64

Readable form:
has_graph    0    1  All
label_raw               
bot(1)     143   44  187
human(2)   540  232  772
All        683  276  959

TRAIN SET
Users: 671

Label distribution:
label_raw
human(2)    540
bot(1)      131
Name: count, dtype: int64

Graph availability:
has_graph
0    478
1    193
Name: count, dtype: int64

Label × Graph:
has_graph    0    1
label_raw          
bot(1)     100   31
human(2)   378  162

VALIDATION SET
Users: 144

Label distribution:
label_raw
human(2)    116
bot(1)       28
Name: count, dtype: int64

Graph availability:
has_graph
0    102
1     42
Name: count, dtype: int64

Label × Graph:
has_graph   0   1
label_raw        
bo

In [21]:
# ============================================================
# Cell 18 — Split Tweets by User-Level Split
# ============================================================

import pandas as pd


# ------------------------------------------------------------
# 1. Load final raw tweet dataset
# ------------------------------------------------------------

TWEETS_PATH = (
    FINAL_DATA_DIR
    / "tweets"
    / "tweets_meta_data.csv"
)

tweets_df = pd.read_csv(
    TWEETS_PATH,
    low_memory=False
)


print("=" * 90)
print("RAW TWEET DATASET")
print("=" * 90)

print("Shape:", tweets_df.shape)
print("Rows:", len(tweets_df))
print(
    "Unique raw screen_names:",
    tweets_df["screen_name"].nunique(dropna=True)
)


# ------------------------------------------------------------
# 2. Create canonical user_key
# ------------------------------------------------------------

def normalize_user_key(series):

    s = (
        series
        .astype("string")
        .str.strip()
        .str.replace(
            r"^@",
            "",
            regex=True
        )
        .str.lower()
    )

    s = s.replace({
        "": pd.NA,
        "nan": pd.NA,
        "none": pd.NA,
        "<na>": pd.NA
    })

    return s


tweets_df["user_key"] = normalize_user_key(
    tweets_df["screen_name"]
)


print(
    "Unique normalized tweet users:",
    tweets_df["user_key"].nunique(dropna=True)
)

print(
    "Missing tweet user_key:",
    tweets_df["user_key"].isna().sum()
)


# ------------------------------------------------------------
# 3. User sets from Cell 17
# ------------------------------------------------------------

train_keys = set(
    train_df["user_key"]
)

val_keys = set(
    val_df["user_key"]
)

test_keys = set(
    test_df["user_key"]
)

gold_keys = (
    train_keys
    | val_keys
    | test_keys
)


# ------------------------------------------------------------
# 4. Keep only Binary Gold users
# ------------------------------------------------------------

gold_tweets = tweets_df[
    tweets_df["user_key"].isin(gold_keys)
].copy()


# ------------------------------------------------------------
# 5. Split tweets according to USER split
# ------------------------------------------------------------

train_tweets = gold_tweets[
    gold_tweets["user_key"].isin(train_keys)
].copy()

val_tweets = gold_tweets[
    gold_tweets["user_key"].isin(val_keys)
].copy()

test_tweets = gold_tweets[
    gold_tweets["user_key"].isin(test_keys)
].copy()


# ------------------------------------------------------------
# 6. Report
# ------------------------------------------------------------

def tweet_split_report(
    tweet_df,
    user_df,
    split_name
):

    print("\n" + "=" * 90)
    print(split_name)
    print("=" * 90)

    print(
        "Users expected:",
        len(user_df)
    )

    print(
        "Users with tweets:",
        tweet_df["user_key"].nunique()
    )

    print(
        "Tweet rows:",
        len(tweet_df)
    )

    tweets_per_user = (
        tweet_df
        .groupby("user_key")
        .size()
    )

    print(
        "Mean tweets/user:",
        round(
            tweets_per_user.mean(),
            2
        )
    )

    print(
        "Median tweets/user:",
        round(
            tweets_per_user.median(),
            2
        )
    )

    print(
        "Min tweets/user:",
        tweets_per_user.min()
    )

    print(
        "Max tweets/user:",
        tweets_per_user.max()
    )


tweet_split_report(
    train_tweets,
    train_df,
    "TRAIN TWEETS"
)

tweet_split_report(
    val_tweets,
    val_df,
    "VALIDATION TWEETS"
)

tweet_split_report(
    test_tweets,
    test_df,
    "TEST TWEETS"
)


# ------------------------------------------------------------
# 7. Strong integrity checks
# ------------------------------------------------------------

train_tweet_users = set(
    train_tweets["user_key"].unique()
)

val_tweet_users = set(
    val_tweets["user_key"].unique()
)

test_tweet_users = set(
    test_tweets["user_key"].unique()
)


# Every Gold Binary user must have tweets
assert train_tweet_users == train_keys

assert val_tweet_users == val_keys

assert test_tweet_users == test_keys


# No user leakage
assert train_tweet_users.isdisjoint(
    val_tweet_users
)

assert train_tweet_users.isdisjoint(
    test_tweet_users
)

assert val_tweet_users.isdisjoint(
    test_tweet_users
)


# All Binary Gold tweets accounted for
assert (
    len(train_tweets)
    + len(val_tweets)
    + len(test_tweets)
    == len(gold_tweets)
)


# ------------------------------------------------------------
# 8. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("FINAL TWEET SPLIT SUMMARY")
print("=" * 90)

print(
    f"Train tweets      : {len(train_tweets):,}"
)

print(
    f"Validation tweets : {len(val_tweets):,}"
)

print(
    f"Test tweets       : {len(test_tweets):,}"
)

print(
    f"\nBinary Gold tweets: {len(gold_tweets):,}"
)

print(
    f"Tweet rows excluded because user is outside "
    f"Binary Gold set: "
    f"{len(tweets_df) - len(gold_tweets):,}"
)

print(
    "\n✓ Tweets follow the User-level split"
)

print(
    "✓ No user appears in more than one Tweet split"
)

print(
    "✓ Every one of the 959 Gold Binary users has tweets"
)

RAW TWEET DATASET
Shape: (75913, 38)
Rows: 75913
Unique raw screen_names: 1099
Unique normalized tweet users: 1099
Missing tweet user_key: 0

TRAIN TWEETS
Users expected: 671
Users with tweets: 671
Tweet rows: 45443
Mean tweets/user: 67.72
Median tweets/user: 9.0
Min tweets/user: 1
Max tweets/user: 2401

VALIDATION TWEETS
Users expected: 144
Users with tweets: 144
Tweet rows: 13968
Mean tweets/user: 97.0
Median tweets/user: 7.5
Min tweets/user: 1
Max tweets/user: 3439

TEST TWEETS
Users expected: 144
Users with tweets: 144
Tweet rows: 11464
Mean tweets/user: 79.61
Median tweets/user: 12.5
Min tweets/user: 1
Max tweets/user: 974

FINAL TWEET SPLIT SUMMARY
Train tweets      : 45,443
Validation tweets : 13,968
Test tweets       : 11,464

Binary Gold tweets: 70,875
Tweet rows excluded because user is outside Binary Gold set: 5,038

✓ Tweets follow the User-level split
✓ No user appears in more than one Tweet split
✓ Every one of the 959 Gold Binary users has tweets


In [22]:
# ============================================================
# Cell 19 — Graph Owner Split by User-Level Split
# ============================================================

import pandas as pd


# ------------------------------------------------------------
# 1. Load final graph user statistics
# ------------------------------------------------------------

GRAPH_STATS_PATH = (
    FINAL_DATA_DIR
    / "graph"
    / "graph_user_statistics.csv"
)

graph_stats = pd.read_csv(
    GRAPH_STATS_PATH,
    low_memory=False
)


print("=" * 90)
print("RAW GRAPH OWNER DATA")
print("=" * 90)

print(
    "Rows:",
    len(graph_stats)
)

print(
    "Raw unique screen_names:",
    graph_stats["screen_name"].nunique(dropna=True)
)


# ------------------------------------------------------------
# 2. Create canonical user_key
# ------------------------------------------------------------

graph_stats["user_key"] = normalize_user_key(
    graph_stats["screen_name"]
)


print(
    "Normalized unique graph owners:",
    graph_stats["user_key"].nunique(dropna=True)
)

print(
    "Missing user_key:",
    graph_stats["user_key"].isna().sum()
)


# ------------------------------------------------------------
# 3. Duplicate audit
# ------------------------------------------------------------

duplicate_mask = (
    graph_stats["user_key"]
    .notna()
    &
    graph_stats.duplicated(
        subset="user_key",
        keep=False
    )
)

print("\n" + "=" * 90)
print("DUPLICATE GRAPH OWNER AUDIT")
print("=" * 90)

print(
    "Rows involved in duplicate user_keys:",
    int(duplicate_mask.sum())
)

print(
    "Duplicated user_keys:",
    graph_stats.loc[
        duplicate_mask,
        "user_key"
    ].nunique()
)


# ------------------------------------------------------------
# 4. User sets already created from Cell 17
# ------------------------------------------------------------

# train_keys
# val_keys
# test_keys


# ------------------------------------------------------------
# 5. Select Gold Binary graph owners
# ------------------------------------------------------------

gold_graph_stats = graph_stats[
    graph_stats["user_key"].isin(
        train_keys | val_keys | test_keys
    )
].copy()


train_graph_stats = gold_graph_stats[
    gold_graph_stats["user_key"].isin(
        train_keys
    )
].copy()

val_graph_stats = gold_graph_stats[
    gold_graph_stats["user_key"].isin(
        val_keys
    )
].copy()

test_graph_stats = gold_graph_stats[
    gold_graph_stats["user_key"].isin(
        test_keys
    )
].copy()


# ------------------------------------------------------------
# 6. Report unique graph owners
# ------------------------------------------------------------

def graph_owner_report(
    df,
    expected_user_df,
    split_name
):

    actual_users = set(
        df["user_key"]
        .dropna()
        .unique()
    )

    expected_graph_users = set(
        expected_user_df.loc[
            expected_user_df["has_graph"] == 1,
            "user_key"
        ]
    )

    print("\n" + "=" * 90)
    print(split_name)
    print("=" * 90)

    print(
        "Graph rows:",
        len(df)
    )

    print(
        "Unique graph owners:",
        len(actual_users)
    )

    print(
        "Expected graph owners:",
        len(expected_graph_users)
    )

    print(
        "Missing expected graph owners:",
        len(
            expected_graph_users
            - actual_users
        )
    )

    print(
        "Unexpected graph owners:",
        len(
            actual_users
            - expected_graph_users
        )
    )


graph_owner_report(
    train_graph_stats,
    train_df,
    "TRAIN GRAPH OWNERS"
)

graph_owner_report(
    val_graph_stats,
    val_df,
    "VALIDATION GRAPH OWNERS"
)

graph_owner_report(
    test_graph_stats,
    test_df,
    "TEST GRAPH OWNERS"
)


# ------------------------------------------------------------
# 7. Strong checks
# ------------------------------------------------------------

train_graph_users = set(
    train_graph_stats["user_key"]
    .dropna()
)

val_graph_users = set(
    val_graph_stats["user_key"]
    .dropna()
)

test_graph_users = set(
    test_graph_stats["user_key"]
    .dropna()
)


assert len(train_graph_users) == 193
assert len(val_graph_users) == 42
assert len(test_graph_users) == 41


assert train_graph_users.isdisjoint(
    val_graph_users
)

assert train_graph_users.isdisjoint(
    test_graph_users
)

assert val_graph_users.isdisjoint(
    test_graph_users
)


# Every graph owner must match has_graph=1
assert train_graph_users == set(
    train_df.loc[
        train_df["has_graph"] == 1,
        "user_key"
    ]
)

assert val_graph_users == set(
    val_df.loc[
        val_df["has_graph"] == 1,
        "user_key"
    ]
)

assert test_graph_users == set(
    test_df.loc[
        test_df["has_graph"] == 1,
        "user_key"
    ]
)


# ------------------------------------------------------------
# 8. Final result
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("FINAL GRAPH OWNER SPLIT")
print("=" * 90)

print(
    f"Train graph owners      : {len(train_graph_users)}"
)

print(
    f"Validation graph owners : {len(val_graph_users)}"
)

print(
    f"Test graph owners       : {len(test_graph_users)}"
)

print(
    "\n✓ Graph owners follow the original User-level split"
)

print(
    "✓ No graph-owner user leakage between splits"
)

print(
    "✓ has_graph masks are consistent with graph records"
)

RAW GRAPH OWNER DATA
Rows: 3443
Raw unique screen_names: 3359
Normalized unique graph owners: 3359
Missing user_key: 0

DUPLICATE GRAPH OWNER AUDIT
Rows involved in duplicate user_keys: 161
Duplicated user_keys: 77

TRAIN GRAPH OWNERS
Graph rows: 193
Unique graph owners: 193
Expected graph owners: 193
Missing expected graph owners: 0
Unexpected graph owners: 0

VALIDATION GRAPH OWNERS
Graph rows: 42
Unique graph owners: 42
Expected graph owners: 42
Missing expected graph owners: 0
Unexpected graph owners: 0

TEST GRAPH OWNERS
Graph rows: 41
Unique graph owners: 41
Expected graph owners: 41
Missing expected graph owners: 0
Unexpected graph owners: 0

FINAL GRAPH OWNER SPLIT
Train graph owners      : 193
Validation graph owners : 42
Test graph owners       : 41

✓ Graph owners follow the original User-level split
✓ No graph-owner user leakage between splits
✓ has_graph masks are consistent with graph records


In [23]:
# ============================================================
# Cell 20 — Audit Graph Edges Across User Splits
# ============================================================

import pandas as pd


# ------------------------------------------------------------
# 1. Load graph edges
# ------------------------------------------------------------

GRAPH_EDGES_PATH = (
    FINAL_DATA_DIR
    / "graph"
    / "graph_edges.csv"
)

graph_edges = pd.read_csv(
    GRAPH_EDGES_PATH,
    low_memory=False
)


print("=" * 90)
print("RAW GRAPH EDGES")
print("=" * 90)

print("Rows:", len(graph_edges))

print(
    "\nRelation distribution:"
)

print(
    graph_edges["relation"]
    .value_counts(dropna=False)
)


# ------------------------------------------------------------
# 2. Normalize source / target usernames
# ------------------------------------------------------------

graph_edges["source_key"] = normalize_user_key(
    graph_edges["source"]
)

graph_edges["target_key"] = normalize_user_key(
    graph_edges["target"]
)


print(
    "\nUnique source nodes:",
    graph_edges["source_key"].nunique(dropna=True)
)

print(
    "Unique target nodes:",
    graph_edges["target_key"].nunique(dropna=True)
)

all_graph_nodes = (
    set(graph_edges["source_key"].dropna())
    |
    set(graph_edges["target_key"].dropna())
)

print(
    "Total unique graph nodes:",
    len(all_graph_nodes)
)


# ------------------------------------------------------------
# 3. User sets
# ------------------------------------------------------------

train_keys = set(
    train_df["user_key"]
)

val_keys = set(
    val_df["user_key"]
)

test_keys = set(
    test_df["user_key"]
)

gold_keys = (
    train_keys
    | val_keys
    | test_keys
)


# Unlabeled operational users
UNLABELED_PATH = (
    FINAL_DATA_DIR
    / "users"
    / "modeling_true_unlabeled.csv"
)

unlabeled_df = pd.read_csv(
    UNLABELED_PATH,
    low_memory=False
)

unlabeled_keys = set(
    unlabeled_df["user_key"]
    .dropna()
)


# ------------------------------------------------------------
# 4. Assign each graph node to a group
# ------------------------------------------------------------

def node_group(user_key):

    if pd.isna(user_key):
        return "missing"

    if user_key in train_keys:
        return "train"

    if user_key in val_keys:
        return "validation"

    if user_key in test_keys:
        return "test"

    if user_key in unlabeled_keys:
        return "unlabeled"

    return "external"


graph_edges["source_group"] = (
    graph_edges["source_key"]
    .map(node_group)
)

graph_edges["target_group"] = (
    graph_edges["target_key"]
    .map(node_group)
)


# ------------------------------------------------------------
# 5. Edge group pair
# ------------------------------------------------------------

graph_edges["split_pair"] = (
    graph_edges["source_group"]
    + " -> "
    + graph_edges["target_group"]
)


print("\n" + "=" * 90)
print("EDGE DISTRIBUTION BY NODE GROUP")
print("=" * 90)

print(
    graph_edges["split_pair"]
    .value_counts()
)


# ------------------------------------------------------------
# 6. Gold-to-Gold edges
# ------------------------------------------------------------

gold_to_gold = graph_edges[
    graph_edges["source_key"].isin(gold_keys)
    &
    graph_edges["target_key"].isin(gold_keys)
].copy()


print("\n" + "=" * 90)
print("GOLD ↔ GOLD EDGES")
print("=" * 90)

print(
    "Total Gold-to-Gold edges:",
    len(gold_to_gold)
)

print(
    "\nDistribution:"
)

print(
    gold_to_gold["split_pair"]
    .value_counts()
)


# ------------------------------------------------------------
# 7. Same-split vs Cross-split Gold edges
# ------------------------------------------------------------

same_split_gold = gold_to_gold[
    gold_to_gold["source_group"]
    ==
    gold_to_gold["target_group"]
]

cross_split_gold = gold_to_gold[
    gold_to_gold["source_group"]
    !=
    gold_to_gold["target_group"]
]


print("\nSame-split Gold edges:",
      len(same_split_gold))

print(
    "Cross-split Gold edges:",
    len(cross_split_gold)
)


if len(gold_to_gold) > 0:

    print(
        "Cross-split percentage:",
        round(
            len(cross_split_gold)
            / len(gold_to_gold)
            * 100,
            2
        ),
        "%"
    )


# ------------------------------------------------------------
# 8. Edges touching each supervised split
# ------------------------------------------------------------

def touching_count(keys):

    mask = (
        graph_edges["source_key"].isin(keys)
        |
        graph_edges["target_key"].isin(keys)
    )

    return int(mask.sum())


print("\n" + "=" * 90)
print("EDGES TOUCHING SUPERVISED USERS")
print("=" * 90)

print(
    "Edges touching Train users:",
    touching_count(train_keys)
)

print(
    "Edges touching Validation users:",
    touching_count(val_keys)
)

print(
    "Edges touching Test users:",
    touching_count(test_keys)
)


# ------------------------------------------------------------
# 9. Graph-owner-centered edge coverage
# ------------------------------------------------------------

def owner_edge_report(owner_keys, name):

    mask = (
        graph_edges["source_key"].isin(owner_keys)
        |
        graph_edges["target_key"].isin(owner_keys)
    )

    subset = graph_edges[mask]

    print("\n" + "-" * 90)
    print(name)
    print("-" * 90)

    print(
        "Owners:",
        len(owner_keys)
    )

    print(
        "Edges touching owners:",
        len(subset)
    )

    print(
        "Unique neighboring nodes:",
        len(
            set(subset["source_key"].dropna())
            |
            set(subset["target_key"].dropna())
        )
    )


owner_edge_report(
    train_graph_users,
    "TRAIN GRAPH OWNERS"
)

owner_edge_report(
    val_graph_users,
    "VALIDATION GRAPH OWNERS"
)

owner_edge_report(
    test_graph_users,
    "TEST GRAPH OWNERS"
)


# ------------------------------------------------------------
# 10. Explicit cross-split supervised connections
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("CROSS-SPLIT GOLD CONNECTIONS")
print("=" * 90)

cross_table = pd.crosstab(
    gold_to_gold["source_group"],
    gold_to_gold["target_group"]
)

print(cross_table)


# ------------------------------------------------------------
# 11. Final audit result
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("GRAPH EDGE AUDIT COMPLETE")
print("=" * 90)

print(
    "✓ No graph edges were modified."
)

print(
    "✓ Train / Validation / Test relationships are now explicitly measured."
)

print(
    "✓ Next step: define leakage-safe graph construction."
)

RAW GRAPH EDGES
Rows: 796485

Relation distribution:
relation
follows    796485
Name: count, dtype: int64

Unique source nodes: 164311
Unique target nodes: 111117
Total unique graph nodes: 230791

EDGE DISTRIBUTION BY NODE GROUP
split_pair
external -> unlabeled       232504
unlabeled -> external       215683
unlabeled -> unlabeled      166403
unlabeled -> train           47484
external -> train            29281
train -> unlabeled           20293
train -> external            17599
external -> external         12312
unlabeled -> validation      11119
unlabeled -> test            10940
external -> validation        6552
external -> test              6022
test -> unlabeled             4530
validation -> unlabeled       3865
validation -> external        3766
test -> external              3532
train -> train                2171
validation -> train            559
train -> test                  511
train -> validation            481
test -> train                  457
validation -> test       

In [24]:
# ============================================================
# Cell 21 — Build Leakage-Safe Owner-Centric Graph Splits
# ============================================================

import ast
import pandas as pd


# ------------------------------------------------------------
# 1. Helper for parsing follower/following lists
# ------------------------------------------------------------

def parse_username_list(value):

    if pd.isna(value):
        return []

    # Already a Python list
    if isinstance(value, list):
        return value

    value = str(value).strip()

    if value == "":
        return []

    # Preferred: parse string representation of list
    try:
        parsed = ast.literal_eval(value)

        if isinstance(parsed, list):
            return parsed

    except Exception:
        pass

    # Fallback: comma-separated string
    return [
        item.strip()
        for item in value.split(",")
        if item.strip()
    ]


def normalize_single_username(value):

    if pd.isna(value):
        return None

    value = str(value).strip()

    if value.startswith("@"):
        value = value[1:]

    value = value.lower().strip()

    if value in {"", "nan", "none", "<na>"}:
        return None

    return value


# ------------------------------------------------------------
# 2. Gold split membership
# ------------------------------------------------------------

gold_split_map = {}

for key in train_keys:
    gold_split_map[key] = "train"

for key in val_keys:
    gold_split_map[key] = "validation"

for key in test_keys:
    gold_split_map[key] = "test"


# ------------------------------------------------------------
# 3. Build graph edges centered on graph owners
# ------------------------------------------------------------

def build_owner_graph(
    owner_df,
    split_name
):

    edges = []

    for _, row in owner_df.iterrows():

        owner = row["user_key"]

        if pd.isna(owner):
            continue

        owner = str(owner)

        # ----------------------------------------
        # Followers:
        #
        # follower -> owner
        # ----------------------------------------

        followers = parse_username_list(
            row["followers_list"]
        )

        for follower in followers:

            follower_key = normalize_single_username(
                follower
            )

            if follower_key is None:
                continue

            # If neighbor is Gold and belongs
            # to another split -> remove edge
            neighbor_split = gold_split_map.get(
                follower_key
            )

            if (
                neighbor_split is not None
                and neighbor_split != split_name
            ):
                continue

            edges.append({
                "source_key": follower_key,
                "target_key": owner,
                "relation": "follows",
                "owner_key": owner,
                "owner_split": split_name
            })


        # ----------------------------------------
        # Following:
        #
        # owner -> followed user
        # ----------------------------------------

        following = parse_username_list(
            row["following_list"]
        )

        for followed in following:

            followed_key = normalize_single_username(
                followed
            )

            if followed_key is None:
                continue

            neighbor_split = gold_split_map.get(
                followed_key
            )

            if (
                neighbor_split is not None
                and neighbor_split != split_name
            ):
                continue

            edges.append({
                "source_key": owner,
                "target_key": followed_key,
                "relation": "follows",
                "owner_key": owner,
                "owner_split": split_name
            })


    edges_df = pd.DataFrame(edges)

    if len(edges_df) > 0:

        edges_df = (
            edges_df
            .drop_duplicates(
                subset=[
                    "source_key",
                    "target_key",
                    "relation",
                    "owner_key"
                ]
            )
            .reset_index(drop=True)
        )

    return edges_df


# ------------------------------------------------------------
# 4. Build Train / Validation / Test graphs
# ------------------------------------------------------------

train_graph_edges = build_owner_graph(
    train_graph_stats,
    "train"
)

val_graph_edges = build_owner_graph(
    val_graph_stats,
    "validation"
)

test_graph_edges = build_owner_graph(
    test_graph_stats,
    "test"
)


# ------------------------------------------------------------
# 5. Helper: graph audit
# ------------------------------------------------------------

def graph_split_audit(
    edges_df,
    owner_keys,
    split_name
):

    print("\n" + "=" * 90)
    print(f"{split_name.upper()} INDUCTIVE GRAPH")
    print("=" * 90)

    print(
        "Graph owners:",
        len(owner_keys)
    )

    print(
        "Edges:",
        len(edges_df)
    )

    nodes = (
        set(edges_df["source_key"])
        |
        set(edges_df["target_key"])
    )

    print(
        "Unique nodes:",
        len(nodes)
    )

    print(
        "Mean edges per owner:",
        round(
            len(edges_df) / len(owner_keys),
            2
        )
    )

    # ----------------------------------------
    # Gold neighbors by split
    # ----------------------------------------

    gold_nodes = nodes & gold_keys

    gold_neighbor_groups = {
        "train": len(
            gold_nodes & train_keys
        ),
        "validation": len(
            gold_nodes & val_keys
        ),
        "test": len(
            gold_nodes & test_keys
        )
    }

    print(
        "\nGold nodes appearing in graph:"
    )

    print(
        gold_neighbor_groups
    )

    # ----------------------------------------
    # Detect forbidden Gold users
    # ----------------------------------------

    allowed_gold = {
        "train": train_keys,
        "validation": val_keys,
        "test": test_keys
    }[split_name]

    forbidden_gold = (
        gold_nodes
        - allowed_gold
    )

    print(
        "Forbidden cross-split Gold nodes:",
        len(forbidden_gold)
    )

    return nodes, forbidden_gold


# ------------------------------------------------------------
# 6. Run audits
# ------------------------------------------------------------

train_graph_nodes, train_forbidden = (
    graph_split_audit(
        train_graph_edges,
        train_graph_users,
        "train"
    )
)

val_graph_nodes, val_forbidden = (
    graph_split_audit(
        val_graph_edges,
        val_graph_users,
        "validation"
    )
)

test_graph_nodes, test_forbidden = (
    graph_split_audit(
        test_graph_edges,
        test_graph_users,
        "test"
    )
)


# ------------------------------------------------------------
# 7. Strong leakage checks
# ------------------------------------------------------------

assert len(train_forbidden) == 0
assert len(val_forbidden) == 0
assert len(test_forbidden) == 0


assert set(
    train_graph_edges["owner_key"]
) == train_graph_users

assert set(
    val_graph_edges["owner_key"]
) == val_graph_users

assert set(
    test_graph_edges["owner_key"]
) == test_graph_users


# ------------------------------------------------------------
# 8. Cross-split owner leakage check
# ------------------------------------------------------------

assert set(
    train_graph_edges["owner_key"]
).isdisjoint(
    set(
        val_graph_edges["owner_key"]
    )
)

assert set(
    train_graph_edges["owner_key"]
).isdisjoint(
    set(
        test_graph_edges["owner_key"]
    )
)

assert set(
    val_graph_edges["owner_key"]
).isdisjoint(
    set(
        test_graph_edges["owner_key"]
    )
)


# ------------------------------------------------------------
# 9. Final result
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("INDUCTIVE GRAPH CONSTRUCTION COMPLETE")
print("=" * 90)

print(
    f"Train edges      : {len(train_graph_edges):,}"
)

print(
    f"Validation edges : {len(val_graph_edges):,}"
)

print(
    f"Test edges       : {len(test_graph_edges):,}"
)

print(
    "\n✓ No cross-split Gold user appears as a graph neighbor."
)

print(
    "✓ Graph owners exactly follow the User-level split."
)

print(
    "✓ External/unlabeled neighbors are preserved as structural context."
)


TRAIN INDUCTIVE GRAPH
Graph owners: 193
Edges: 60936
Unique nodes: 42954
Mean edges per owner: 315.73

Gold nodes appearing in graph:
{'train': 573, 'validation': 0, 'test': 0}
Forbidden cross-split Gold nodes: 0

VALIDATION INDUCTIVE GRAPH
Graph owners: 42
Edges: 13006
Unique nodes: 11436
Mean edges per owner: 309.67

Gold nodes appearing in graph:
{'train': 0, 'validation': 84, 'test': 0}
Forbidden cross-split Gold nodes: 0

TEST INDUCTIVE GRAPH
Graph owners: 41
Edges: 12536
Unique nodes: 10453
Mean edges per owner: 305.76

Gold nodes appearing in graph:
{'train': 0, 'validation': 0, 'test': 89}
Forbidden cross-split Gold nodes: 0

INDUCTIVE GRAPH CONSTRUCTION COMPLETE
Train edges      : 60,936
Validation edges : 13,006
Test edges       : 12,536

✓ No cross-split Gold user appears as a graph neighbor.
✓ Graph owners exactly follow the User-level split.
✓ External/unlabeled neighbors are preserved as structural context.


In [25]:
# ============================================================
# Cell 22 — Save and Freeze Final Data Splits
# ============================================================

from pathlib import Path
import json
import pandas as pd


# ------------------------------------------------------------
# 1. Output directories
# ------------------------------------------------------------

SPLITS_DIR = FINAL_DATA_DIR / "splits"

SPLIT_USERS_DIR = SPLITS_DIR / "users"
SPLIT_TWEETS_DIR = SPLITS_DIR / "tweets"
SPLIT_GRAPH_DIR = SPLITS_DIR / "graph"

for directory in [
    SPLITS_DIR,
    SPLIT_USERS_DIR,
    SPLIT_TWEETS_DIR,
    SPLIT_GRAPH_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )


# ------------------------------------------------------------
# 2. Final validation BEFORE saving
# ------------------------------------------------------------

# User splits
assert len(train_df) == 671
assert len(val_df) == 144
assert len(test_df) == 144

assert set(train_df["user_key"]).isdisjoint(
    set(val_df["user_key"])
)

assert set(train_df["user_key"]).isdisjoint(
    set(test_df["user_key"])
)

assert set(val_df["user_key"]).isdisjoint(
    set(test_df["user_key"])
)


# Tweet splits
assert train_tweets["user_key"].nunique() == 671
assert val_tweets["user_key"].nunique() == 144
assert test_tweets["user_key"].nunique() == 144


# Graph owner splits
assert len(train_graph_users) == 193
assert len(val_graph_users) == 42
assert len(test_graph_users) == 41


# Graph leakage checks
assert len(train_forbidden) == 0
assert len(val_forbidden) == 0
assert len(test_forbidden) == 0


# ------------------------------------------------------------
# 3. Save USER splits
# ------------------------------------------------------------

train_df.to_csv(
    SPLIT_USERS_DIR / "train_users.csv",
    index=False
)

val_df.to_csv(
    SPLIT_USERS_DIR / "validation_users.csv",
    index=False
)

test_df.to_csv(
    SPLIT_USERS_DIR / "test_users.csv",
    index=False
)


# ------------------------------------------------------------
# 4. Save compact split assignment file
# ------------------------------------------------------------

train_assignment = train_df[
    ["user_key", "label_binary", "has_graph"]
].copy()

train_assignment["split"] = "train"


val_assignment = val_df[
    ["user_key", "label_binary", "has_graph"]
].copy()

val_assignment["split"] = "validation"


test_assignment = test_df[
    ["user_key", "label_binary", "has_graph"]
].copy()

test_assignment["split"] = "test"


split_assignment = pd.concat(
    [
        train_assignment,
        val_assignment,
        test_assignment
    ],
    ignore_index=True
)


assert len(split_assignment) == 959
assert split_assignment["user_key"].is_unique


split_assignment.to_csv(
    SPLITS_DIR / "user_split_assignment.csv",
    index=False
)


# ------------------------------------------------------------
# 5. Save TWEET splits
# ------------------------------------------------------------

train_tweets.to_csv(
    SPLIT_TWEETS_DIR / "train_tweets.csv",
    index=False
)

val_tweets.to_csv(
    SPLIT_TWEETS_DIR / "validation_tweets.csv",
    index=False
)

test_tweets.to_csv(
    SPLIT_TWEETS_DIR / "test_tweets.csv",
    index=False
)


# ------------------------------------------------------------
# 6. Save leakage-safe GRAPH edge splits
# ------------------------------------------------------------

train_graph_edges.to_csv(
    SPLIT_GRAPH_DIR / "train_graph_edges.csv",
    index=False
)

val_graph_edges.to_csv(
    SPLIT_GRAPH_DIR / "validation_graph_edges.csv",
    index=False
)

test_graph_edges.to_csv(
    SPLIT_GRAPH_DIR / "test_graph_edges.csv",
    index=False
)


# ------------------------------------------------------------
# 7. Save graph-owner records
# ------------------------------------------------------------

train_graph_stats.to_csv(
    SPLIT_GRAPH_DIR / "train_graph_owners.csv",
    index=False
)

val_graph_stats.to_csv(
    SPLIT_GRAPH_DIR / "validation_graph_owners.csv",
    index=False
)

test_graph_stats.to_csv(
    SPLIT_GRAPH_DIR / "test_graph_owners.csv",
    index=False
)


# ------------------------------------------------------------
# 8. Split manifest
# ------------------------------------------------------------

split_manifest = {

    "version": "phase2_final_split_v1",

    "random_seed": SEED,

    "split_strategy": {
        "level": "user",
        "train_ratio": 0.70,
        "validation_ratio": 0.15,
        "test_ratio": 0.15,
        "stratified_by": [
            "label_binary",
            "has_graph"
        ]
    },

    "users": {
        "total": 959,
        "train": len(train_df),
        "validation": len(val_df),
        "test": len(test_df)
    },

    "labels": {
        "train": {
            "human": int(
                (train_df["label_binary"] == 0).sum()
            ),
            "bot": int(
                (train_df["label_binary"] == 1).sum()
            )
        },

        "validation": {
            "human": int(
                (val_df["label_binary"] == 0).sum()
            ),
            "bot": int(
                (val_df["label_binary"] == 1).sum()
            )
        },

        "test": {
            "human": int(
                (test_df["label_binary"] == 0).sum()
            ),
            "bot": int(
                (test_df["label_binary"] == 1).sum()
            )
        }
    },

    "tweets": {
        "train_rows": len(train_tweets),
        "validation_rows": len(val_tweets),
        "test_rows": len(test_tweets),
        "total_binary_gold_tweets": len(gold_tweets)
    },

    "graph": {
        "train_owners": len(train_graph_users),
        "validation_owners": len(val_graph_users),
        "test_owners": len(test_graph_users),

        "train_edges": len(train_graph_edges),
        "validation_edges": len(val_graph_edges),
        "test_edges": len(test_graph_edges),

        "evaluation_type": "inductive",

        "cross_split_gold_neighbors_allowed": False,

        "external_and_unlabeled_neighbors_preserved": True
    },

    "leakage_policy": {
        "user_overlap": False,
        "tweet_user_overlap": False,
        "cross_split_gold_graph_neighbors": False
    }
}


with open(
    SPLITS_DIR / "split_manifest.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        split_manifest,
        f,
        ensure_ascii=False,
        indent=2
    )


# ------------------------------------------------------------
# 9. Final report
# ------------------------------------------------------------

print("=" * 90)
print("FINAL SPLITS SAVED")
print("=" * 90)

print("\nUSER SPLITS")
print(
    f"Train      : {len(train_df):,}"
)
print(
    f"Validation : {len(val_df):,}"
)
print(
    f"Test       : {len(test_df):,}"
)


print("\nTWEET SPLITS")
print(
    f"Train      : {len(train_tweets):,}"
)
print(
    f"Validation : {len(val_tweets):,}"
)
print(
    f"Test       : {len(test_tweets):,}"
)


print("\nGRAPH SPLITS")
print(
    f"Train      : {len(train_graph_edges):,} edges"
)
print(
    f"Validation : {len(val_graph_edges):,} edges"
)
print(
    f"Test       : {len(test_graph_edges):,} edges"
)


print("\nSaved to:")
print(SPLITS_DIR)


print("\n✓ User-level split frozen")
print("✓ Tweet splits follow user split")
print("✓ Graph splits are inductive")
print("✓ Cross-split Gold graph leakage removed")
print("✓ Split manifest saved")

FINAL SPLITS SAVED

USER SPLITS
Train      : 671
Validation : 144
Test       : 144

TWEET SPLITS
Train      : 45,443
Validation : 13,968
Test       : 11,464

GRAPH SPLITS
Train      : 60,936 edges
Validation : 13,006 edges
Test       : 12,536 edges

Saved to:
C:\Users\p.vazifeh\Desktop\University\Project\Bot Detection Implementation\final_data\splits

✓ User-level split frozen
✓ Tweet splits follow user split
✓ Graph splits are inductive
✓ Cross-split Gold graph leakage removed
✓ Split manifest saved


In [26]:
# ============================================================
# Cell 23 — Final Tabular Feature Missingness Audit
# ============================================================

import json
import pandas as pd


# ------------------------------------------------------------
# 1. Load frozen splits from disk
# ------------------------------------------------------------

SPLITS_DIR = FINAL_DATA_DIR / "splits"

train_users = pd.read_csv(
    SPLITS_DIR / "users" / "train_users.csv",
    low_memory=False
)

val_users = pd.read_csv(
    SPLITS_DIR / "users" / "validation_users.csv",
    low_memory=False
)

test_users = pd.read_csv(
    SPLITS_DIR / "users" / "test_users.csv",
    low_memory=False
)


# ------------------------------------------------------------
# 2. Load final feature policy
# ------------------------------------------------------------

FEATURE_POLICY_PATH = (
    FINAL_DATA_DIR
    / "config"
    / "model_feature_policy.json"
)

with open(
    FEATURE_POLICY_PATH,
    "r",
    encoding="utf-8"
) as f:
    feature_policy = json.load(f)


final_features = (
    feature_policy["final_model_tabular_features"]
)

numeric_features = (
    feature_policy["numeric_features"]
)

binary_features = (
    feature_policy["binary_features"]
)


# ------------------------------------------------------------
# 3. Basic checks
# ------------------------------------------------------------

assert len(final_features) == 42
assert len(numeric_features) == 36
assert len(binary_features) == 6

assert set(final_features) == (
    set(numeric_features)
    | set(binary_features)
)

for feature in final_features:

    assert feature in train_users.columns
    assert feature in val_users.columns
    assert feature in test_users.columns


print("=" * 90)
print("FINAL TABULAR FEATURE SPACE")
print("=" * 90)

print(
    "Total features:",
    len(final_features)
)

print(
    "Numeric features:",
    len(numeric_features)
)

print(
    "Binary features:",
    len(binary_features)
)


# ------------------------------------------------------------
# 4. Missingness helper
# ------------------------------------------------------------

def missingness_table(df):

    missing_count = (
        df[final_features]
        .isna()
        .sum()
    )

    missing_pct = (
        missing_count
        / len(df)
        * 100
    )

    result = pd.DataFrame({
        "missing_count": missing_count,
        "missing_percent": missing_pct
    })

    return (
        result[
            result["missing_count"] > 0
        ]
        .sort_values(
            "missing_percent",
            ascending=False
        )
    )


# ------------------------------------------------------------
# 5. Train missingness
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("TRAIN MISSINGNESS")
print("=" * 90)

train_missing = missingness_table(
    train_users
)

print(
    train_missing
    if len(train_missing) > 0
    else "No missing values."
)


# ------------------------------------------------------------
# 6. Validation missingness
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("VALIDATION MISSINGNESS")
print("=" * 90)

val_missing = missingness_table(
    val_users
)

print(
    val_missing
    if len(val_missing) > 0
    else "No missing values."
)


# ------------------------------------------------------------
# 7. Test missingness
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("TEST MISSINGNESS")
print("=" * 90)

test_missing = missingness_table(
    test_users
)

print(
    test_missing
    if len(test_missing) > 0
    else "No missing values."
)


# ------------------------------------------------------------
# 8. Binary feature integrity
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("BINARY FEATURE VALUES")
print("=" * 90)

for feature in binary_features:

    train_values = sorted(
        train_users[feature]
        .dropna()
        .unique()
        .tolist()
    )

    print(
        f"{feature}: {train_values}"
    )


# ------------------------------------------------------------
# 9. Final check
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("AUDIT COMPLETE")
print("=" * 90)

print("✓ Frozen splits loaded from disk")
print("✓ Exactly 42 approved model features used")
print("✓ No excluded detector/annotation columns included")
print("✓ No preprocessing has been fitted yet")

FINAL TABULAR FEATURE SPACE
Total features: 42
Numeric features: 36
Binary features: 6

TRAIN MISSINGNESS
                        missing_count  missing_percent
followers_count                     1         0.149031
friends_count                       1         0.149031
followers_friend_ratio              1         0.149031

VALIDATION MISSINGNESS
No missing values.

TEST MISSINGNESS
                        missing_count  missing_percent
followers_count                     1         0.694444
friends_count                       1         0.694444
followers_friend_ratio              1         0.694444

BINARY FEATURE VALUES
default_profile: [0.0, 1.0]
default_profile_image: [0.0, 1.0]
has_custom_timelines: [0.0, 1.0]
possibly_sensitive: [0.0, 1.0]
hashtag_in_description: [0.0, 1.0]
numbers_in_description: [0.0, 1.0]

AUDIT COMPLETE
✓ Frozen splits loaded from disk
✓ Exactly 42 approved model features used
✓ No excluded detector/annotation columns included
✓ No preprocessing has been fitt

In [27]:
# ============================================================
# Cell 24 — Train-only Median Imputation
# ============================================================

import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer


# ------------------------------------------------------------
# 1. Copy frozen splits
# ------------------------------------------------------------

train_processed = train_users.copy()
val_processed = val_users.copy()
test_processed = test_users.copy()


# ------------------------------------------------------------
# 2. Numeric imputer
#
# IMPORTANT:
# fit ONLY on Train
# ------------------------------------------------------------

numeric_imputer = SimpleImputer(
    strategy="median"
)

numeric_imputer.fit(
    train_processed[numeric_features]
)


# ------------------------------------------------------------
# 3. Transform Train / Validation / Test
# ------------------------------------------------------------

train_processed[numeric_features] = (
    numeric_imputer.transform(
        train_processed[numeric_features]
    )
)

val_processed[numeric_features] = (
    numeric_imputer.transform(
        val_processed[numeric_features]
    )
)

test_processed[numeric_features] = (
    numeric_imputer.transform(
        test_processed[numeric_features]
    )
)


# ------------------------------------------------------------
# 4. Binary features
#
# No imputation needed because they contain no missing values.
# Keep exactly 0 / 1.
# ------------------------------------------------------------

for feature in binary_features:

    train_processed[feature] = (
        train_processed[feature]
        .astype(float)
    )

    val_processed[feature] = (
        val_processed[feature]
        .astype(float)
    )

    test_processed[feature] = (
        test_processed[feature]
        .astype(float)
    )


# ------------------------------------------------------------
# 5. Missing-value verification
# ------------------------------------------------------------

print("=" * 90)
print("MISSING VALUES AFTER TRAIN-ONLY IMPUTATION")
print("=" * 90)

print(
    "Train:",
    int(
        train_processed[
            final_features
        ].isna().sum().sum()
    )
)

print(
    "Validation:",
    int(
        val_processed[
            final_features
        ].isna().sum().sum()
    )
)

print(
    "Test:",
    int(
        test_processed[
            final_features
        ].isna().sum().sum()
    )
)


# ------------------------------------------------------------
# 6. Show learned medians ONLY for originally missing columns
# ------------------------------------------------------------

imputer_statistics = pd.Series(
    numeric_imputer.statistics_,
    index=numeric_features
)

missing_columns = sorted(
    set(train_missing.index)
    | set(val_missing.index)
    | set(test_missing.index)
)


print("\n" + "=" * 90)
print("TRAIN-DERIVED IMPUTATION VALUES")
print("=" * 90)

for feature in missing_columns:

    print(
        f"{feature:30s} -> "
        f"{imputer_statistics[feature]}"
    )


# ------------------------------------------------------------
# 7. Check for infinity
# ------------------------------------------------------------

def count_infinite_values(df):

    numeric_array = (
        df[final_features]
        .select_dtypes(include=[np.number])
        .to_numpy()
    )

    return int(
        np.isinf(numeric_array).sum()
    )


print("\n" + "=" * 90)
print("INFINITY CHECK")
print("=" * 90)

print(
    "Train:",
    count_infinite_values(
        train_processed
    )
)

print(
    "Validation:",
    count_infinite_values(
        val_processed
    )
)

print(
    "Test:",
    count_infinite_values(
        test_processed
    )
)


# ------------------------------------------------------------
# 8. Strong checks
# ------------------------------------------------------------

assert (
    train_processed[
        final_features
    ].isna().sum().sum()
    == 0
)

assert (
    val_processed[
        final_features
    ].isna().sum().sum()
    == 0
)

assert (
    test_processed[
        final_features
    ].isna().sum().sum()
    == 0
)

assert count_infinite_values(
    train_processed
) == 0

assert count_infinite_values(
    val_processed
) == 0

assert count_infinite_values(
    test_processed
) == 0


print("\n" + "=" * 90)
print("IMPUTATION COMPLETE")
print("=" * 90)

print(
    "✓ Numeric imputer fitted ONLY on Train"
)

print(
    "✓ Validation and Test were transform-only"
)

print(
    "✓ Binary features were not imputed"
)

print(
    "✓ No missing or infinite values remain"
)

MISSING VALUES AFTER TRAIN-ONLY IMPUTATION
Train: 0
Validation: 0
Test: 0

TRAIN-DERIVED IMPUTATION VALUES
followers_count                -> 6037.5
followers_friend_ratio         -> 4.201163805206788
friends_count                  -> 1575.0

INFINITY CHECK
Train: 0
Validation: 0
Test: 0

IMPUTATION COMPLETE
✓ Numeric imputer fitted ONLY on Train
✓ Validation and Test were transform-only
✓ Binary features were not imputed
✓ No missing or infinite values remain


In [28]:
# ============================================================
# Cell 25 — Numeric Feature Distribution Audit
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Use TRAIN only for distribution decisions
# ------------------------------------------------------------

train_numeric = train_processed[
    numeric_features
].copy()


# ------------------------------------------------------------
# 2. Basic descriptive statistics
# ------------------------------------------------------------

stats = pd.DataFrame({
    "min": train_numeric.min(),
    "median": train_numeric.median(),
    "mean": train_numeric.mean(),
    "max": train_numeric.max(),
    "std": train_numeric.std(),
    "skewness": train_numeric.skew()
})


# ------------------------------------------------------------
# 3. IQR-based outlier percentage
# ------------------------------------------------------------

outlier_percentages = {}

for feature in numeric_features:

    q1 = train_numeric[feature].quantile(0.25)
    q3 = train_numeric[feature].quantile(0.75)

    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    outlier_mask = (
        (train_numeric[feature] < lower)
        |
        (train_numeric[feature] > upper)
    )

    outlier_percentages[feature] = (
        outlier_mask.mean() * 100
    )


stats["outlier_percent"] = pd.Series(
    outlier_percentages
)


# ------------------------------------------------------------
# 4. Absolute skewness
# ------------------------------------------------------------

stats["abs_skewness"] = (
    stats["skewness"].abs()
)


# ------------------------------------------------------------
# 5. Sort by strongest skewness
# ------------------------------------------------------------

stats_sorted = stats.sort_values(
    "abs_skewness",
    ascending=False
)


print("=" * 100)
print("NUMERIC FEATURE DISTRIBUTION AUDIT — TRAIN ONLY")
print("=" * 100)

display(
    stats_sorted[
        [
            "min",
            "median",
            "mean",
            "max",
            "skewness",
            "outlier_percent"
        ]
    ]
)


# ------------------------------------------------------------
# 6. Strongly skewed features
#
# |skew| > 1
# ------------------------------------------------------------

strongly_skewed = stats_sorted[
    stats_sorted["abs_skewness"] > 1
].index.tolist()


print("\n" + "=" * 100)
print("STRONGLY SKEWED FEATURES |skew| > 1")
print("=" * 100)

print(
    f"Count: {len(strongly_skewed)}"
)

for feature in strongly_skewed:
    print(
        f"{feature:35s} "
        f"skew={stats.loc[feature, 'skewness']:.3f} | "
        f"outliers={stats.loc[feature, 'outlier_percent']:.2f}%"
    )


# ------------------------------------------------------------
# 7. Features containing negative values
#
# Important before considering log1p
# ------------------------------------------------------------

negative_features = [
    feature
    for feature in numeric_features
    if (train_numeric[feature] < 0).any()
]


print("\n" + "=" * 100)
print("FEATURES WITH NEGATIVE VALUES")
print("=" * 100)

print(
    f"Count: {len(negative_features)}"
)

print(
    negative_features
)


# ------------------------------------------------------------
# 8. Zero-heavy features
# ------------------------------------------------------------

zero_percent = (
    (train_numeric == 0)
    .mean()
    * 100
)

zero_heavy = (
    zero_percent[
        zero_percent >= 50
    ]
    .sort_values(
        ascending=False
    )
)


print("\n" + "=" * 100)
print("ZERO-HEAVY FEATURES (>= 50% zeros)")
print("=" * 100)

if len(zero_heavy) > 0:
    print(zero_heavy)
else:
    print("None")


# ------------------------------------------------------------
# 9. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("AUDIT COMPLETE")
print("=" * 100)

print(
    f"Numeric features: {len(numeric_features)}"
)

print(
    f"Strongly skewed: {len(strongly_skewed)}"
)

print(
    f"Features with negatives: {len(negative_features)}"
)

print(
    f"Zero-heavy features: {len(zero_heavy)}"
)

print(
    "\n✓ Distribution decisions based ONLY on Train"
)

NUMERIC FEATURE DISTRIBUTION AUDIT — TRAIN ONLY


,min,median,mean,max,skewness,outlier_percent
followers_friend_ratio,0.010526,4.201164,9.989465e+02,4.419730e+05,24.665423,16.542474
mean_no_words,1.000000,24.612900,3.559440e+01,1.956000e+03,16.047281,4.769001
mean_user_mentions_per_tweet,0.000000,0.156250,4.957909e-01,2.900000e+01,14.351409,4.172876
no_type_retweet_with_comment,0.000000,1.000000,6.670641e+00,3.970000e+02,10.026761,16.095380
listed_count,0.000000,9.000000,4.737407e+01,2.775000e+03,10.025620,13.561848
no_type_reply,0.000000,1.000000,2.078241e+01,1.304000e+03,9.558525,15.499255
followers_count,1.000000,6037.500000,2.138662e+04,9.100520e+05,9.256868,12.965723
normal_followers_count,1.000000,6033.000000,2.138266e+04,9.100520e+05,9.256670,12.965723
friends_growth_rate,0.000000,0.730337,1.891239e+00,6.623797e+01,8.724162,9.687034
friends_count,0.000000,1575.000000,2.789703e+03,7.571000e+04,8.029049,6.110283



STRONGLY SKEWED FEATURES |skew| > 1
Count: 33
followers_friend_ratio              skew=24.665 | outliers=16.54%
mean_no_words                       skew=16.047 | outliers=4.77%
mean_user_mentions_per_tweet        skew=14.351 | outliers=4.17%
no_type_retweet_with_comment        skew=10.027 | outliers=16.10%
listed_count                        skew=10.026 | outliers=13.56%
no_type_reply                       skew=9.559 | outliers=15.50%
followers_count                     skew=9.257 | outliers=12.97%
normal_followers_count              skew=9.257 | outliers=12.97%
friends_growth_rate                 skew=8.724 | outliers=9.69%
friends_count                       skew=8.029 | outliers=6.11%
no_type_tweet                       skew=7.776 | outliers=14.31%
follower_growth_rate                skew=7.494 | outliers=9.84%
retweet_as_tweet_rate               skew=7.186 | outliers=11.03%
num_digits_in_name                  skew=7.165 | outliers=3.87%
max_tweets_per_hour                 skew=6.9

In [29]:
# ============================================================
# Cell 26 — Feature Redundancy & Correlation Audit
# TRAIN ONLY
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Train numeric features only
# ------------------------------------------------------------

X_train_num = train_processed[
    numeric_features
].copy()


# ------------------------------------------------------------
# 2. Exact duplicate feature check
# ------------------------------------------------------------

exact_duplicates = []

for i, col1 in enumerate(numeric_features):

    for col2 in numeric_features[i + 1:]:

        if X_train_num[col1].equals(
            X_train_num[col2]
        ):

            exact_duplicates.append(
                (col1, col2)
            )


print("=" * 100)
print("EXACT DUPLICATE FEATURES")
print("=" * 100)

if exact_duplicates:

    for col1, col2 in exact_duplicates:
        print(
            f"{col1}  ==  {col2}"
        )

else:
    print("None")


# ------------------------------------------------------------
# 3. Near-duplicate check
#
# Fraction of rows with equal values
# ------------------------------------------------------------

near_duplicates = []

for i, col1 in enumerate(numeric_features):

    for col2 in numeric_features[i + 1:]:

        equal_ratio = np.isclose(
            X_train_num[col1].to_numpy(),
            X_train_num[col2].to_numpy(),
            rtol=1e-8,
            atol=1e-10
        ).mean()

        if (
            equal_ratio >= 0.95
            and equal_ratio < 1.0
        ):

            near_duplicates.append({
                "feature_1": col1,
                "feature_2": col2,
                "equal_percent": equal_ratio * 100
            })


near_duplicates_df = pd.DataFrame(
    near_duplicates
)

print("\n" + "=" * 100)
print("NEAR-DUPLICATE FEATURES (>=95% equal rows)")
print("=" * 100)

if len(near_duplicates_df) > 0:

    display(
        near_duplicates_df.sort_values(
            "equal_percent",
            ascending=False
        )
    )

else:
    print("None")


# ------------------------------------------------------------
# 4. Pearson correlation matrix
# ------------------------------------------------------------

corr_matrix = (
    X_train_num
    .corr(method="pearson")
    .abs()
)


high_corr_pairs = []

for i, col1 in enumerate(numeric_features):

    for col2 in numeric_features[i + 1:]:

        corr = corr_matrix.loc[
            col1,
            col2
        ]

        if pd.notna(corr) and corr >= 0.95:

            high_corr_pairs.append({
                "feature_1": col1,
                "feature_2": col2,
                "pearson_abs_corr": corr
            })


high_corr_df = pd.DataFrame(
    high_corr_pairs
)


print("\n" + "=" * 100)
print("HIGHLY CORRELATED FEATURES |Pearson r| >= 0.95")
print("=" * 100)

if len(high_corr_df) > 0:

    display(
        high_corr_df.sort_values(
            "pearson_abs_corr",
            ascending=False
        )
    )

else:
    print("None")


# ------------------------------------------------------------
# 5. Spearman correlation
#
# Useful because many features are strongly skewed
# ------------------------------------------------------------

spearman_matrix = (
    X_train_num
    .corr(method="spearman")
    .abs()
)


high_spearman_pairs = []

for i, col1 in enumerate(numeric_features):

    for col2 in numeric_features[i + 1:]:

        corr = spearman_matrix.loc[
            col1,
            col2
        ]

        if pd.notna(corr) and corr >= 0.95:

            high_spearman_pairs.append({
                "feature_1": col1,
                "feature_2": col2,
                "spearman_abs_corr": corr
            })


high_spearman_df = pd.DataFrame(
    high_spearman_pairs
)


print("\n" + "=" * 100)
print("HIGHLY CORRELATED FEATURES |Spearman r| >= 0.95")
print("=" * 100)

if len(high_spearman_df) > 0:

    display(
        high_spearman_df.sort_values(
            "spearman_abs_corr",
            ascending=False
        )
    )

else:
    print("None")


# ------------------------------------------------------------
# 6. Specifically inspect suspicious pairs
# ------------------------------------------------------------

pairs_to_check = [

    (
        "status_count",
        "statuses_count"
    ),

    (
        "followers_count",
        "normal_followers_count"
    ),

    (
        "status_count",
        "no_retweet_tweets"
    )
]


print("\n" + "=" * 100)
print("SPECIFIC SUSPICIOUS PAIRS")
print("=" * 100)

for col1, col2 in pairs_to_check:

    if (
        col1 in X_train_num.columns
        and col2 in X_train_num.columns
    ):

        equal_pct = (
            np.isclose(
                X_train_num[col1],
                X_train_num[col2],
                rtol=1e-8,
                atol=1e-10
            ).mean()
            * 100
        )

        pearson = (
            X_train_num[
                [col1, col2]
            ]
            .corr(method="pearson")
            .iloc[0, 1]
        )

        spearman = (
            X_train_num[
                [col1, col2]
            ]
            .corr(method="spearman")
            .iloc[0, 1]
        )

        print(
            f"\n{col1}  vs  {col2}"
        )

        print(
            f"Equal rows : {equal_pct:.3f}%"
        )

        print(
            f"Pearson    : {pearson:.6f}"
        )

        print(
            f"Spearman   : {spearman:.6f}"
        )


# ------------------------------------------------------------
# 7. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("REDUNDANCY AUDIT COMPLETE")
print("=" * 100)

print(
    "Exact duplicate pairs:",
    len(exact_duplicates)
)

print(
    "Near-duplicate pairs:",
    len(near_duplicates_df)
)

print(
    "Pearson >= 0.95 pairs:",
    len(high_corr_df)
)

print(
    "Spearman >= 0.95 pairs:",
    len(high_spearman_df)
)

print(
    "\n✓ Audit based ONLY on Train"
)

EXACT DUPLICATE FEATURES
statuses_count  ==  status_count

NEAR-DUPLICATE FEATURES (>=95% equal rows)


,feature_1,feature_2,equal_percent
0,followers_count,normal_followers_count,99.850969
1,min_tweets_per_hour,min_tweets_per_day,97.466468



HIGHLY CORRELATED FEATURES |Pearson r| >= 0.95


,feature_1,feature_2,pearson_abs_corr
1,statuses_count,status_count,1.000000
0,followers_count,normal_followers_count,0.999999
2,statuses_count,no_retweet_tweets,0.999996
3,status_count,no_retweet_tweets,0.999996



HIGHLY CORRELATED FEATURES |Spearman r| >= 0.95


,feature_1,feature_2,spearman_abs_corr
1,statuses_count,status_count,1.000000
2,statuses_count,no_retweet_tweets,0.999957
3,status_count,no_retweet_tweets,0.999957
0,followers_count,normal_followers_count,0.998739



SPECIFIC SUSPICIOUS PAIRS

status_count  vs  statuses_count
Equal rows : 100.000%
Pearson    : 1.000000
Spearman   : 1.000000

followers_count  vs  normal_followers_count
Equal rows : 99.851%
Pearson    : 0.999999
Spearman   : 0.998739

status_count  vs  no_retweet_tweets
Equal rows : 0.000%
Pearson    : 0.999996
Spearman   : 0.999957

REDUNDANCY AUDIT COMPLETE
Exact duplicate pairs: 1
Near-duplicate pairs: 2
Pearson >= 0.95 pairs: 4
Spearman >= 0.95 pairs: 4

✓ Audit based ONLY on Train


In [30]:
# ============================================================
# Cell 27 — Final Redundancy Pruning
# ============================================================

import json


# ------------------------------------------------------------
# 1. Features removed because they are redundant
# ------------------------------------------------------------

redundant_features_removed = [
    "status_count",
    "normal_followers_count"
]


print("=" * 90)
print("REDUNDANT FEATURES TO REMOVE")
print("=" * 90)

for feature in redundant_features_removed:
    print("REMOVE:", feature)


# ------------------------------------------------------------
# 2. Build final tabular feature list
# ------------------------------------------------------------

final_model_features = [
    feature
    for feature in final_features
    if feature not in redundant_features_removed
]


final_numeric_features = [
    feature
    for feature in numeric_features
    if feature not in redundant_features_removed
]


final_binary_features = binary_features.copy()


# ------------------------------------------------------------
# 3. Checks
# ------------------------------------------------------------

assert len(final_model_features) == 40
assert len(final_numeric_features) == 34
assert len(final_binary_features) == 6

assert (
    set(final_model_features)
    ==
    set(final_numeric_features)
    | set(final_binary_features)
)


# ------------------------------------------------------------
# 4. Create model matrices
# ------------------------------------------------------------

X_train_tabular = train_processed[
    final_model_features
].copy()

X_val_tabular = val_processed[
    final_model_features
].copy()

X_test_tabular = test_processed[
    final_model_features
].copy()


y_train = (
    train_processed["label_binary"]
    .astype(int)
    .copy()
)

y_val = (
    val_processed["label_binary"]
    .astype(int)
    .copy()
)

y_test = (
    test_processed["label_binary"]
    .astype(int)
    .copy()
)


# ------------------------------------------------------------
# 5. Strong integrity checks
# ------------------------------------------------------------

assert X_train_tabular.shape == (671, 40)
assert X_val_tabular.shape == (144, 40)
assert X_test_tabular.shape == (144, 40)

assert X_train_tabular.isna().sum().sum() == 0
assert X_val_tabular.isna().sum().sum() == 0
assert X_test_tabular.isna().sum().sum() == 0


# ------------------------------------------------------------
# 6. Report
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("FINAL TABULAR FEATURE SPACE")
print("=" * 90)

print(
    "Original approved features:",
    len(final_features)
)

print(
    "Removed redundant features:",
    len(redundant_features_removed)
)

print(
    "Final model features:",
    len(final_model_features)
)

print(
    "Numeric:",
    len(final_numeric_features)
)

print(
    "Binary:",
    len(final_binary_features)
)


print("\nTrain shape:", X_train_tabular.shape)
print("Validation shape:", X_val_tabular.shape)
print("Test shape:", X_test_tabular.shape)


# ------------------------------------------------------------
# 7. Save preprocessing policy
# ------------------------------------------------------------

tabular_preprocessing_policy = {

    "input_feature_count": 42,

    "redundancy_analysis": {
        "performed_on": "train_only",

        "removed_features": {
            "status_count": (
                "Exact duplicate of statuses_count "
                "on the training split."
            ),

            "normal_followers_count": (
                "Near-duplicate of followers_count; "
                "99.85% identical rows and "
                "Pearson correlation approximately 1."
            )
        }
    },

    "final_feature_count": 40,

    "numeric_feature_count": 34,

    "binary_feature_count": 6,

    "final_model_features": final_model_features,

    "numeric_features": final_numeric_features,

    "binary_features": final_binary_features,

    "retained_high_correlation_features": {
        "no_retweet_tweets": (
            "Retained despite very high correlation "
            "with statuses_count because it represents "
            "a semantically distinct behavioral quantity."
        ),

        "min_tweets_per_hour_and_day": (
            "Both retained because they represent "
            "different temporal granularities."
        )
    }
}


POLICY_OUTPUT_PATH = (
    FINAL_DATA_DIR
    / "config"
    / "tabular_preprocessing_policy.json"
)


with open(
    POLICY_OUTPUT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        tabular_preprocessing_policy,
        f,
        ensure_ascii=False,
        indent=2
    )


print(
    "\nSaved preprocessing policy:"
)

print(
    POLICY_OUTPUT_PATH
)


print("\n✓ Feature selection based ONLY on Train")
print("✓ No Validation/Test statistics used")
print("✓ Final tabular dimension = 40")

REDUNDANT FEATURES TO REMOVE
REMOVE: status_count
REMOVE: normal_followers_count

FINAL TABULAR FEATURE SPACE
Original approved features: 42
Removed redundant features: 2
Final model features: 40
Numeric: 34
Binary: 6

Train shape: (671, 40)
Validation shape: (144, 40)
Test shape: (144, 40)

Saved preprocessing policy:
C:\Users\p.vazifeh\Desktop\University\Project\Bot Detection Implementation\final_data\config\tabular_preprocessing_policy.json

✓ Feature selection based ONLY on Train
✓ No Validation/Test statistics used
✓ Final tabular dimension = 40


In [31]:
# ============================================================
# Cell 28 — Final Transformation Policy Audit
# TRAIN ONLY
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Detect numeric features that are actually binary-like
# ------------------------------------------------------------

binary_like_numeric = []

for feature in final_numeric_features:

    unique_values = set(
        train_processed[feature]
        .dropna()
        .unique()
        .tolist()
    )

    if unique_values.issubset({0, 1, 0.0, 1.0}):
        binary_like_numeric.append(feature)


print("=" * 95)
print("BINARY-LIKE FEATURES INSIDE NUMERIC GROUP")
print("=" * 95)

print(binary_like_numeric)


# ------------------------------------------------------------
# 2. Continuous numeric features
# ------------------------------------------------------------

continuous_numeric_features = [
    feature
    for feature in final_numeric_features
    if feature not in binary_like_numeric
]


# ------------------------------------------------------------
# 3. Effective binary features
#
# Original 6 binary features
# +
# any 0/1 feature detected in numeric group
# ------------------------------------------------------------

effective_binary_features = (
    final_binary_features
    + binary_like_numeric
)


# ------------------------------------------------------------
# 4. Calculate skewness using TRAIN only
# ------------------------------------------------------------

train_skewness = (
    train_processed[
        continuous_numeric_features
    ]
    .skew()
)


# ------------------------------------------------------------
# 5. Log1p candidates
#
# Rule:
# abs(skewness) > 1
#
# All features are non-negative based on previous audit.
# ------------------------------------------------------------

log1p_features = (
    train_skewness[
        train_skewness.abs() > 1
    ]
    .index
    .tolist()
)


non_log_continuous_features = [
    feature
    for feature in continuous_numeric_features
    if feature not in log1p_features
]


# ------------------------------------------------------------
# 6. Safety check for log1p
# ------------------------------------------------------------

negative_log_features = []

for feature in log1p_features:

    if (
        train_processed[feature] < 0
    ).any():

        negative_log_features.append(feature)


assert len(negative_log_features) == 0


# ------------------------------------------------------------
# 7. Report
# ------------------------------------------------------------

print("\n" + "=" * 95)
print("FINAL TRANSFORMATION GROUPS")
print("=" * 95)

print(
    "Total final model features:",
    len(final_model_features)
)

print(
    "Continuous numeric features:",
    len(continuous_numeric_features)
)

print(
    "Effective binary features:",
    len(effective_binary_features)
)

print(
    "Log1p features:",
    len(log1p_features)
)

print(
    "Continuous features without Log1p:",
    len(non_log_continuous_features)
)


print("\n" + "=" * 95)
print("LOG1P FEATURES")
print("=" * 95)

for feature in log1p_features:

    print(
        f"{feature:35s} "
        f"skew={train_skewness[feature]:.3f}"
    )


print("\n" + "=" * 95)
print("CONTINUOUS FEATURES WITHOUT LOG1P")
print("=" * 95)

for feature in non_log_continuous_features:

    print(
        f"{feature:35s} "
        f"skew={train_skewness[feature]:.3f}"
    )


print("\n" + "=" * 95)
print("EFFECTIVE BINARY FEATURES")
print("=" * 95)

for feature in effective_binary_features:
    print(feature)


# ------------------------------------------------------------
# 8. Feature-space integrity
# ------------------------------------------------------------

assert (
    set(continuous_numeric_features)
    |
    set(effective_binary_features)
) == set(final_model_features)

assert (
    set(continuous_numeric_features)
    &
    set(effective_binary_features)
) == set()


print("\n" + "=" * 95)
print("TRANSFORMATION POLICY AUDIT COMPLETE")
print("=" * 95)

print("✓ Feature groups are mutually exclusive")
print("✓ All 40 final features are accounted for")
print("✓ Log1p candidates selected using TRAIN only")
print("✓ No Validation/Test statistics were used")

BINARY-LIKE FEATURES INSIDE NUMERIC GROUP
['url_in_description']

FINAL TRANSFORMATION GROUPS
Total final model features: 40
Continuous numeric features: 33
Effective binary features: 7
Log1p features: 30
Continuous features without Log1p: 3

LOG1P FEATURES
followers_count                     skew=9.257
friends_count                       skew=8.029
favourites_count                    skew=3.023
listed_count                        skew=10.026
media_count                         skew=6.095
statuses_count                      skew=5.967
follower_growth_rate                skew=7.494
friends_growth_rate                 skew=8.724
no_type_tweet                       skew=7.776
no_type_retweet_with_comment        skew=10.027
no_type_reply                       skew=9.559
mean_no_words                       skew=16.047
no_languages                        skew=2.134
mean_no_hashtags                    skew=1.559
mean_favourites_per_tweet           skew=4.371
time_between_tweets               

In [32]:
# ============================================================
# Cell 29 — Apply Log1p Transformation
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Create clean copies
# ------------------------------------------------------------

train_transformed = train_processed.copy()
val_transformed = val_processed.copy()
test_transformed = test_processed.copy()


# ------------------------------------------------------------
# 2. Safety check
#
# log1p requires values >= 0
# ------------------------------------------------------------

for feature in log1p_features:

    assert (
        train_transformed[feature] >= 0
    ).all()

    assert (
        val_transformed[feature] >= 0
    ).all()

    assert (
        test_transformed[feature] >= 0
    ).all()


# ------------------------------------------------------------
# 3. Apply SAME log1p transformation
# to Train / Validation / Test
# ------------------------------------------------------------

for feature in log1p_features:

    train_transformed[feature] = np.log1p(
        train_transformed[feature]
    )

    val_transformed[feature] = np.log1p(
        val_transformed[feature]
    )

    test_transformed[feature] = np.log1p(
        test_transformed[feature]
    )


# ------------------------------------------------------------
# 4. Compare Train skewness before vs after
# ------------------------------------------------------------

skew_before = (
    train_processed[
        log1p_features
    ]
    .skew()
)

skew_after = (
    train_transformed[
        log1p_features
    ]
    .skew()
)


skew_comparison = pd.DataFrame({

    "skew_before": skew_before,

    "skew_after": skew_after,

    "abs_before": skew_before.abs(),

    "abs_after": skew_after.abs()

})


skew_comparison["improvement"] = (
    skew_comparison["abs_before"]
    -
    skew_comparison["abs_after"]
)


skew_comparison = (
    skew_comparison
    .sort_values(
        "abs_after",
        ascending=False
    )
)


print("=" * 100)
print("LOG1P SKEWNESS COMPARISON — TRAIN ONLY")
print("=" * 100)

display(
    skew_comparison[
        [
            "skew_before",
            "skew_after",
            "improvement"
        ]
    ]
)


# ------------------------------------------------------------
# 5. Features still strongly skewed
#
# |skew| > 1 after log1p
# ------------------------------------------------------------

still_skewed = (
    skew_comparison[
        skew_comparison["abs_after"] > 1
    ]
    .index
    .tolist()
)


print("\n" + "=" * 100)
print("STILL STRONGLY SKEWED AFTER LOG1P |skew| > 1")
print("=" * 100)

print(
    "Count:",
    len(still_skewed)
)

for feature in still_skewed:

    print(
        f"{feature:35s} "
        f"before={skew_before[feature]:8.3f} | "
        f"after={skew_after[feature]:8.3f}"
    )


# ------------------------------------------------------------
# 6. Check features where log1p made skewness worse
# ------------------------------------------------------------

worse_after_log = (
    skew_comparison[
        skew_comparison["abs_after"]
        >
        skew_comparison["abs_before"]
    ]
    .index
    .tolist()
)


print("\n" + "=" * 100)
print("FEATURES WORSE AFTER LOG1P")
print("=" * 100)

print(
    "Count:",
    len(worse_after_log)
)

for feature in worse_after_log:

    print(
        f"{feature:35s} "
        f"before={skew_before[feature]:.3f} | "
        f"after={skew_after[feature]:.3f}"
    )


# ------------------------------------------------------------
# 7. Check NaN / infinity after transformation
# ------------------------------------------------------------

def transformation_integrity(df):

    values = (
        df[final_model_features]
        .to_numpy(dtype=float)
    )

    return {
        "missing": int(
            np.isnan(values).sum()
        ),

        "infinite": int(
            np.isinf(values).sum()
        )
    }


train_integrity = transformation_integrity(
    train_transformed
)

val_integrity = transformation_integrity(
    val_transformed
)

test_integrity = transformation_integrity(
    test_transformed
)


print("\n" + "=" * 100)
print("POST-LOG1P INTEGRITY")
print("=" * 100)

print(
    "Train:",
    train_integrity
)

print(
    "Validation:",
    val_integrity
)

print(
    "Test:",
    test_integrity
)


# ------------------------------------------------------------
# 8. Binary features must remain unchanged
# ------------------------------------------------------------

for feature in effective_binary_features:

    assert np.array_equal(
        train_processed[feature].to_numpy(),
        train_transformed[feature].to_numpy()
    )

    assert np.array_equal(
        val_processed[feature].to_numpy(),
        val_transformed[feature].to_numpy()
    )

    assert np.array_equal(
        test_processed[feature].to_numpy(),
        test_transformed[feature].to_numpy()
    )


# ------------------------------------------------------------
# 9. Strong checks
# ------------------------------------------------------------

assert train_integrity["missing"] == 0
assert train_integrity["infinite"] == 0

assert val_integrity["missing"] == 0
assert val_integrity["infinite"] == 0

assert test_integrity["missing"] == 0
assert test_integrity["infinite"] == 0


print("\n" + "=" * 100)
print("LOG1P TRANSFORMATION COMPLETE")
print("=" * 100)

print(
    f"✓ Log1p applied to {len(log1p_features)} features"
)

print(
    "✓ Same transformation applied to Train / Validation / Test"
)

print(
    "✓ 7 effective binary features were untouched"
)

print(
    "✓ No missing or infinite values introduced"
)

LOG1P SKEWNESS COMPARISON — TRAIN ONLY


,skew_before,skew_after,improvement
num_digits_in_name,7.165480,5.898184,1.267296
favourites_count,3.023295,-2.446143,0.577152
mean_user_mentions_per_tweet,14.351409,2.230331,12.121078
max_tweets_per_hour,6.981579,1.937332,5.044247
followers_friend_ratio,24.665423,1.767747,22.897676
min_tweets_per_hour,2.147048,1.574506,0.572542
max_occurence_of_same_gap,6.251413,1.520494,4.730919
min_tweets_per_day,2.125454,1.467204,0.658250
max_tweets_per_day,5.406361,1.429141,3.977220
no_type_retweet_with_comment,10.026761,1.429065,8.597696



STILL STRONGLY SKEWED AFTER LOG1P |skew| > 1
Count: 20
num_digits_in_name                  before=   7.165 | after=   5.898
favourites_count                    before=   3.023 | after=  -2.446
mean_user_mentions_per_tweet        before=  14.351 | after=   2.230
max_tweets_per_hour                 before=   6.982 | after=   1.937
followers_friend_ratio              before=  24.665 | after=   1.768
min_tweets_per_hour                 before=   2.147 | after=   1.575
max_occurence_of_same_gap           before=   6.251 | after=   1.520
min_tweets_per_day                  before=   2.125 | after=   1.467
max_tweets_per_day                  before=   5.406 | after=   1.429
no_type_retweet_with_comment        before=  10.027 | after=   1.429
friends_count                       before=   8.029 | after=  -1.279
no_type_reply                       before=   9.559 | after=   1.239
friends_growth_rate                 before=   8.724 | after=   1.201
unique_mention_rate_per_tweet       before=   1

In [33]:
# ============================================================
# Cell 30 — Train-only Robust Scaling
# ============================================================

import numpy as np
import pandas as pd

from sklearn.preprocessing import RobustScaler


# ------------------------------------------------------------
# 1. Copies after log1p transformation
# ------------------------------------------------------------

train_scaled = train_transformed.copy()
val_scaled = val_transformed.copy()
test_scaled = test_transformed.copy()


# ------------------------------------------------------------
# 2. Fit scaler ONLY on Train
#
# Scale only continuous numeric features.
# Binary features remain untouched.
# ------------------------------------------------------------

robust_scaler = RobustScaler(
    with_centering=True,
    with_scaling=True,
    quantile_range=(25.0, 75.0)
)


robust_scaler.fit(
    train_scaled[
        continuous_numeric_features
    ]
)


# ------------------------------------------------------------
# 3. Transform Train / Validation / Test
# ------------------------------------------------------------

train_scaled[
    continuous_numeric_features
] = robust_scaler.transform(
    train_scaled[
        continuous_numeric_features
    ]
)


val_scaled[
    continuous_numeric_features
] = robust_scaler.transform(
    val_scaled[
        continuous_numeric_features
    ]
)


test_scaled[
    continuous_numeric_features
] = robust_scaler.transform(
    test_scaled[
        continuous_numeric_features
    ]
)


# ------------------------------------------------------------
# 4. Binary features must remain unchanged
# ------------------------------------------------------------

for feature in effective_binary_features:

    assert np.array_equal(
        train_scaled[feature].to_numpy(),
        train_transformed[feature].to_numpy()
    )

    assert np.array_equal(
        val_scaled[feature].to_numpy(),
        val_transformed[feature].to_numpy()
    )

    assert np.array_equal(
        test_scaled[feature].to_numpy(),
        test_transformed[feature].to_numpy()
    )


# ------------------------------------------------------------
# 5. Build final model matrices
# ------------------------------------------------------------

X_train_final = (
    train_scaled[
        final_model_features
    ]
    .copy()
)

X_val_final = (
    val_scaled[
        final_model_features
    ]
    .copy()
)

X_test_final = (
    test_scaled[
        final_model_features
    ]
    .copy()
)


y_train_final = (
    train_scaled["label_binary"]
    .astype(int)
    .copy()
)

y_val_final = (
    val_scaled["label_binary"]
    .astype(int)
    .copy()
)

y_test_final = (
    test_scaled["label_binary"]
    .astype(int)
    .copy()
)


# ------------------------------------------------------------
# 6. Integrity check
# ------------------------------------------------------------

def matrix_integrity(df):

    arr = df.to_numpy(
        dtype=float
    )

    return {
        "shape": arr.shape,
        "missing": int(
            np.isnan(arr).sum()
        ),
        "infinite": int(
            np.isinf(arr).sum()
        )
    }


print("=" * 95)
print("FINAL TABULAR MATRICES")
print("=" * 95)

print(
    "Train:",
    matrix_integrity(
        X_train_final
    )
)

print(
    "Validation:",
    matrix_integrity(
        X_val_final
    )
)

print(
    "Test:",
    matrix_integrity(
        X_test_final
    )
)


# ------------------------------------------------------------
# 7. Check Train scaling statistics
#
# RobustScaler should produce approximately:
# median ~ 0
# IQR ~ 1
# ------------------------------------------------------------

train_scaled_stats = pd.DataFrame({
    "median": (
        X_train_final[
            continuous_numeric_features
        ].median()
    ),

    "q1": (
        X_train_final[
            continuous_numeric_features
        ].quantile(0.25)
    ),

    "q3": (
        X_train_final[
            continuous_numeric_features
        ].quantile(0.75)
    )
})


train_scaled_stats["iqr"] = (
    train_scaled_stats["q3"]
    -
    train_scaled_stats["q1"]
)


print("\n" + "=" * 95)
print("TRAIN ROBUST-SCALING CHECK")
print("=" * 95)

display(
    train_scaled_stats
)


# ------------------------------------------------------------
# 8. Binary integrity
# ------------------------------------------------------------

print("\n" + "=" * 95)
print("FINAL BINARY FEATURE VALUES")
print("=" * 95)

for feature in effective_binary_features:

    values = sorted(
        X_train_final[feature]
        .unique()
        .tolist()
    )

    print(
        f"{feature:30s} -> {values}"
    )


# ------------------------------------------------------------
# 9. Strong checks
# ------------------------------------------------------------

assert X_train_final.shape == (671, 40)
assert X_val_final.shape == (144, 40)
assert X_test_final.shape == (144, 40)

assert (
    X_train_final
    .isna()
    .sum()
    .sum()
    == 0
)

assert (
    X_val_final
    .isna()
    .sum()
    .sum()
    == 0
)

assert (
    X_test_final
    .isna()
    .sum()
    .sum()
    == 0
)


assert not np.isinf(
    X_train_final.to_numpy(
        dtype=float
    )
).any()

assert not np.isinf(
    X_val_final.to_numpy(
        dtype=float
    )
).any()

assert not np.isinf(
    X_test_final.to_numpy(
        dtype=float
    )
).any()


# ------------------------------------------------------------
# 10. Final report
# ------------------------------------------------------------

print("\n" + "=" * 95)
print("ROBUST SCALING COMPLETE")
print("=" * 95)

print(
    f"Continuous features scaled: "
    f"{len(continuous_numeric_features)}"
)

print(
    f"Binary features untouched: "
    f"{len(effective_binary_features)}"
)

print(
    f"Final tabular dimension: "
    f"{len(final_model_features)}"
)

print(
    "\n✓ RobustScaler fitted ONLY on Train"
)

print(
    "✓ Validation/Test were transform-only"
)

print(
    "✓ Binary features remain 0/1"
)

print(
    "✓ No missing or infinite values remain"
)

FINAL TABULAR MATRICES
Train: {'shape': (671, 40), 'missing': 0, 'infinite': 0}
Validation: {'shape': (144, 40), 'missing': 0, 'infinite': 0}
Test: {'shape': (144, 40), 'missing': 0, 'infinite': 0}

TRAIN ROBUST-SCALING CHECK


,median,q1,q3,iqr
followers_count,0.0,-0.341759,0.658241,1.0
friends_count,0.0,-0.612941,0.387059,1.0
favourites_count,0.0,-0.517242,0.482758,1.0
listed_count,0.0,-0.386853,0.613147,1.0
media_count,0.0,-0.466643,0.533357,1.0
statuses_count,0.0,-0.519392,0.480608,1.0
user_age,0.0,-0.397380,0.602620,1.0
follower_growth_rate,0.0,-0.498625,0.501375,1.0
friends_growth_rate,0.0,-0.390868,0.609132,1.0
no_type_tweet,0.0,-0.359988,0.640012,1.0



FINAL BINARY FEATURE VALUES
default_profile                -> [0.0, 1.0]
default_profile_image          -> [0.0, 1.0]
has_custom_timelines           -> [0.0, 1.0]
possibly_sensitive             -> [0.0, 1.0]
hashtag_in_description         -> [0.0, 1.0]
numbers_in_description         -> [0.0, 1.0]
url_in_description             -> [0.0, 1.0]

ROBUST SCALING COMPLETE
Continuous features scaled: 33
Binary features untouched: 7
Final tabular dimension: 40

✓ RobustScaler fitted ONLY on Train
✓ Validation/Test were transform-only
✓ Binary features remain 0/1
✓ No missing or infinite values remain


In [34]:
# ============================================================
# Cell 31 — Freeze and Save Final Tabular Preprocessing
# ============================================================

from pathlib import Path
import json
import joblib
import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 1. Output directories
# ------------------------------------------------------------

TABULAR_SPLIT_DIR = (
    FINAL_DATA_DIR
    / "splits"
    / "tabular"
)

PREPROCESSOR_DIR = (
    PROJECT_ROOT
    / "models"
    / "preprocessing"
)

TABULAR_SPLIT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

PREPROCESSOR_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# 2. Build final tabular files
#
# Keep user_key + screen_name + label
# so we can later align them with Tweets and Graph.
# ------------------------------------------------------------

def build_final_tabular_file(
    original_user_df,
    X_final
):

    output = pd.DataFrame({
        "screen_name":
            original_user_df["screen_name"].values,

        "user_key":
            original_user_df["user_key"].values,

        "label_binary":
            original_user_df["label_binary"]
            .astype(int)
            .values
    })

    # Add the 40 final model features
    for feature in final_model_features:

        output[feature] = (
            X_final[feature]
            .to_numpy()
        )

    return output


train_tabular_final = build_final_tabular_file(
    train_users,
    X_train_final
)

val_tabular_final = build_final_tabular_file(
    val_users,
    X_val_final
)

test_tabular_final = build_final_tabular_file(
    test_users,
    X_test_final
)


# ------------------------------------------------------------
# 3. Strong alignment checks
# ------------------------------------------------------------

assert (
    train_tabular_final["user_key"].tolist()
    ==
    train_users["user_key"].tolist()
)

assert (
    val_tabular_final["user_key"].tolist()
    ==
    val_users["user_key"].tolist()
)

assert (
    test_tabular_final["user_key"].tolist()
    ==
    test_users["user_key"].tolist()
)


assert train_tabular_final.shape == (671, 43)
assert val_tabular_final.shape == (144, 43)
assert test_tabular_final.shape == (144, 43)


# 3 metadata columns:
# screen_name, user_key, label_binary
# +
# 40 model features
assert (
    len(train_tabular_final.columns)
    == 43
)


# ------------------------------------------------------------
# 4. Save final preprocessed tabular splits
# ------------------------------------------------------------

train_tabular_final.to_csv(
    TABULAR_SPLIT_DIR
    / "train_tabular.csv",
    index=False
)

val_tabular_final.to_csv(
    TABULAR_SPLIT_DIR
    / "validation_tabular.csv",
    index=False
)

test_tabular_final.to_csv(
    TABULAR_SPLIT_DIR
    / "test_tabular.csv",
    index=False
)


# ------------------------------------------------------------
# 5. Save fitted preprocessing objects
#
# IMPORTANT:
# These objects were fitted ONLY on Train.
# ------------------------------------------------------------

joblib.dump(
    numeric_imputer,
    PREPROCESSOR_DIR
    / "numeric_median_imputer.joblib"
)

joblib.dump(
    robust_scaler,
    PREPROCESSOR_DIR
    / "robust_scaler.joblib"
)


# ------------------------------------------------------------
# 6. Complete transformation policy
# ------------------------------------------------------------

final_preprocessing_policy = {

    "version": "tabular_preprocessing_v1",

    "fit_scope": "train_only",

    "input": {

        "initial_tabular_features": 42,

        "numeric_features_before_pruning":
            numeric_features,

        "binary_features_before_pruning":
            binary_features
    },

    "imputation": {

        "method": "median",

        "fit_on": "train_only",

        "input_features":
            numeric_features,

        "learned_missing_value_examples": {
            "followers_count":
                float(
                    imputer_statistics[
                        "followers_count"
                    ]
                ),

            "friends_count":
                float(
                    imputer_statistics[
                        "friends_count"
                    ]
                ),

            "followers_friend_ratio":
                float(
                    imputer_statistics[
                        "followers_friend_ratio"
                    ]
                )
        }
    },

    "redundancy_pruning": {

        "removed_features": [
            "status_count",
            "normal_followers_count"
        ],

        "feature_count_after_pruning": 40
    },

    "transformation": {

        "log1p_features":
            log1p_features,

        "n_log1p_features":
            len(log1p_features),

        "continuous_without_log1p":
            non_log_continuous_features,

        "continuous_features":
            continuous_numeric_features,

        "effective_binary_features":
            effective_binary_features
    },

    "scaling": {

        "method": "RobustScaler",

        "fit_on": "train_only",

        "quantile_range": [
            25.0,
            75.0
        ],

        "scaled_features":
            continuous_numeric_features,

        "binary_features_scaled": False
    },

    "final_model_input": {

        "feature_count":
            len(final_model_features),

        "feature_order":
            final_model_features,

        "continuous_feature_count":
            len(continuous_numeric_features),

        "binary_feature_count":
            len(effective_binary_features)
    }
}


FINAL_POLICY_PATH = (
    FINAL_DATA_DIR
    / "config"
    / "tabular_preprocessing_policy.json"
)


with open(
    FINAL_POLICY_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        final_preprocessing_policy,
        f,
        ensure_ascii=False,
        indent=2
    )


# ------------------------------------------------------------
# 7. Reload files to verify actual saved output
# ------------------------------------------------------------

check_train = pd.read_csv(
    TABULAR_SPLIT_DIR
    / "train_tabular.csv"
)

check_val = pd.read_csv(
    TABULAR_SPLIT_DIR
    / "validation_tabular.csv"
)

check_test = pd.read_csv(
    TABULAR_SPLIT_DIR
    / "test_tabular.csv"
)


# ------------------------------------------------------------
# 8. Final integrity
# ------------------------------------------------------------

assert check_train.shape == (671, 43)
assert check_val.shape == (144, 43)
assert check_test.shape == (144, 43)


feature_columns_saved = [
    c for c in check_train.columns
    if c not in [
        "screen_name",
        "user_key",
        "label_binary"
    ]
]

assert (
    feature_columns_saved
    == final_model_features
)


for df in [
    check_train,
    check_val,
    check_test
]:

    values = (
        df[final_model_features]
        .to_numpy(dtype=float)
    )

    assert not np.isnan(values).any()
    assert not np.isinf(values).any()


# ------------------------------------------------------------
# 9. Final report
# ------------------------------------------------------------

print("=" * 95)
print("FINAL TABULAR PREPROCESSING FROZEN")
print("=" * 95)

print("\nSaved tabular datasets:")

print(
    "Train      :",
    check_train.shape
)

print(
    "Validation :",
    check_val.shape
)

print(
    "Test       :",
    check_test.shape
)


print("\nFinal model input:")
print(
    f"{len(final_model_features)} features"
)

print(
    f"{len(continuous_numeric_features)} continuous"
)

print(
    f"{len(effective_binary_features)} binary"
)


print("\nSaved preprocessing artifacts:")

print(
    PREPROCESSOR_DIR
    / "numeric_median_imputer.joblib"
)

print(
    PREPROCESSOR_DIR
    / "robust_scaler.joblib"
)


print("\nSaved policy:")
print(
    FINAL_POLICY_PATH
)


print("\n✓ Median imputer frozen")
print("✓ Redundancy pruning frozen")
print("✓ Log1p policy frozen")
print("✓ RobustScaler frozen")
print("✓ Final feature order frozen")
print("✓ Train / Validation / Test tabular matrices saved")

FINAL TABULAR PREPROCESSING FROZEN

Saved tabular datasets:
Train      : (671, 43)
Validation : (144, 43)
Test       : (144, 43)

Final model input:
40 features
33 continuous
7 binary

Saved preprocessing artifacts:
C:\Users\p.vazifeh\Desktop\University\Project\Bot Detection Implementation\models\preprocessing\numeric_median_imputer.joblib
C:\Users\p.vazifeh\Desktop\University\Project\Bot Detection Implementation\models\preprocessing\robust_scaler.joblib

Saved policy:
C:\Users\p.vazifeh\Desktop\University\Project\Bot Detection Implementation\final_data\config\tabular_preprocessing_policy.json

✓ Median imputer frozen
✓ Redundancy pruning frozen
✓ Log1p policy frozen
✓ RobustScaler frozen
✓ Final feature order frozen
✓ Train / Validation / Test tabular matrices saved


In [35]:
# ============================================================
# Cell 32 — Tweet/Text Quality and Leakage Audit
# ============================================================

import pandas as pd
import numpy as np
import re


# ------------------------------------------------------------
# 1. Load FROZEN tweet splits from disk
# ------------------------------------------------------------

TWEET_SPLIT_DIR = (
    FINAL_DATA_DIR
    / "splits"
    / "tweets"
)

train_tweets_frozen = pd.read_csv(
    TWEET_SPLIT_DIR / "train_tweets.csv",
    low_memory=False
)

val_tweets_frozen = pd.read_csv(
    TWEET_SPLIT_DIR / "validation_tweets.csv",
    low_memory=False
)

test_tweets_frozen = pd.read_csv(
    TWEET_SPLIT_DIR / "test_tweets.csv",
    low_memory=False
)


# ------------------------------------------------------------
# 2. Basic integrity
# ------------------------------------------------------------

print("=" * 100)
print("FROZEN TWEET SPLITS")
print("=" * 100)

for name, df in [
    ("Train", train_tweets_frozen),
    ("Validation", val_tweets_frozen),
    ("Test", test_tweets_frozen)
]:

    print(
        f"{name:12s} "
        f"rows={len(df):,} | "
        f"users={df['user_key'].nunique():,}"
    )


# ------------------------------------------------------------
# 3. Text availability
# ------------------------------------------------------------

def text_quality_report(df, name):

    text = df["text"].astype("string")

    missing = text.isna()

    empty = (
        text.fillna("")
        .str.strip()
        .eq("")
    )

    lengths = (
        text.fillna("")
        .str.len()
    )

    print("\n" + "=" * 100)
    print(f"{name.upper()} TEXT QUALITY")
    print("=" * 100)

    print(
        "Missing text:",
        int(missing.sum())
    )

    print(
        "Empty text:",
        int(empty.sum())
    )

    print(
        "Non-empty text:",
        int((~empty).sum())
    )

    print(
        "Median characters:",
        round(
            lengths[~empty].median(),
            2
        )
    )

    print(
        "Mean characters:",
        round(
            lengths[~empty].mean(),
            2
        )
    )

    print(
        "Max characters:",
        int(
            lengths.max()
        )
    )


text_quality_report(
    train_tweets_frozen,
    "Train"
)

text_quality_report(
    val_tweets_frozen,
    "Validation"
)

text_quality_report(
    test_tweets_frozen,
    "Test"
)


# ------------------------------------------------------------
# 4. Tweet type distribution
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("TWEET TYPE DISTRIBUTION")
print("=" * 100)

for name, df in [
    ("Train", train_tweets_frozen),
    ("Validation", val_tweets_frozen),
    ("Test", test_tweets_frozen)
]:

    print(f"\n{name}:")

    if "type" in df.columns:

        print(
            df["type"]
            .value_counts(
                dropna=False
            )
        )

    else:
        print("No 'type' column.")


# ------------------------------------------------------------
# 5. Language distribution
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("LANGUAGE DISTRIBUTION — TOP 15")
print("=" * 100)

for name, df in [
    ("Train", train_tweets_frozen),
    ("Validation", val_tweets_frozen),
    ("Test", test_tweets_frozen)
]:

    print(f"\n{name}:")

    if "lang" in df.columns:

        print(
            df["lang"]
            .value_counts(
                dropna=False
            )
            .head(15)
        )


# ------------------------------------------------------------
# 6. Exact Tweet ID duplicate audit
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("TWEET ID DUPLICATES")
print("=" * 100)

for name, df in [
    ("Train", train_tweets_frozen),
    ("Validation", val_tweets_frozen),
    ("Test", test_tweets_frozen)
]:

    duplicate_ids = (
        df["id"]
        .duplicated(
            keep=False
        )
        .sum()
    )

    print(
        f"{name:12s}: "
        f"{duplicate_ids:,} rows involved "
        f"in duplicate tweet IDs"
    )


# ------------------------------------------------------------
# 7. Normalize text ONLY for duplicate/leakage AUDIT
#
# IMPORTANT:
# This is NOT the final model text preprocessing.
# XLM-R will later receive minimally cleaned text.
# ------------------------------------------------------------

def normalize_text_for_audit(text):

    if pd.isna(text):
        return pd.NA

    text = str(text)

    text = text.lower()

    # Normalize whitespace
    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text if text else pd.NA


for df in [
    train_tweets_frozen,
    val_tweets_frozen,
    test_tweets_frozen
]:

    df["text_audit_key"] = (
        df["text"]
        .map(
            normalize_text_for_audit
        )
    )


# ------------------------------------------------------------
# 8. Within-split exact text duplicates
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("WITHIN-SPLIT EXACT TEXT DUPLICATES")
print("=" * 100)

for name, df in [
    ("Train", train_tweets_frozen),
    ("Validation", val_tweets_frozen),
    ("Test", test_tweets_frozen)
]:

    valid_texts = df[
        "text_audit_key"
    ].dropna()

    duplicated_rows = (
        valid_texts
        .duplicated(
            keep=False
        )
        .sum()
    )

    unique_texts = (
        valid_texts
        .nunique()
    )

    print(
        f"{name:12s}: "
        f"unique texts={unique_texts:,} | "
        f"rows in duplicates={duplicated_rows:,}"
    )


# ------------------------------------------------------------
# 9. Cross-split exact text overlap
# ------------------------------------------------------------

train_texts = set(
    train_tweets_frozen[
        "text_audit_key"
    ].dropna()
)

val_texts = set(
    val_tweets_frozen[
        "text_audit_key"
    ].dropna()
)

test_texts = set(
    test_tweets_frozen[
        "text_audit_key"
    ].dropna()
)


train_val_overlap = (
    train_texts
    & val_texts
)

train_test_overlap = (
    train_texts
    & test_texts
)

val_test_overlap = (
    val_texts
    & test_texts
)


print("\n" + "=" * 100)
print("CROSS-SPLIT EXACT TEXT OVERLAP")
print("=" * 100)

print(
    "Train ∩ Validation:",
    len(train_val_overlap)
)

print(
    "Train ∩ Test:",
    len(train_test_overlap)
)

print(
    "Validation ∩ Test:",
    len(val_test_overlap)
)


# ------------------------------------------------------------
# 10. Tweets per user distribution
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("TWEETS PER USER DISTRIBUTION")
print("=" * 100)

for name, df in [
    ("Train", train_tweets_frozen),
    ("Validation", val_tweets_frozen),
    ("Test", test_tweets_frozen)
]:

    counts = (
        df.groupby("user_key")
        .size()
    )

    print(f"\n{name}:")

    print(
        counts.describe(
            percentiles=[
                0.25,
                0.50,
                0.75,
                0.90,
                0.95,
                0.99
            ]
        )
    )


# ------------------------------------------------------------
# 11. Temporal field audit
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("PUBLISHED_AT AUDIT")
print("=" * 100)

for name, df in [
    ("Train", train_tweets_frozen),
    ("Validation", val_tweets_frozen),
    ("Test", test_tweets_frozen)
]:

    parsed_time = pd.to_datetime(
        df["published_at"],
        errors="coerce",
        utc=True
    )

    print(
        f"{name:12s}: "
        f"valid={parsed_time.notna().sum():,} | "
        f"missing/invalid={parsed_time.isna().sum():,} | "
        f"min={parsed_time.min()} | "
        f"max={parsed_time.max()}"
    )


# ------------------------------------------------------------
# 12. Final audit report
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("TWEET/TEXT AUDIT COMPLETE")
print("=" * 100)

print("✓ Frozen tweet splits loaded")
print("✓ Text availability checked")
print("✓ Tweet types checked")
print("✓ Language distribution checked")
print("✓ Tweet-ID duplicates checked")
print("✓ Exact text overlap across splits measured")
print("✓ Tweets-per-user imbalance measured")
print("✓ Tweet timestamps checked")
print("✓ No model preprocessing applied yet")

FROZEN TWEET SPLITS
Train        rows=45,443 | users=671
Validation   rows=13,968 | users=144
Test         rows=11,464 | users=144

TRAIN TEXT QUALITY
Missing text: 0
Empty text: 0
Non-empty text: 45443
Median characters: 136.0
Mean characters: 163.93
Max characters: 9597

VALIDATION TEXT QUALITY
Missing text: 0
Empty text: 0
Non-empty text: 13968
Median characters: 102.0
Mean characters: 130.36
Max characters: 8207

TEST TEXT QUALITY
Missing text: 0
Empty text: 0
Non-empty text: 11464
Median characters: 153.0
Mean characters: 175.01
Max characters: 3996

TWEET TYPE DISTRIBUTION

Train:
type
tweet                   28388
reply                   12586
retweet_with_comment     4469
Name: count, dtype: int64

Validation:
type
tweet                   6826
reply                   5937
retweet_with_comment    1205
Name: count, dtype: int64

Test:
type
tweet                   6604
reply                   2981
retweet_with_comment    1879
Name: count, dtype: int64

LANGUAGE DISTRIBUTION — TOP 

In [36]:
# ============================================================
# Cell 33 — Prepare Leakage-Safe Tweet Sets for Text Encoder
# ============================================================

import pandas as pd
import numpy as np
import re
import html
import unicodedata


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

MAX_TWEETS_PER_USER = 50


# ------------------------------------------------------------
# 2. Minimal text cleaning
#
# Keep:
# - hashtags
# - emojis
# - Persian/English text
#
# Normalize:
# - Unicode
# - URLs
# - mentions
# - whitespace
# ------------------------------------------------------------

def clean_tweet_text(text):

    if pd.isna(text):
        return ""

    text = str(text)

    # Decode HTML entities
    text = html.unescape(text)

    # Unicode normalization
    text = unicodedata.normalize(
        "NFKC",
        text
    )

    # Persian character normalization
    text = (
        text
        .replace("ي", "ی")
        .replace("ى", "ی")
        .replace("ك", "ک")
    )

    # Replace URLs
    text = re.sub(
        r"https?://\S+|www\.\S+",
        " URL ",
        text,
        flags=re.IGNORECASE
    )

    # Replace mentions
    text = re.sub(
        r"(?<!\w)@\w+",
        " @USER ",
        text
    )

    # Normalize whitespace
    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text


# ------------------------------------------------------------
# 3. Prepare each split
# ------------------------------------------------------------

def prepare_text_split(df):

    out = df.copy()

    out["published_at"] = pd.to_datetime(
        out["published_at"],
        errors="coerce",
        utc=True
    )

    out["text_model"] = (
        out["text"]
        .map(clean_tweet_text)
    )

    # Key used ONLY for exact duplicate detection
    out["text_key"] = (
        out["text_model"]
        .str.lower()
        .str.strip()
    )

    # Remove empty text if any
    out = out[
        out["text_model"].str.len() > 0
    ].copy()

    return out


train_text = prepare_text_split(
    train_tweets_frozen
)

val_text = prepare_text_split(
    val_tweets_frozen
)

test_text = prepare_text_split(
    test_tweets_frozen
)


# ------------------------------------------------------------
# 4. Remove repeated exact text WITHIN THE SAME USER
#
# Important:
# Same text used by DIFFERENT users is retained.
# ------------------------------------------------------------

def deduplicate_within_user(df):

    return (
        df
        .sort_values(
            ["user_key", "published_at"],
            ascending=[True, False]
        )
        .drop_duplicates(
            subset=[
                "user_key",
                "text_key"
            ],
            keep="first"
        )
        .copy()
    )


train_text_dedup = deduplicate_within_user(
    train_text
)

val_text_dedup = deduplicate_within_user(
    val_text
)

test_text_dedup = deduplicate_within_user(
    test_text
)


# ------------------------------------------------------------
# 5. Cross-split text decontamination
#
# Validation:
# remove exact texts already present in Train
#
# Test:
# remove exact texts present in Train OR Validation
# ------------------------------------------------------------

train_text_keys = set(
    train_text_dedup[
        "text_key"
    ]
)


val_before = len(
    val_text_dedup
)

val_text_clean = val_text_dedup[
    ~val_text_dedup[
        "text_key"
    ].isin(
        train_text_keys
    )
].copy()


validation_text_keys = set(
    val_text_clean[
        "text_key"
    ]
)


test_before = len(
    test_text_dedup
)

test_text_clean = test_text_dedup[
    ~test_text_dedup[
        "text_key"
    ].isin(
        train_text_keys
        |
        validation_text_keys
    )
].copy()


train_text_clean = (
    train_text_dedup.copy()
)


# ------------------------------------------------------------
# 6. Cap number of tweets per user
#
# Keep the most recent 50 unique tweets.
# ------------------------------------------------------------

def cap_tweets_per_user(
    df,
    max_tweets=50
):

    return (
        df
        .sort_values(
            ["user_key", "published_at"],
            ascending=[True, False]
        )
        .groupby(
            "user_key",
            group_keys=False
        )
        .head(max_tweets)
        .reset_index(drop=True)
    )


train_text_final = cap_tweets_per_user(
    train_text_clean,
    MAX_TWEETS_PER_USER
)

val_text_final = cap_tweets_per_user(
    val_text_clean,
    MAX_TWEETS_PER_USER
)

test_text_final = cap_tweets_per_user(
    test_text_clean,
    MAX_TWEETS_PER_USER
)


# ------------------------------------------------------------
# 7. Report helper
# ------------------------------------------------------------

def report_text_set(
    name,
    original_df,
    dedup_df,
    clean_df,
    final_df,
    expected_users
):

    counts = (
        final_df
        .groupby("user_key")
        .size()
    )

    actual_users = set(
        final_df["user_key"]
    )

    missing_users = (
        expected_users
        - actual_users
    )

    print("\n" + "=" * 100)
    print(name)
    print("=" * 100)

    print(
        "Original rows:",
        f"{len(original_df):,}"
    )

    print(
        "After within-user dedup:",
        f"{len(dedup_df):,}"
    )

    print(
        "After cross-split decontamination:",
        f"{len(clean_df):,}"
    )

    print(
        "After 50-tweet cap:",
        f"{len(final_df):,}"
    )

    print(
        "Users:",
        final_df["user_key"].nunique()
    )

    print(
        "Expected users:",
        len(expected_users)
    )

    print(
        "Users lost:",
        len(missing_users)
    )

    print(
        "Mean tweets/user:",
        round(
            counts.mean(),
            2
        )
    )

    print(
        "Median tweets/user:",
        round(
            counts.median(),
            2
        )
    )

    print(
        "Max tweets/user:",
        counts.max()
    )

    if len(missing_users) > 0:

        print(
            "Lost user_keys:",
            sorted(missing_users)
        )


# ------------------------------------------------------------
# 8. Reports
# ------------------------------------------------------------

report_text_set(
    "TRAIN TEXT SET",
    train_text,
    train_text_dedup,
    train_text_clean,
    train_text_final,
    train_keys
)

report_text_set(
    "VALIDATION TEXT SET",
    val_text,
    val_text_dedup,
    val_text_clean,
    val_text_final,
    val_keys
)

report_text_set(
    "TEST TEXT SET",
    test_text,
    test_text_dedup,
    test_text_clean,
    test_text_final,
    test_keys
)


# ------------------------------------------------------------
# 9. Final cross-split overlap check
# ------------------------------------------------------------

train_final_texts = set(
    train_text_final["text_key"]
)

val_final_texts = set(
    val_text_final["text_key"]
)

test_final_texts = set(
    test_text_final["text_key"]
)


print("\n" + "=" * 100)
print("FINAL TEXT LEAKAGE CHECK")
print("=" * 100)

print(
    "Train ∩ Validation:",
    len(
        train_final_texts
        & val_final_texts
    )
)

print(
    "Train ∩ Test:",
    len(
        train_final_texts
        & test_final_texts
    )
)

print(
    "Validation ∩ Test:",
    len(
        val_final_texts
        & test_final_texts
    )
)


# ------------------------------------------------------------
# 10. Strong checks
# ------------------------------------------------------------

assert (
    len(
        train_final_texts
        & val_final_texts
    )
    == 0
)

assert (
    len(
        train_final_texts
        & test_final_texts
    )
    == 0
)

assert (
    len(
        val_final_texts
        & test_final_texts
    )
    == 0
)


assert (
    train_text_final
    .groupby("user_key")
    .size()
    .max()
    <= MAX_TWEETS_PER_USER
)

assert (
    val_text_final
    .groupby("user_key")
    .size()
    .max()
    <= MAX_TWEETS_PER_USER
)

assert (
    test_text_final
    .groupby("user_key")
    .size()
    .max()
    <= MAX_TWEETS_PER_USER
)


print("\n" + "=" * 100)
print("TEXT SET PREPARATION AUDIT COMPLETE")
print("=" * 100)

print("✓ Tweet IDs were NOT used for deduplication")
print("✓ Duplicate text removed only within each user")
print("✓ Evaluation text decontaminated across splits")
print("✓ Maximum 50 tweets per user")
print("✓ Hashtags and emojis preserved")
print("✓ Mentions and URLs normalized")
print("✓ Ready for tokenizer-length audit")


TRAIN TEXT SET
Original rows: 45,443
After within-user dedup: 41,846
After cross-split decontamination: 41,846
After 50-tweet cap: 12,976
Users: 671
Expected users: 671
Users lost: 0
Mean tweets/user: 19.34
Median tweets/user: 9.0
Max tweets/user: 50

VALIDATION TEXT SET
Original rows: 13,968
After within-user dedup: 10,319
After cross-split decontamination: 10,152
After 50-tweet cap: 2,618
Users: 144
Expected users: 144
Users lost: 0
Mean tweets/user: 18.18
Median tweets/user: 7.0
Max tweets/user: 50

TEST TEXT SET
Original rows: 11,464
After within-user dedup: 10,397
After cross-split decontamination: 10,249
After 50-tweet cap: 3,006
Users: 143
Expected users: 144
Users lost: 1
Mean tweets/user: 21.02
Median tweets/user: 11.0
Max tweets/user: 50
Lost user_keys: ['mostafamehraeen']

FINAL TEXT LEAKAGE CHECK
Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0

TEXT SET PREPARATION AUDIT COMPLETE
✓ Tweet IDs were NOT used for deduplication
✓ Duplicate text removed only within ea

In [37]:
# ============================================================
# Cell 34 — Inspect Test User Lost After Text Decontamination
# ============================================================

LOST_USER = "mostafamehraeen"


# ------------------------------------------------------------
# 1. Original tweets of the user
# ------------------------------------------------------------

lost_original = test_text[
    test_text["user_key"] == LOST_USER
].copy()


lost_dedup = test_text_dedup[
    test_text_dedup["user_key"] == LOST_USER
].copy()


# ------------------------------------------------------------
# 2. Check overlap with Train / Validation
# ------------------------------------------------------------

lost_dedup["overlap_train"] = (
    lost_dedup["text_key"]
    .isin(train_text_keys)
)

lost_dedup["overlap_validation"] = (
    lost_dedup["text_key"]
    .isin(validation_text_keys)
)

lost_dedup["overlap_any"] = (
    lost_dedup["overlap_train"]
    |
    lost_dedup["overlap_validation"]
)


# ------------------------------------------------------------
# 3. Summary
# ------------------------------------------------------------

print("=" * 100)
print("LOST TEST USER TEXT AUDIT")
print("=" * 100)

print(
    "User:",
    LOST_USER
)

print(
    "Original tweet rows:",
    len(lost_original)
)

print(
    "Unique tweets after within-user dedup:",
    len(lost_dedup)
)

print(
    "Overlap with Train:",
    int(lost_dedup["overlap_train"].sum())
)

print(
    "Overlap with Validation:",
    int(lost_dedup["overlap_validation"].sum())
)

print(
    "Overlap with Train OR Validation:",
    int(lost_dedup["overlap_any"].sum())
)

print(
    "Non-overlapping tweets remaining:",
    int((~lost_dedup["overlap_any"]).sum())
)


# ------------------------------------------------------------
# 4. Tweet type distribution
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("TWEET TYPES")
print("=" * 100)

print(
    lost_dedup["type"]
    .value_counts(dropna=False)
)


# ------------------------------------------------------------
# 5. Language distribution
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("LANGUAGES")
print("=" * 100)

print(
    lost_dedup["lang"]
    .value_counts(dropna=False)
)


# ------------------------------------------------------------
# 6. Inspect examples
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("SAMPLE OVERLAPPING TEXTS")
print("=" * 100)

display(
    lost_dedup[
        [
            "published_at",
            "type",
            "lang",
            "text_model",
            "overlap_train",
            "overlap_validation"
        ]
    ]
    .head(20)
)


# ------------------------------------------------------------
# 7. Strong diagnostic check
# ------------------------------------------------------------

if (
    len(lost_dedup) > 0
    and lost_dedup["overlap_any"].all()
):

    print(
        "\n✓ Every unique tweet from this user "
        "already exists in Train and/or Validation."
    )

else:

    print(
        "\nWARNING: At least one non-overlapping tweet exists. "
        "The filtering logic should be rechecked."
    )

LOST TEST USER TEXT AUDIT
User: mostafamehraeen
Original tweet rows: 2
Unique tweets after within-user dedup: 1
Overlap with Train: 1
Overlap with Validation: 0
Overlap with Train OR Validation: 1
Non-overlapping tweets remaining: 0

TWEET TYPES
type
tweet    1
Name: count, dtype: int64

LANGUAGES
lang
zxx    1
Name: count, dtype: int64

SAMPLE OVERLAPPING TEXTS


,published_at,type,lang,text_model,overlap_train,overlap_validation
5534,2025-03-11 03:33:08+00:00,tweet,zxx,URL,True,False



✓ Every unique tweet from this user already exists in Train and/or Validation.


In [38]:
# ============================================================
# Cell 35 — Freeze Final Leakage-Safe Text Sets
# ============================================================

from pathlib import Path
import json
import pandas as pd


# ------------------------------------------------------------
# 1. Output directories
# ------------------------------------------------------------

TEXT_FINAL_DIR = (
    FINAL_DATA_DIR
    / "splits"
    / "text"
)

TEXT_FINAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# 2. Keep only useful columns for Text Encoder
# ------------------------------------------------------------

text_columns = [
    "user_key",
    "screen_name",
    "published_at",
    "type",
    "lang",
    "text_model"
]


train_text_model = (
    train_text_final[
        text_columns
    ]
    .copy()
)

val_text_model = (
    val_text_final[
        text_columns
    ]
    .copy()
)

test_text_model = (
    test_text_final[
        text_columns
    ]
    .copy()
)


# ------------------------------------------------------------
# 3. Create user-level text availability
# ------------------------------------------------------------

def build_text_availability(
    user_df,
    text_df,
    split_name
):

    availability = user_df[
        [
            "screen_name",
            "user_key",
            "label_binary"
        ]
    ].copy()

    available_users = set(
        text_df["user_key"]
        .unique()
    )

    tweet_counts = (
        text_df
        .groupby("user_key")
        .size()
        .to_dict()
    )

    availability["split"] = split_name

    availability["has_text_model"] = (
        availability["user_key"]
        .isin(available_users)
        .astype(int)
    )

    availability["n_text_tweets"] = (
        availability["user_key"]
        .map(tweet_counts)
        .fillna(0)
        .astype(int)
    )

    return availability


train_text_availability = (
    build_text_availability(
        train_users,
        train_text_model,
        "train"
    )
)

val_text_availability = (
    build_text_availability(
        val_users,
        val_text_model,
        "validation"
    )
)

test_text_availability = (
    build_text_availability(
        test_users,
        test_text_model,
        "test"
    )
)


text_availability = pd.concat(
    [
        train_text_availability,
        val_text_availability,
        test_text_availability
    ],
    ignore_index=True
)


# ------------------------------------------------------------
# 4. Strong integrity checks
# ------------------------------------------------------------

assert len(text_availability) == 959

assert (
    text_availability["user_key"]
    .is_unique
)

assert (
    train_text_availability[
        "has_text_model"
    ].sum()
    == 671
)

assert (
    val_text_availability[
        "has_text_model"
    ].sum()
    == 144
)

assert (
    test_text_availability[
        "has_text_model"
    ].sum()
    == 143
)


missing_test_text_users = (
    test_text_availability.loc[
        test_text_availability[
            "has_text_model"
        ] == 0,
        "user_key"
    ]
    .tolist()
)

assert missing_test_text_users == [
    "mostafamehraeen"
]


# ------------------------------------------------------------
# 5. Cross-split leakage verification
# ------------------------------------------------------------

train_model_texts = set(
    train_text_model[
        "text_model"
    ]
    .str.lower()
    .str.strip()
)

val_model_texts = set(
    val_text_model[
        "text_model"
    ]
    .str.lower()
    .str.strip()
)

test_model_texts = set(
    test_text_model[
        "text_model"
    ]
    .str.lower()
    .str.strip()
)


assert len(
    train_model_texts
    & val_model_texts
) == 0

assert len(
    train_model_texts
    & test_model_texts
) == 0

assert len(
    val_model_texts
    & test_model_texts
) == 0


# ------------------------------------------------------------
# 6. Save final Text Encoder datasets
# ------------------------------------------------------------

train_text_model.to_csv(
    TEXT_FINAL_DIR
    / "train_text.csv",
    index=False
)

val_text_model.to_csv(
    TEXT_FINAL_DIR
    / "validation_text.csv",
    index=False
)

test_text_model.to_csv(
    TEXT_FINAL_DIR
    / "test_text.csv",
    index=False
)

text_availability.to_csv(
    TEXT_FINAL_DIR
    / "text_modality_availability.csv",
    index=False
)


# ------------------------------------------------------------
# 7. Save Text preprocessing policy
# ------------------------------------------------------------

text_policy = {

    "version": "text_preprocessing_v1",

    "base_model_family": "XLM-R",

    "split_level": "user",

    "cleaning": {
        "unicode_normalization": "NFKC",
        "persian_character_normalization": True,
        "url_replacement": "URL",
        "mention_replacement": "@USER",
        "hashtags_preserved": True,
        "emojis_preserved": True,
        "case_used_only_for_duplicate_audit": True
    },

    "deduplication": {
        "tweet_id_used": False,
        "within_user_exact_text_deduplication": True,
        "cross_split_exact_text_decontamination": True
    },

    "tweet_selection": {
        "strategy": "most_recent_unique_tweets",
        "max_tweets_per_user": 50
    },

    "final_rows": {
        "train": len(train_text_model),
        "validation": len(val_text_model),
        "test": len(test_text_model)
    },

    "text_users": {
        "train": int(
            train_text_availability[
                "has_text_model"
            ].sum()
        ),
        "validation": int(
            val_text_availability[
                "has_text_model"
            ].sum()
        ),
        "test": int(
            test_text_availability[
                "has_text_model"
            ].sum()
        )
    },

    "missing_text_modality_users": {
        "train": [],
        "validation": [],
        "test": missing_test_text_users
    },

    "missing_modality_policy": (
        "Keep the user in the frozen evaluation split "
        "and mask the text modality rather than restoring "
        "cross-split overlapping text."
    )
}


TEXT_POLICY_PATH = (
    FINAL_DATA_DIR
    / "config"
    / "text_preprocessing_policy.json"
)


with open(
    TEXT_POLICY_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        text_policy,
        f,
        ensure_ascii=False,
        indent=2
    )


# ------------------------------------------------------------
# 8. Reload and verify
# ------------------------------------------------------------

check_train_text = pd.read_csv(
    TEXT_FINAL_DIR
    / "train_text.csv"
)

check_val_text = pd.read_csv(
    TEXT_FINAL_DIR
    / "validation_text.csv"
)

check_test_text = pd.read_csv(
    TEXT_FINAL_DIR
    / "test_text.csv"
)


assert len(check_train_text) == 12976
assert len(check_val_text) == 2618
assert len(check_test_text) == 3006


# ------------------------------------------------------------
# 9. Final report
# ------------------------------------------------------------

print("=" * 95)
print("FINAL TEXT DATASETS FROZEN")
print("=" * 95)

print(
    f"Train      : "
    f"{len(check_train_text):,} tweets | "
    f"{check_train_text['user_key'].nunique()} users"
)

print(
    f"Validation : "
    f"{len(check_val_text):,} tweets | "
    f"{check_val_text['user_key'].nunique()} users"
)

print(
    f"Test       : "
    f"{len(check_test_text):,} tweets | "
    f"{check_test_text['user_key'].nunique()} users"
)

print(
    "\nTest users without usable text:"
)

print(
    missing_test_text_users
)

print(
    "\nSaved to:"
)

print(
    TEXT_FINAL_DIR
)

print(
    "\n✓ User-level split preserved"
)

print(
    "✓ Exact cross-split text leakage = 0"
)

print(
    "✓ Maximum 50 tweets/user frozen"
)

print(
    "✓ Missing text modality explicitly recorded"
)

print(
    "✓ Test still contains all 144 Gold users"
)

FINAL TEXT DATASETS FROZEN
Train      : 12,976 tweets | 671 users
Validation : 2,618 tweets | 144 users
Test       : 3,006 tweets | 143 users

Test users without usable text:
['mostafamehraeen']

Saved to:
C:\Users\p.vazifeh\Desktop\University\Project\Bot Detection Implementation\final_data\splits\text

✓ User-level split preserved
✓ Exact cross-split text leakage = 0
✓ Maximum 50 tweets/user frozen
✓ Missing text modality explicitly recorded
✓ Test still contains all 144 Gold users


In [39]:
# ============================================================
# Cell 36 — XLM-R Token Length Audit
# ============================================================

import numpy as np
import pandas as pd

from transformers import AutoTokenizer


# ------------------------------------------------------------
# 1. Model / tokenizer
# ------------------------------------------------------------

TEXT_MODEL_NAME = "xlm-roberta-base"

tokenizer = AutoTokenizer.from_pretrained(
    TEXT_MODEL_NAME
)


print("=" * 95)
print("XLM-R TOKENIZER")
print("=" * 95)

print("Model:", TEXT_MODEL_NAME)
print("Tokenizer:", tokenizer.__class__.__name__)


# ------------------------------------------------------------
# 2. Load FROZEN text datasets
# ------------------------------------------------------------

TEXT_FINAL_DIR = (
    FINAL_DATA_DIR
    / "splits"
    / "text"
)

train_text_frozen = pd.read_csv(
    TEXT_FINAL_DIR / "train_text.csv",
    low_memory=False
)

val_text_frozen = pd.read_csv(
    TEXT_FINAL_DIR / "validation_text.csv",
    low_memory=False
)

test_text_frozen = pd.read_csv(
    TEXT_FINAL_DIR / "test_text.csv",
    low_memory=False
)


print("\nFrozen text datasets:")

print(
    "Train      :",
    train_text_frozen.shape
)

print(
    "Validation :",
    val_text_frozen.shape
)

print(
    "Test       :",
    test_text_frozen.shape
)


# ------------------------------------------------------------
# 3. Token length function
#
# IMPORTANT:
# - no truncation
# - no padding
# - measure TRUE token length
# ------------------------------------------------------------

def get_token_lengths(
    texts,
    batch_size=256
):

    texts = (
        texts
        .fillna("")
        .astype(str)
        .tolist()
    )

    lengths = []

    for start in range(
        0,
        len(texts),
        batch_size
    ):

        batch = texts[
            start:start + batch_size
        ]

        encoded = tokenizer(
            batch,
            add_special_tokens=True,
            truncation=False,
            padding=False
        )

        lengths.extend(
            len(ids)
            for ids in encoded["input_ids"]
        )

    return np.asarray(
        lengths,
        dtype=np.int32
    )


# ------------------------------------------------------------
# 4. Calculate token lengths
# ------------------------------------------------------------

train_token_lengths = get_token_lengths(
    train_text_frozen["text_model"]
)

val_token_lengths = get_token_lengths(
    val_text_frozen["text_model"]
)

test_token_lengths = get_token_lengths(
    test_text_frozen["text_model"]
)


# ------------------------------------------------------------
# 5. Report function
# ------------------------------------------------------------

def token_length_report(
    name,
    lengths
):

    print("\n" + "=" * 95)
    print(name)
    print("=" * 95)

    print(
        "Tweets:",
        f"{len(lengths):,}"
    )

    print(
        "Mean tokens:",
        round(
            lengths.mean(),
            2
        )
    )

    print(
        "Median tokens:",
        round(
            np.median(lengths),
            2
        )
    )

    print(
        "Max tokens:",
        int(
            lengths.max()
        )
    )

    print("\nPercentiles:")

    for percentile in [
        50,
        75,
        90,
        95,
        97,
        99,
        99.5
    ]:

        value = np.percentile(
            lengths,
            percentile
        )

        print(
            f"P{percentile:<4} -> "
            f"{value:.1f}"
        )

    print("\nTruncation rates:")

    for max_length in [
        64,
        128,
        192,
        256,
        384,
        512
    ]:

        rate = (
            lengths > max_length
        ).mean() * 100

        print(
            f"max_length={max_length:<3} "
            f"-> {rate:.2f}%"
        )


# ------------------------------------------------------------
# 6. Reports
# ------------------------------------------------------------

token_length_report(
    "TRAIN TOKEN LENGTHS",
    train_token_lengths
)

token_length_report(
    "VALIDATION TOKEN LENGTHS",
    val_token_lengths
)

token_length_report(
    "TEST TOKEN LENGTHS",
    test_token_lengths
)


# ------------------------------------------------------------
# 7. Combined summary
#
# Reporting only.
# Selection decision should mainly use Train.
# ------------------------------------------------------------

all_token_lengths = np.concatenate(
    [
        train_token_lengths,
        val_token_lengths,
        test_token_lengths
    ]
)


print("\n" + "=" * 95)
print("COMBINED TOKEN LENGTH SUMMARY")
print("=" * 95)

print(
    "Total tweets:",
    f"{len(all_token_lengths):,}"
)

print(
    "Median:",
    round(
        np.median(all_token_lengths),
        2
    )
)

print(
    "P90:",
    round(
        np.percentile(
            all_token_lengths,
            90
        ),
        2
    )
)

print(
    "P95:",
    round(
        np.percentile(
            all_token_lengths,
            95
        ),
        2
    )
)

print(
    "P99:",
    round(
        np.percentile(
            all_token_lengths,
            99
        ),
        2
    )
)

print(
    "Maximum:",
    int(
        all_token_lengths.max()
    )
)


# ------------------------------------------------------------
# 8. Integrity checks
# ------------------------------------------------------------

assert len(train_token_lengths) == 12976
assert len(val_token_lengths) == 2618
assert len(test_token_lengths) == 3006

assert (
    train_token_lengths > 0
).all()

assert (
    val_token_lengths > 0
).all()

assert (
    test_token_lengths > 0
).all()


print("\n" + "=" * 95)
print("TOKEN LENGTH AUDIT COMPLETE")
print("=" * 95)

print("✓ XLM-R tokenizer used")
print("✓ No truncation applied during audit")
print("✓ No padding applied during audit")
print("✓ max_length has NOT been selected yet")

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

XLM-R TOKENIZER
Model: xlm-roberta-base
Tokenizer: XLMRobertaTokenizer

Frozen text datasets:
Train      : (12976, 6)
Validation : (2618, 6)
Test       : (3006, 6)


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (577 > 512). Running this sequence through the model will result in indexing errors



TRAIN TOKEN LENGTHS
Tweets: 12,976
Mean tokens: 53.91
Median tokens: 43.0
Max tokens: 2578

Percentiles:
P50   -> 43.0
P75   -> 70.0
P90   -> 87.0
P95   -> 96.0
P97   -> 104.0
P99   -> 219.8
P99.5 -> 422.6

Truncation rates:
max_length=64  -> 29.18%
max_length=128 -> 1.91%
max_length=192 -> 1.16%
max_length=256 -> 0.91%
max_length=384 -> 0.58%
max_length=512 -> 0.39%

VALIDATION TOKEN LENGTHS
Tweets: 2,618
Mean tokens: 50.17
Median tokens: 42.0
Max tokens: 2360

Percentiles:
P50   -> 42.0
P75   -> 63.0
P90   -> 81.0
P95   -> 90.0
P97   -> 97.5
P99   -> 166.8
P99.5 -> 308.1

Truncation rates:
max_length=64  -> 23.83%
max_length=128 -> 1.60%
max_length=192 -> 0.80%
max_length=256 -> 0.65%
max_length=384 -> 0.27%
max_length=512 -> 0.19%

TEST TOKEN LENGTHS
Tweets: 3,006
Mean tokens: 57.75
Median tokens: 48.0
Max tokens: 1147

Percentiles:
P50   -> 48.0
P75   -> 75.0
P90   -> 90.0
P95   -> 99.0
P97   -> 114.0
P99   -> 250.6
P99.5 -> 438.3

Truncation rates:
max_length=64  -> 33.00%
max_le

In [40]:
# ============================================================
# Cell 37 — Final Preprocessing Manifest
# ============================================================

from pathlib import Path
import json
import pandas as pd


# ------------------------------------------------------------
# 1. Freeze final Text Encoder configuration
# ------------------------------------------------------------

TEXT_MODEL_NAME = "xlm-roberta-base"
MAX_TOKEN_LENGTH = 128
MAX_TWEETS_PER_USER = 50


# ------------------------------------------------------------
# 2. Update Text preprocessing policy
# ------------------------------------------------------------

TEXT_POLICY_PATH = (
    FINAL_DATA_DIR
    / "config"
    / "text_preprocessing_policy.json"
)

with open(
    TEXT_POLICY_PATH,
    "r",
    encoding="utf-8"
) as f:
    text_policy = json.load(f)


text_policy["tokenization"] = {

    "model_name": TEXT_MODEL_NAME,

    "tokenizer": "XLMRobertaTokenizer",

    "max_length": MAX_TOKEN_LENGTH,

    "truncation": True,

    "padding_strategy": "dynamic_per_batch",

    "add_special_tokens": True,

    "selection_basis": "training_token_length_distribution",

    "training_distribution": {
        "median_tokens": 43.0,
        "p90_tokens": 87.0,
        "p95_tokens": 96.0,
        "p97_tokens": 104.0,
        "p99_tokens": 219.8,
        "max_tokens": 2578,
        "truncation_rate_at_128_percent": 1.91
    },

    "validation_truncation_rate_at_128_percent": 1.60,

    "test_truncation_rate_at_128_percent": 2.46
}


with open(
    TEXT_POLICY_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        text_policy,
        f,
        ensure_ascii=False,
        indent=2
    )


# ------------------------------------------------------------
# 3. Load frozen split metadata
# ------------------------------------------------------------

SPLITS_DIR = (
    FINAL_DATA_DIR
    / "splits"
)

train_users_check = pd.read_csv(
    SPLITS_DIR / "users" / "train_users.csv",
    low_memory=False
)

val_users_check = pd.read_csv(
    SPLITS_DIR / "users" / "validation_users.csv",
    low_memory=False
)

test_users_check = pd.read_csv(
    SPLITS_DIR / "users" / "test_users.csv",
    low_memory=False
)


train_tabular_check = pd.read_csv(
    SPLITS_DIR / "tabular" / "train_tabular.csv"
)

val_tabular_check = pd.read_csv(
    SPLITS_DIR / "tabular" / "validation_tabular.csv"
)

test_tabular_check = pd.read_csv(
    SPLITS_DIR / "tabular" / "test_tabular.csv"
)


train_text_check = pd.read_csv(
    SPLITS_DIR / "text" / "train_text.csv"
)

val_text_check = pd.read_csv(
    SPLITS_DIR / "text" / "validation_text.csv"
)

test_text_check = pd.read_csv(
    SPLITS_DIR / "text" / "test_text.csv"
)


train_graph_check = pd.read_csv(
    SPLITS_DIR / "graph" / "train_graph_edges.csv"
)

val_graph_check = pd.read_csv(
    SPLITS_DIR / "graph" / "validation_graph_edges.csv"
)

test_graph_check = pd.read_csv(
    SPLITS_DIR / "graph" / "test_graph_edges.csv"
)


# ------------------------------------------------------------
# 4. Final preprocessing manifest
# ------------------------------------------------------------

preprocessing_manifest = {

    "version": "preprocessing_final_v1",

    "status": "FROZEN",

    "task": "binary_twitter_bot_detection",

    "target": {
        "column": "label_binary",
        "human": 0,
        "bot": 1
    },

    # --------------------------------------------------------
    # Gold supervised data
    # --------------------------------------------------------

    "gold_binary_dataset": {
        "total_users": 959,
        "human": 772,
        "bot": 187
    },

    # --------------------------------------------------------
    # User split
    # --------------------------------------------------------

    "user_split": {

        "strategy": "stratified_user_level",

        "stratified_by": [
            "label_binary",
            "has_graph"
        ],

        "random_seed": 42,

        "train": {
            "users": 671,
            "human": 540,
            "bot": 131,
            "graph_owners": 193
        },

        "validation": {
            "users": 144,
            "human": 116,
            "bot": 28,
            "graph_owners": 42
        },

        "test": {
            "users": 144,
            "human": 116,
            "bot": 28,
            "graph_owners": 41
        }
    },

    # --------------------------------------------------------
    # Tabular modality
    # --------------------------------------------------------

    "tabular": {

        "initial_features": 42,

        "removed_redundant_features": [
            "status_count",
            "normal_followers_count"
        ],

        "final_features": 40,

        "continuous_features": 33,

        "effective_binary_features": 7,

        "missing_value_strategy": "median",

        "imputer_fit_scope": "train_only",

        "log1p_features": 30,

        "scaler": "RobustScaler",

        "scaler_fit_scope": "train_only",

        "train_shape": list(
            train_tabular_check.shape
        ),

        "validation_shape": list(
            val_tabular_check.shape
        ),

        "test_shape": list(
            test_tabular_check.shape
        )
    },

    # --------------------------------------------------------
    # Text modality
    # --------------------------------------------------------

    "text": {

        "encoder_family": TEXT_MODEL_NAME,

        "max_tweets_per_user": MAX_TWEETS_PER_USER,

        "max_token_length": MAX_TOKEN_LENGTH,

        "truncation": True,

        "padding": "dynamic_per_batch",

        "train": {
            "tweets": len(train_text_check),
            "users": train_text_check[
                "user_key"
            ].nunique()
        },

        "validation": {
            "tweets": len(val_text_check),
            "users": val_text_check[
                "user_key"
            ].nunique()
        },

        "test": {
            "tweets": len(test_text_check),
            "users_with_text": test_text_check[
                "user_key"
            ].nunique(),

            "total_users": 144,

            "missing_text_users": [
                "mostafamehraeen"
            ]
        },

        "cross_split_exact_text_overlap": 0
    },

    # --------------------------------------------------------
    # Graph modality
    # --------------------------------------------------------

    "graph": {

        "evaluation_type": "inductive",

        "relation": "follows",

        "cross_split_gold_neighbors_allowed": False,

        "external_and_unlabeled_neighbors_preserved": True,

        "train": {
            "graph_owners": 193,
            "edges": len(
                train_graph_check
            )
        },

        "validation": {
            "graph_owners": 42,
            "edges": len(
                val_graph_check
            )
        },

        "test": {
            "graph_owners": 41,
            "edges": len(
                test_graph_check
            )
        }
    },

    # --------------------------------------------------------
    # Leakage prevention
    # --------------------------------------------------------

    "leakage_controls": {

        "user_overlap_between_splits": False,

        "tweet_user_overlap_between_splits": False,

        "exact_text_overlap_between_splits": False,

        "cross_split_gold_graph_neighbors": False,

        "imputer_fitted_on_validation_or_test": False,

        "scaler_fitted_on_validation_or_test": False,

        "feature_selection_used_validation_or_test": False
    },

    # --------------------------------------------------------
    # Unlabeled data
    # --------------------------------------------------------

    "unlabeled_pool": {

        "users": 18331,

        "used_for_current_supervised_split": False,

        "future_role": [
            "pseudo_labeling",
            "semi_supervised_learning"
        ],

        "preprocessing_rule":
            "Reuse frozen Train-fitted preprocessing; never refit."
    }
}


# ------------------------------------------------------------
# 5. Save manifest
# ------------------------------------------------------------

MANIFEST_PATH = (
    FINAL_DATA_DIR
    / "config"
    / "preprocessing_manifest.json"
)


with open(
    MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        preprocessing_manifest,
        f,
        ensure_ascii=False,
        indent=2
    )


# ------------------------------------------------------------
# 6. Final strong checks
# ------------------------------------------------------------

assert len(train_users_check) == 671
assert len(val_users_check) == 144
assert len(test_users_check) == 144

assert train_tabular_check.shape == (671, 43)
assert val_tabular_check.shape == (144, 43)
assert test_tabular_check.shape == (144, 43)

assert len(train_text_check) == 12976
assert len(val_text_check) == 2618
assert len(test_text_check) == 3006

assert len(train_graph_check) == 60936
assert len(val_graph_check) == 13006
assert len(test_graph_check) == 12536


# ------------------------------------------------------------
# 7. Final report
# ------------------------------------------------------------

print("=" * 100)
print("PHASE 2 — PREPROCESSING COMPLETE")
print("=" * 100)

print("\nGOLD DATA")
print("959 users → 772 Human + 187 Bot")

print("\nUSER SPLIT")
print("Train      : 671")
print("Validation : 144")
print("Test       : 144")

print("\nTABULAR")
print("40 final model features")
print("33 continuous + 7 binary")
print("Median Imputation + Log1p + RobustScaler")

print("\nTEXT")
print("Encoder        :", TEXT_MODEL_NAME)
print("Max tweets/user:", MAX_TWEETS_PER_USER)
print("Max token length:", MAX_TOKEN_LENGTH)

print(
    f"Train      : {len(train_text_check):,} tweets"
)

print(
    f"Validation : {len(val_text_check):,} tweets"
)

print(
    f"Test       : {len(test_text_check):,} tweets"
)

print("\nGRAPH")
print(
    f"Train      : {len(train_graph_check):,} edges"
)

print(
    f"Validation : {len(val_graph_check):,} edges"
)

print(
    f"Test       : {len(test_graph_check):,} edges"
)

print("\nManifest saved:")
print(MANIFEST_PATH)

print("\n✓ User preprocessing frozen")
print("✓ Tabular preprocessing frozen")
print("✓ Text preprocessing frozen")
print("✓ XLM-R max_length = 128 frozen")
print("✓ Graph preprocessing frozen")
print("✓ Train / Validation / Test frozen")
print("✓ Leakage controls verified")

print("\nPHASE 2 IS COMPLETE.")

PHASE 2 — PREPROCESSING COMPLETE

GOLD DATA
959 users → 772 Human + 187 Bot

USER SPLIT
Train      : 671
Validation : 144
Test       : 144

TABULAR
40 final model features
33 continuous + 7 binary
Median Imputation + Log1p + RobustScaler

TEXT
Encoder        : xlm-roberta-base
Max tweets/user: 50
Max token length: 128
Train      : 12,976 tweets
Validation : 2,618 tweets
Test       : 3,006 tweets

GRAPH
Train      : 60,936 edges
Validation : 13,006 edges
Test       : 12,536 edges

Manifest saved:
C:\Users\p.vazifeh\Desktop\University\Project\Bot Detection Implementation\final_data\config\preprocessing_manifest.json

✓ User preprocessing frozen
✓ Tabular preprocessing frozen
✓ Text preprocessing frozen
✓ XLM-R max_length = 128 frozen
✓ Graph preprocessing frozen
✓ Train / Validation / Test frozen
✓ Leakage controls verified

PHASE 2 IS COMPLETE.
